In [ ]:
# ============================================================================
# CELL 1: State Management & Utilities
# ============================================================================
# CHANGES from v3:
# 1. NEW: TAUGHT_BAG_TRANSITIONS_FILE path
# 2. NEW: ITEM_KNOWLEDGE_FILE path
# 3. NEW: DEFAULT_ITEM_KNOWLEDGE dict — per-item empirical effect tracking
# 4. All other code unchanged from v3
# ============================================================================

from pathlib import Path
import json
import numpy as np
import time
from collections import deque

BASE_PATH = Path("C:/Users/HP/Documents/cogai/")
ACTION_FILE = BASE_PATH / "action.json"
STATE_FILE = BASE_PATH / "game_state.json"
TAUGHT_TRANSITIONS_FILE = BASE_PATH / "taught_transitions.json"
TAUGHT_BATTLE_TRANSITIONS_FILE = BASE_PATH / "taught_battle_transitions.json"
TAUGHT_BAG_TRANSITIONS_FILE = BASE_PATH / "taught_bag_transitions.json"
EXPLORATION_MEMORY_FILE = BASE_PATH / "exploration_memory.json"
MODEL_CHECKPOINT_FILE = BASE_PATH / "model_checkpoint.json"
TAUGHT_EXPLORATION_FILE = BASE_PATH / "taught_exploration_memory.json"
TAUGHT_NAV_TARGETS_FILE = BASE_PATH / "taught_nav_targets.json"
ROSTER_FILE = BASE_PATH / "roster.json"
MOVE_KNOWLEDGE_FILE = BASE_PATH / "move_knowledge.json"
ITEM_KNOWLEDGE_FILE = BASE_PATH / "item_knowledge.json"
EVENT_TIMELINE_FILE = BASE_PATH / "event_timeline.json"

# === MARKOV SIMILARITY WEIGHTS ===
MARKOV_IMMEDIATE_WEIGHT = 0.5
MARKOV_SEQUENTIAL_WEIGHT = 0.3
MARKOV_PARTIAL_WEIGHT = 0.2
MARKOV_FAMILIARITY_THRESHOLD = 0.6

MARKOV_SEQ_FULL_WEIGHT = 1.0
MARKOV_SEQ_MEDIUM_WEIGHT = 0.6
MARKOV_SEQ_SHORT_WEIGHT = 0.3

MARKOV_POS_EXACT_BONUS = 0.35
MARKOV_POS_NEAR_BONUS = 0.25
MARKOV_POS_FAR_BONUS = 0.1
MARKOV_POS_MAX_DIST = 5

# === BATTLE MARKOV WEIGHTS ===
BATTLE_MARKOV_ACTION_SEQ_WEIGHT = 0.70
BATTLE_MARKOV_PALETTE_WEIGHT = 0.20
BATTLE_MARKOV_MENU_STATE_WEIGHT = 0.10
BATTLE_MARKOV_THRESHOLD_LOW = 0.35
BATTLE_MARKOV_THRESHOLD_HIGH = 0.45

# === BAG MARKOV WEIGHTS ===
BAG_MARKOV_MENU_STATE_WEIGHT = 0.40
BAG_MARKOV_ACTION_SEQ_WEIGHT = 0.35
BAG_MARKOV_PARTY_CONTEXT_WEIGHT = 0.25
BAG_MARKOV_THRESHOLD = 0.35

EXPECTED_STATE_DIM = 6
PALETTE_DIM = 768
TILE_DIM = 600

# === DEFAULT BATTLE DATA (all -1 = unavailable) ===
DEFAULT_BATTLE_DATA = {
    'battle_cursor': -1,
    'move_cursor': -1,
    'party_cursor': -1,
    'player_species': -1,
    'enemy_species': -1,
    'player_hp': -1,
    'player_max_hp': -1,
    'enemy_hp': -1,
    'enemy_max_hp': -1,
    'player_level': -1,
    'enemy_level': -1,
    'player_status': 0,
    'enemy_status': 0,
    'battle_type': 0,
    'move0': -1, 'move1': -1, 'move2': -1, 'move3': -1,
    'pp0': -1, 'pp1': -1, 'pp2': -1, 'pp3': -1,
    'player_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
    'enemy_move0': -1, 'enemy_move1': -1, 'enemy_move2': -1, 'enemy_move3': -1,
    'enemy_pp0': -1, 'enemy_pp1': -1, 'enemy_pp2': -1, 'enemy_pp3': -1,
    'enemy_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
}

# === DEFAULT PARTY DATA ===
DEFAULT_PARTY_SLOT = {
    'level': 0, 'hp': 0, 'max_hp': 0,
    'atk': 0, 'def': 0, 'spd': 0, 'spatk': 0, 'spdef': 0,
    'status': 0
}

DEFAULT_PARTY_DATA = {
    'count': 0,
    'slots': []
}

# === DEFAULT MENU DATA (from "mu" field) ===
DEFAULT_MENU_DATA = {
    'mc': -1,
    'mm': -1,
    'pc': -1,
    'sc': -1,
}

# === DEFAULT BAG DATA (from "bg" field) ===
DEFAULT_BAG_DATA = {
    'pocket': -1,
    'cursor': -1,
    'active': 0,
    'items': [],
}

# === DEFAULT ITEM KNOWLEDGE ===
# Per-item empirical effect tracking. Learned through observation:
# use item → snapshot party state before → detect changes after → record effect.
#
# Structure per item ID:
#   'uses': int — total times observed being used
#   'category': str — learned category:
#       'unknown', 'heal_hp', 'heal_status', 'heal_both',
#       'catch', 'stat_boost', 'battle_item', 'key_item', 'other'
#   'confidence': float — 0.0 to 1.0, increases with uses
#   'avg_hp_restored': float — average HP gained when used (0 if not healing)
#   'status_cured': list — status condition values that were cured [1,2,4,...]
#   'catch_attempts': int — times used when enemy was present (Pokeball detection)
#   'catch_successes': int — times party count increased after use
#   'last_used_timestep': int
#
# Future phases will add:
#   'stat_effects': dict — which stats changed and by how much
#   'battle_only': bool — whether item only works in battle
#   'target_type': str — 'single_party', 'all_party', 'enemy', 'self'
DEFAULT_ITEM_KNOWLEDGE_ENTRY = {
    'uses': 0,
    'category': 'unknown',
    'confidence': 0.0,
    'avg_hp_restored': 0.0,
    'total_hp_restored': 0,
    'status_cured': [],
    'catch_attempts': 0,
    'catch_successes': 0,
    'last_used_timestep': 0,
}


def normalize_game_state(raw_state):
    if len(raw_state) < 6:
        return raw_state
    normalized = raw_state.copy()
    normalized[0] = raw_state[0] / 255.0
    normalized[1] = raw_state[1] / 255.0
    normalized[2] = np.clip(raw_state[2], 0, 255)
    normalized[3] = 1.0 if raw_state[3] > 0 else 0.0
    normalized[4] = 1.0 if raw_state[4] > 0 else 0.0
    normalized[5] = int(raw_state[5]) % 4
    return normalized


def compute_derived_features(current, prev):
    if prev is None:
        return np.zeros(8)
    vel_x = current[0] - prev[0]
    vel_y = current[1] - prev[1]
    map_changed = 1.0 if abs(current[2] - prev[2]) > 0.5 else 0.0
    battle_started = 1.0 if current[3] > prev[3] else 0.0
    battle_ended = 1.0 if current[3] < prev[3] else 0.0
    menu_opened = 1.0 if current[4] > prev[4] else 0.0
    menu_closed = 1.0 if current[4] < prev[4] else 0.0
    direction_changed = 1.0 if current[5] != prev[5] else 0.0
    return np.array([vel_x, vel_y, map_changed, battle_started, battle_ended,
                     menu_opened, menu_closed, direction_changed])


def build_learning_state(derived, palette, tiles, in_battle):
    if in_battle > 0.5:
        state = np.concatenate([derived, palette])
    else:
        state = np.concatenate([derived, tiles, palette])
    noise = np.random.randn(len(state)) * 0.0001
    return state + noise


def _pad_or_trim(arr, target_dim):
    if arr.shape[0] < target_dim:
        return np.pad(arr, (0, target_dim - arr.shape[0]))
    elif arr.shape[0] > target_dim:
        return arr[:target_dim]
    return arr


def parse_battle_data(data):
    """Parse battle data from the 'b' field in game_state.json."""
    b = data.get('b')
    if b is None:
        return DEFAULT_BATTLE_DATA.copy()

    pss = b.get('pss')
    if not isinstance(pss, list) or len(pss) != 7:
        pss = [-1, -1, -1, -1, -1, -1, -1]
    else:
        pss = list(pss)

    ess = b.get('ess')
    if not isinstance(ess, list) or len(ess) != 7:
        ess = [-1, -1, -1, -1, -1, -1, -1]
    else:
        ess = list(ess)

    return {
        'battle_cursor': b.get('bc', -1),
        'move_cursor':   b.get('mc', -1),
        'party_cursor':  b.get('pc', -1),
        'player_species': b.get('ps', -1),
        'enemy_species':  b.get('es', -1),
        'player_hp':     b.get('ph', -1),
        'player_max_hp': b.get('pm', -1),
        'enemy_hp':      b.get('eh', -1),
        'enemy_max_hp':  b.get('em', -1),
        'player_level':  b.get('pl', -1),
        'enemy_level':   b.get('el', -1),
        'player_status': b.get('pst', 0),
        'enemy_status':  b.get('est', 0),
        'battle_type':   b.get('bt', 0),
        'move0': b.get('m0', -1),
        'move1': b.get('m1', -1),
        'move2': b.get('m2', -1),
        'move3': b.get('m3', -1),
        'pp0': b.get('pp0', -1),
        'pp1': b.get('pp1', -1),
        'pp2': b.get('pp2', -1),
        'pp3': b.get('pp3', -1),
        'player_stat_stages': pss,
        'enemy_move0': b.get('em0', -1),
        'enemy_move1': b.get('em1', -1),
        'enemy_move2': b.get('em2', -1),
        'enemy_move3': b.get('em3', -1),
        'enemy_pp0': b.get('epp0', -1),
        'enemy_pp1': b.get('epp1', -1),
        'enemy_pp2': b.get('epp2', -1),
        'enemy_pp3': b.get('epp3', -1),
        'enemy_stat_stages': ess,
    }


def parse_party_data(data):
    """Parse party data from the 'pa' field in game_state.json."""
    pa = data.get('pa')
    if pa is None:
        return DEFAULT_PARTY_DATA.copy()

    count = pa.get('c', 0)
    raw_slots = pa.get('s', [])

    slots = []
    for s in raw_slots:
        if not isinstance(s, dict):
            continue
        slots.append({
            'level':  s.get('l', 0),
            'hp':     s.get('h', 0),
            'max_hp': s.get('m', 0),
            'atk':    s.get('a', 0),
            'def':    s.get('d', 0),
            'spd':    s.get('sp', 0),
            'spatk':  s.get('sa', 0),
            'spdef':  s.get('sd', 0),
            'status': s.get('st', 0),
        })

    return {
        'count': count,
        'slots': slots
    }


def parse_menu_data(data):
    """Parse menu data from the 'mu' field in game_state.json."""
    mu = data.get('mu')
    if mu is None:
        return DEFAULT_MENU_DATA.copy()

    return {
        'mc': mu.get('mc', -1),
        'mm': mu.get('mm', -1),
        'pc': mu.get('pc', -1),
        'sc': mu.get('sc', -1),
    }


def parse_bag_data(data):
    """Parse bag data from the 'bg' field in game_state.json."""
    bg = data.get('bg')
    if bg is None:
        return DEFAULT_BAG_DATA.copy()

    items = []
    raw_items = bg.get('it', [])
    if isinstance(raw_items, list):
        for item in raw_items:
            if isinstance(item, dict) and 'id' in item:
                items.append({
                    'id': item.get('id', 0),
                    'q': item.get('q', 0),
                })

    return {
        'pocket': bg.get('pk', -1),
        'cursor': bg.get('bc', -1),
        'active': bg.get('a', 0),
        'items': items,
    }


def parse_game_state_data(data):
    """Parse game state dict handling both long and short key formats.

    Returns: (raw, palette_raw, tiles_raw, dead, battle_data, party_data,
              game_state_raw, menu_data, bag_data)
    """
    raw = data.get("state") or data.get("s") or []
    palette_raw = data.get("palette") or data.get("p") or []
    tiles_raw = data.get("tiles") or data.get("t") or []
    dead = bool(data.get("dead", False))
    battle_data = parse_battle_data(data)
    party_data = parse_party_data(data)
    game_state_raw = data.get("gs", 0)
    menu_data = parse_menu_data(data)
    bag_data = parse_bag_data(data)
    return raw, palette_raw, tiles_raw, dead, battle_data, party_data, game_state_raw, menu_data, bag_data


def read_game_state(max_retries=3):
    """
    Read game state from JSON file.

    Returns: (context_state, palette_state, tile_state, dead,
              raw_position, battle_data, party_data,
              game_state_raw, menu_data, bag_data)
    """
    if not STATE_FILE.exists():
        return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy())

    for attempt in range(max_retries):
        try:
            with open(STATE_FILE, "r") as f:
                data = json.loads(f.read())

            (raw, palette_raw, tiles_raw, dead, battle_data, party_data,
             game_state_raw, menu_data, bag_data) = parse_game_state_data(data)

            raw_x = int(raw[0]) if len(raw) > 0 else 0
            raw_y = int(raw[1]) if len(raw) > 1 else 0
            raw_position = (raw_x, raw_y)

            context_state = normalize_game_state(np.array(raw, dtype=float))
            palette_state = np.array(palette_raw, dtype=float) if palette_raw else np.zeros(PALETTE_DIM)
            tile_state = np.array(tiles_raw, dtype=float) if tiles_raw else np.zeros(TILE_DIM)

            context_state = _pad_or_trim(context_state, EXPECTED_STATE_DIM)
            palette_state = _pad_or_trim(palette_state, PALETTE_DIM)
            tile_state = _pad_or_trim(tile_state, TILE_DIM)

            return (context_state, palette_state, tile_state, dead, raw_position,
                    battle_data, party_data, game_state_raw, menu_data, bag_data)

        except (json.JSONDecodeError, ValueError):
            if attempt < max_retries - 1:
                time.sleep(0.001)
                continue
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy())
        except Exception:
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy())


def write_action(action_name):
    if action_name:
        action_name = action_name.upper()
    try:
        with open(ACTION_FILE, "w") as f:
            json.dump({"action": action_name}, f)
            f.flush()
    except Exception as e:
        print(f"[ERROR] Failed to write action: {e}")

In [ ]:
# ============================================================================
# CELL 2: Perceptron Classes
# ============================================================================
# CHANGES:
# 1. Added cluster_activations (deque maxlen=50) for clustering comparison
# 2. Entity perceptrons record into cluster_activations on every predict()
# ============================================================================

class Perceptron:
    def __init__(self, kind, action=None, group=None, entity_type=None):
        self.kind = kind
        self.action = action
        self.group = group
        self.entity_type = entity_type
        
        self.utility = 1.0
        self.weights = None
        
        self.eligibility_fast = 0.0
        self.eligibility_slow = 0.0
        
        self.familiarity = 0.0
        self.activation_history = deque(maxlen=10)
        self.cluster_activations = deque(maxlen=50)  # NEW: longer history for clustering
        
        self.learning_rate = 0.01
        self.prediction_errors = deque(maxlen=50)

    def ensure_weights(self, dim):
        if self.weights is None:
            self.weights = np.random.randn(dim) * 0.001

    def predict(self, state):
        self.ensure_weights(len(state))
        
        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            raw_activation = np.dot(self.weights[:min_dim], state[:min_dim])
        else:
            raw_activation = np.dot(self.weights, state)
        
        if self.kind == "entity":
            novelty_factor = 1.0 / (1.0 + np.sqrt(self.familiarity * 0.5))
            decayed_activation = raw_activation * novelty_factor
            self.activation_history.append(abs(raw_activation))
            self.cluster_activations.append(abs(raw_activation))  # NEW
            return decayed_activation
        else:
            return raw_activation

    def adapt_learning_rate(self):
        if len(self.prediction_errors) >= 50:
            avg_error = np.mean(self.prediction_errors)
            
            if avg_error < 0.1:
                self.learning_rate = max(0.001, self.learning_rate * 0.99)
            elif avg_error > 0.5:
                self.learning_rate = min(0.05, self.learning_rate * 1.01)

    def update(self, state, error, gamma_fast=0.5, gamma_slow=0.95, stagnation=0.0):
        self.ensure_weights(len(state))
        
        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            state = state[:min_dim]
            self.weights = self.weights[:min_dim]
        
        self.eligibility_fast = gamma_fast * self.eligibility_fast + 1.0
        self.eligibility_slow = gamma_slow * self.eligibility_slow + 1.0
        
        self.adapt_learning_rate()
        
        fast_update = 0.7 * self.learning_rate * error * state * self.eligibility_fast
        slow_update = 0.3 * self.learning_rate * error * state * self.eligibility_slow
        self.weights += fast_update + slow_update

        if self.kind == "action":
            if error > 0.01:
                if stagnation > 0.5:
                    self.utility *= 0.97
                elif error > 0.2:
                    self.utility = min(self.utility * 1.02, 2.0)
                else:
                    self.utility *= 0.995
            
            if self.group == "move":
                self.utility = np.clip(self.utility, 0.1, 2.0)
            else:
                self.utility = np.clip(self.utility, 0.01, 2.0)
        
        if self.kind == "entity" and len(self.activation_history) > 0:
            recent_avg = np.mean(self.activation_history)
            if recent_avg > 0.1:
                self.familiarity += 0.03
        
        if self.kind == "entity":
            prediction = self.predict(state)
            self.prediction_errors.append(abs(prediction - error))


class ControlSwapPerceptron(Perceptron):
    def __init__(self):
        super().__init__(kind="control_swap")
        self.swap_history = deque(maxlen=100)
        self.confidence = 0.0
        
    def should_swap(self, state, movement_stagnation):
        if self.weights is None:
            return False, 0.0
        
        self.ensure_weights(len(state))
        swap_score = np.dot(self.weights, state)
        stagnation_factor = np.tanh(movement_stagnation / 5.0)
        combined_score = swap_score * 0.7 + stagnation_factor * 0.3
        
        return combined_score > 0.5, abs(combined_score)
    
    def record_swap_outcome(self, state, swapped, novelty_gained):
        self.swap_history.append((swapped, novelty_gained))
        
        if len(self.swap_history) >= 20:
            recent = list(self.swap_history)[-20:]
            successful = sum(1 for swap, nov in recent if swap and nov > 0.2)
            self.confidence = successful / 20.0

In [ ]:
# ============================================================================
# CELL 3 PART 1: Brain Class - Initialization (All State Variables)
# ============================================================================
# CHANGES from bag thread version:
# 1. NEW: Event timeline state variables — event_timeline, event_segments,
#    event_preparation_points, event_timeline_metadata, event_timeline_loaded
# 2. NEW: load_event_timeline() — loads structured playthrough history
# 3. NEW: get_nearest_nav_order() — maps position to journey progress
# 4. NEW: get_upcoming_events() — lookahead into upcoming events
# 5. NEW: get_segment_difficulty() — pre-computed difficulty between targets
# 6. NEW: get_preparation_point() — check if current nav order has prep data
# 7. NEW: get_upcoming_battles() — shortcut for battle-specific lookahead
# 8. NEW: get_upcoming_trainer_battles() — trainer battles only
# 9. All other methods unchanged
# ============================================================================

class Brain:
    def __init__(self):
        self.perceptrons = []

        self.prev_learning_states = deque(maxlen=50)
        self.prev_context_states = deque(maxlen=10)
        self.last_positions = deque(maxlen=30)
        self.action_history = deque(maxlen=100)

        self.control_mode = "move"
        self.timestep = 0
        self.last_action = None
        self.last_direction = 0

        self.MOVE_UTILITY_FLOOR = 0.05
        self.INTERACT_UTILITY_FLOOR = 0.15

        # === PERSISTENT EXPLORATION MEMORY ===
        self.EXPLORATION_MEMORY_FILE = BASE_PATH / "exploration_memory.json"
        self.exploration_memory = {}
        self.current_map_id = None
        self.SAVE_INTERVAL = 100

        self.DIRECTION_NAMES = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
        self.DIRECTION_TO_INT = {"DOWN": 0, "UP": 1, "LEFT": 2, "RIGHT": 3}
        self.INT_TO_ACTION = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}

        self.DIRECTION_DELTAS_INT = {0: (0, 1), 1: (0, -1), 2: (-1, 0), 3: (1, 0)}
        self.ACTION_DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
        self.DELTA_TO_DIRECTION = {(0, 1): 0, (0, -1): 1, (-1, 0): 2, (1, 0): 3}

        self.load_exploration_memory()

        # === MARKOV TRANSITION SYSTEM (OVERWORLD) ===
        self.taught_transitions = []
        self.taught_batches = []
        self.taught_metadata = {}
        self.markov_enabled = True
        self.markov_action_count = 0
        self.curiosity_action_count = 0
        self.last_markov_score = 0.0
        self.last_markov_action = None

        # === BATTLE THREAD STATE ===
        self.battle_transitions = []
        self.battle_sequences = []
        self.battle_metadata = {}
        self.battle_loaded = False
        self.battle_action_count = 0
        self.battle_markov_action_count = 0
        self.current_battle_id = 0
        self.battle_frame_count = 0
        self.last_battle_markov_score = 0.0
        self.last_battle_markov_action = None
        self.in_battle_last_frame = False
        self.battle_action_history = deque(maxlen=100)

        # === BATTLE DATA FROM MEMORY ADDRESSES (EXPANDED) ===
        self.battle_data = DEFAULT_BATTLE_DATA.copy()
        self.prev_battle_data = DEFAULT_BATTLE_DATA.copy()

        # === PARTY DATA (from "pa" field, always available) ===
        self.party_data = DEFAULT_PARTY_DATA.copy()
        self.prev_party_data = DEFAULT_PARTY_DATA.copy()

        # === MENU DATA (from "mu" field — Lua v3) ===
        self.menu_data = DEFAULT_MENU_DATA.copy()
        self.prev_menu_data = DEFAULT_MENU_DATA.copy()

        # === BAG DATA (from "bg" field — Lua v3) ===
        self.bag_data = DEFAULT_BAG_DATA.copy()
        self.prev_bag_data = DEFAULT_BAG_DATA.copy()

        # === RAW GAME STATE (from "gs" field — Lua v3) ===
        self.game_state_raw = 0

        # === BAG THREAD STATE ===
        self.bag_thread_active = False
        self.bag_thread_context = "none"
        self.bag_thread_entered_at = 0
        self.bag_thread_action_count = 0
        self.bag_thread_last_action = None
        self.BAG_THREAD_TIMEOUT = 180
        self.bag_thread_total_actions = 0
        self.bag_thread_markov_actions = 0

        self.bag_transitions = []
        self.bag_metadata = {}
        self.bag_loaded = False
        self.bag_action_history = deque(maxlen=50)
        self.last_bag_markov_score = 0.0
        self.last_bag_markov_action = None

        # === ITEM KNOWLEDGE SYSTEM ===
        self.item_knowledge = {}
        self.item_knowledge_dirty = False

        # === ITEM OBSERVATION STATE ===
        self.pending_item_observation = None
        self.ITEM_OBSERVE_WAIT_FRAMES = 15

        # === EVENT TIMELINE (from teaching code analysis) ===
        # Structured chronological record of human's playthrough:
        # battles, bag sessions, switches, map transitions, with party snapshots.
        # Phase 2: load and query. Phase 3: preparation triggers use this.
        self.event_timeline = []              # list of event dicts (chronological)
        self.event_segments = []              # pre-computed between-target summaries
        self.event_preparation_points = []    # explicit preparation moments
        self.event_timeline_metadata = {}
        self.event_timeline_loaded = False

        # === PARTY MENU THREAD STATE ===
        self.party_menu_active = False
        self.party_menu_context = "none"
        self.party_menu_entered_at = 0
        self.party_menu_target_slot = -1
        self.party_menu_action_count = 0
        self.PARTY_MENU_TIMEOUT = 120

        self.party_menu_battle_cursor_on_entry = -1
        self.party_menu_awaiting_entry = False
        self.party_menu_last_action = None

        # === BATTLE AWARENESS — current battle tracking ===
        self.battle_player_species = -1
        self.battle_enemy_species = -1
        self.battle_start_hp = -1
        self.battle_enemy_start_hp = -1
        self.battle_menu_state = "unknown"
        self.battle_cursor_action_count = 0

        # Per-turn tracking
        self.turn_start_player_hp = -1
        self.turn_start_enemy_hp = -1
        self.turn_start_pp = [-1, -1, -1, -1]
        self.turn_start_enemy_pp = [-1, -1, -1, -1]
        self.turn_start_player_stats = [-1] * 7
        self.turn_start_enemy_stats = [-1] * 7
        self.turn_start_player_status = 0
        self.turn_start_enemy_status = 0
        self.turn_count = 0
        self.last_move_used = -1
        self.last_move_slot = -1

        # === RUNNING DECISION STATE ===
        self.battle_no_damage_turns = 0
        self.battle_hp_trend = deque(maxlen=10)
        self.battle_low_hp_exits = 0
        self.BATTLE_RUN_NO_DAMAGE_THRESHOLD = 4
        self.BATTLE_RUN_HP_RATIO_THRESHOLD = 0.25
        self.BATTLE_RUN_ENABLED = True

        # === FORCED SWITCH STATE ===
        self.forced_switch_pending = False
        self.forced_switch_target_slot = -1

        # === ROSTER SYSTEM ===
        self.roster = {}
        self.roster_dirty = False

        # === MOVE KNOWLEDGE SYSTEM ===
        self.move_knowledge = {}
        self.move_knowledge_dirty = False

        # === ENEMY MOVE KNOWLEDGE ===
        self.enemy_move_knowledge = {}

        # === TAUGHT MODEL REFERENCE ===
        self.taught_reference = {
            'utilities': {},
            'weights': {},
            'loaded': False
        }

        # === BLEND SYSTEM ===
        self.blend_tier = 0
        self.last_blend_timestep = 0
        self.BLEND_COOLDOWN = 50
        self.blend_count = 0

        self.BLEND_RATIOS = {
            1: (0.80, 0.20),
            2: (0.60, 0.40),
            3: (0.40, 0.60)
        }

        self.BLEND_TIER_TRIGGERS = {
            1: {'pattern_repeats': 3, 'pos_stagnation': 8, 'consecutive': 12},
            2: {'pattern_repeats': 6, 'pos_stagnation': 15, 'consecutive': 15},
            3: {'pattern_repeats': 10, 'state_stagnation_mult': 2.0}
        }

        # === ACTION EXECUTION CONFIRMATION ===
        self.pending_action = None
        self.pending_action_frames = 0
        self.ACTION_CONFIRM_FRAMES = 3
        self.last_confirmed_action = None

        # === TILE INTERACTION PROBING ===
        self.INTERACTION_VERIFY_FRAMES = 8
        self.MIN_SUCCESS_RATE_THRESHOLD = 0.1
        self.pending_interaction_verify = None
        self.interaction_verify_countdown = 0

        # === MENU ESCAPE B-BOOST ===
        self.menu_trap_frames = 0
        self.menu_trap_b_boost = 1.0
        self.menu_trap_position = None
        self.B_BOOST_INCREMENT = 0.15
        self.B_BOOST_MAX = 3.0
        self.MENU_TRAP_THRESHOLD = 5
        self.original_b_utility = None

        # === ADAPTIVE MODE SWAPPING ===
        self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD = 15
        self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD = 25
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.THRESHOLD_INCREMENT = 15
        self.MAX_THRESHOLD = 150
        self.frames_in_current_mode = 0
        self.swap_chain_count = 0
        self.position_at_mode_swap = None
        self.last_map_id = None
        self.last_battle_state = None

        # === UNPRODUCTIVE MODE SWAP TRACKING ===
        self.UNPRODUCTIVE_SWAP_THRESHOLD = 3
        self.unproductive_swap_count = 0
        self.utilities_before_swapping = {}
        self.swap_chain_active = False

        # === STATE STAGNATION DETECTION ===
        self.STATE_STAGNATION_THRESHOLD = 20
        self.state_stagnation_count = 0
        self.last_context_state_hash = None
        self.stagnation_initiator_action = None
        self.STAGNATION_INITIATOR_PENALTY = 0.7

        # === "BOTH" MODE THRESHOLDS ===
        self.BOTH_MODE_STAGNATION_THRESHOLD = 35
        self.BOTH_MODE_SWAP_THRESHOLD = 5

        # === TURN AS PROGRESS TRACKING ===
        self.last_direction_for_progress = None
        self.direction_change_counts_as_progress = True

        # === NOVELTY WEIGHTS ===
        self.UNVISITED_TILE_BONUS = 1.5
        self.OBSTRUCTION_PENALTY = 0.25

        # === TRANSITION SYSTEM ===
        self.TRANSITION_ATTRACTION_WEIGHT = 0.6
        self.TEMP_DEBT_ACCUMULATION = 0.5
        self.TEMP_DEBT_DECAY = 0.02
        self.TEMP_DEBT_MAX = 15.0

        # === DEBT CAPS AND DECAY ===
        self.MAX_MAP_DEBT = 10.0
        self.MAX_LOCATION_DEBT = 5.0
        self.DEBT_DECAY_RATE = 0.005

        # === TRANSITION BAN SYSTEM ===
        self.transition_bans = {}
        self.BAN_VICINITY_RADIUS = 3
        self.BAN_COVERAGE_LIFT_THRESHOLD = 0.6
        self.BAN_TIMEOUT_STEPS = 300

        # Multi-scale memory
        self.visited_maps = {}
        self.map_novelty_debt = {}
        self.location_memory = {}
        self.location_novelty = {}
        self.action_execution_count = {}

        self.swap_perceptron = ControlSwapPerceptron()
        self.error_history = deque(maxlen=1000)
        self.numeric_error_history = deque(maxlen=1000)
        self.visual_error_history = deque(maxlen=1000)
        self._entity_norms_cache = {}
        self._cache_valid = False
        self.innate_entities_spawned = False

        # === REPETITION CORRECTION ===
        self.consecutive_action_count = 0
        self.current_repeated_action = None
        self.LEARNING_SLOWDOWN_START = 3
        self.LEARNING_SLOWDOWN_MAX = 10
        self.PENALTY_THRESHOLD = 12
        self.HARD_RESET_THRESHOLD = 18

        # === PATTERN DETECTION ===
        self.PATTERN_CHECK_WINDOW = 50
        self.PATTERN_MIN_REPEATS = 3
        self.PATTERN_MAX_LENGTH = 10
        self.detected_pattern = None
        self.pattern_repeat_count = 0

        # === PROBE ACTION CACHE ===
        self._cached_probe_action = None
        self._cached_probe_dir = None
        self._probe_cache_position = None

        # === NAVIGATION MODE ===
        self.nav_active = False
        self.nav_path = []
        self.nav_path_index = 0
        self.nav_target = None
        self.nav_target_list = []
        self.nav_target_index = 0
        self.nav_struck_targets = set()
        self.nav_steps_taken = 0
        self.nav_stagnation_count = 0
        self.nav_last_position = None

        self.KNOWN_AREA_TRIGGER = 20
        self.known_area_counter = 0
        self.NAV_STAGNATION_LIMIT = 8
        self.NAV_MAX_STEPS = 100
        self.NAV_CURIOSITY_WINDOW = 5
        self.nav_curiosity_countdown = 0
        self.NAV_LEARNING_DAMPENING = 0.3

        # === CROSS-MAP NAVIGATION ===
        self.nav_map_chain = []
        self.nav_chain_index = 0
        self.nav_cross_map_target = None
        self.nav_cross_map_target_data = None
        self.nav_paused = False
        self.nav_paused_reason = ""
        self.nav_paused_target_map = None
        self.NAV_PAUSE_CHECK_INTERVAL = 50
        self.nav_pause_check_countdown = 0
        self.NAV_CROSS_MAP_REFRESH_INTERVAL = 40
        self.nav_cross_map_refresh_countdown = 0
        self._map_graph = {}
        self._map_graph_dirty = True

        # === ENTITY SPAWNING & CLUSTERING ===
        self.ENTITY_INITIAL_CAPACITY = 20
        self.entity_capacity = self.ENTITY_INITIAL_CAPACITY
        self.ENTITY_CAPACITY_GROWTH = 1.5
        self.ENTITY_CLUSTER_SIMILARITY = 0.85
        self.ENTITY_MIN_ACTIVATIONS = 10
        self.entity_spawn_count = 0
        self.entity_merge_count = 0

        # === TAUGHT NAVIGATION TARGETS ===
        self.taught_nav_targets = {}
        self.taught_nav_global_order = []
        self.nav_visited_targets = set()
        self.taught_nav_loaded = False

    # =========================================================================
    # EVENT TIMELINE — structured playthrough history from teaching code
    # =========================================================================

    def load_event_timeline(self, filepath=None):
        """
        Load event timeline from teaching code analysis.
        File: event_timeline.json

        Contains three sections:
        - events: chronological list of battles, bag sessions, switches, etc.
        - segments: pre-computed difficulty between nav targets
        - preparation_points: where the human prepared before hard sections
        """
        filepath = filepath or EVENT_TIMELINE_FILE
        try:
            if not Path(filepath).exists():
                self.event_timeline_loaded = False
                print(f"  📅 No event timeline found at {filepath}")
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            self.event_timeline = data.get('events', [])
            self.event_segments = data.get('segments', [])
            self.event_preparation_points = data.get('preparation_points', [])
            self.event_timeline_metadata = data.get('metadata', {})
            self.event_timeline_loaded = True

            n_events = len(self.event_timeline)
            n_segments = len(self.event_segments)
            n_prep = len(self.event_preparation_points)

            type_counts = {}
            for e in self.event_timeline:
                t = e.get('type', 'unknown')
                type_counts[t] = type_counts.get(t, 0) + 1

            print(f"  📅 Event timeline loaded:")
            print(f"     Events: {n_events} | Segments: {n_segments} | Prep points: {n_prep}")
            if type_counts:
                types_str = ', '.join(f"{k}:{v}" for k, v in type_counts.items())
                print(f"     Types: {types_str}")
            nav_covered = self.event_timeline_metadata.get('nav_targets_covered', [])
            if nav_covered:
                print(f"     Nav targets covered: {nav_covered}")
            total_battles = type_counts.get('battle', 0)
            trainer_battles = sum(1 for e in self.event_timeline
                                  if e.get('type') == 'battle' and
                                  e.get('battle_info', {}).get('battle_type') == 'trainer')
            if total_battles > 0:
                print(f"     Battles: {total_battles} ({trainer_battles} trainer, {total_battles - trainer_battles} wild)")
            if n_prep > 0:
                for pp in self.event_preparation_points[:3]:
                    reason = pp.get('reason', '?')
                    before = pp.get('before_nav_order', '?')
                    threshold = pp.get('party_hp_threshold', '?')
                    print(f"     Prep: before #{before} — {reason} (HP threshold {threshold})")

        except Exception as e:
            print(f"  ⚠️ Error loading event timeline: {e}")
            self.event_timeline = []
            self.event_segments = []
            self.event_preparation_points = []
            self.event_timeline_metadata = {}
            self.event_timeline_loaded = False

    def get_nearest_nav_order(self, raw_x, raw_y, map_id):
        """
        Find which nav target order we're closest to in the journey.
        Uses spatial proximity to taught nav targets on the current map.
        Falls back to highest visited target if no targets on this map.
        Returns order number or -1 if unknown.
        """
        if not self.taught_nav_loaded:
            return -1

        # Check targets on current map
        map_targets = self.taught_nav_targets.get(map_id, [])
        if map_targets:
            best_order = -1
            best_dist = float('inf')
            for t in map_targets:
                pos = t.get('position', [0, 0])
                dist = abs(raw_x - pos[0]) + abs(raw_y - pos[1])
                if dist < best_dist:
                    best_dist = dist
                    best_order = t.get('order', -1)
            return best_order

        # No targets on this map — use highest visited target as estimate
        if self.nav_visited_targets:
            return max(self.nav_visited_targets)

        return -1

    def get_upcoming_events(self, current_nav_order, lookahead=5):
        """
        Get events that occur between current_nav_order and
        current_nav_order + lookahead in the timeline.

        Returns: list of event dicts, sorted by order.
        """
        if not self.event_timeline_loaded or current_nav_order < 0:
            return []

        upcoming = []
        for event in self.event_timeline:
            event_nav = event.get('nav_target_order', -1)
            if event_nav < 0:
                continue
            if current_nav_order <= event_nav <= current_nav_order + lookahead:
                upcoming.append(event)

        upcoming.sort(key=lambda e: (e.get('nav_target_order', 0), e.get('order', 0)))
        return upcoming

    def get_upcoming_battles(self, current_nav_order, lookahead=5):
        """
        Get battle events in the upcoming stretch.
        Returns: list of battle event dicts.
        """
        upcoming = self.get_upcoming_events(current_nav_order, lookahead)
        return [e for e in upcoming if e.get('type') == 'battle']

    def get_upcoming_trainer_battles(self, current_nav_order, lookahead=5):
        """
        Get trainer battle events specifically (can't run from these).
        Returns: list of trainer battle event dicts.
        """
        battles = self.get_upcoming_battles(current_nav_order, lookahead)
        return [b for b in battles
                if b.get('battle_info', {}).get('battle_type') == 'trainer']

    def get_segment_difficulty(self, from_nav_order, to_nav_order):
        """
        Get pre-computed difficulty summary for a segment between nav targets.

        Returns dict with:
        - battles_expected, avg_hp_cost_per_battle, total_hp_cost
        - trainer_battles, wild_battles, bag_sessions, map_transitions
        Or None if segment not found.
        """
        if not self.event_timeline_loaded:
            return None

        for seg in self.event_segments:
            if (seg.get('from_nav_order') == from_nav_order and
                seg.get('to_nav_order') == to_nav_order):
                return seg

        # Try finding a segment that contains the range
        for seg in self.event_segments:
            seg_from = seg.get('from_nav_order', -1)
            seg_to = seg.get('to_nav_order', -1)
            if seg_from <= from_nav_order and seg_to >= to_nav_order:
                return seg

        return None

    def get_preparation_point(self, nav_order):
        """
        Check if there's a preparation point at or just before this nav order.
        The human prepared here — the AI should consider preparing too.

        Returns: preparation_point dict or None.
        """
        if not self.event_timeline_loaded:
            return None

        for pp in self.event_preparation_points:
            before_order = pp.get('before_nav_order', -1)
            # Match if we're at or approaching the prep point
            if before_order >= 0 and abs(nav_order - before_order) <= 1:
                return pp

        return None

    def get_estimated_hp_cost_ahead(self, current_nav_order, lookahead=5):
        """
        Estimate total HP cost for the upcoming stretch based on timeline data.
        Uses segment data if available, falls back to counting battles.

        Returns: estimated total HP ratio cost (0.0 - N.0, can exceed 1.0)
        """
        if not self.event_timeline_loaded:
            return 0.0

        # Try segment data first
        for seg in self.event_segments:
            seg_from = seg.get('from_nav_order', -1)
            seg_to = seg.get('to_nav_order', -1)
            if (seg_from >= 0 and seg_to >= 0 and
                seg_from <= current_nav_order and
                seg_to <= current_nav_order + lookahead):
                return seg.get('total_hp_cost', 0.0)

        # Fallback: sum hp_cost from individual battle events
        battles = self.get_upcoming_battles(current_nav_order, lookahead)
        total_cost = 0.0
        for b in battles:
            total_cost += b.get('battle_info', {}).get('hp_cost', 0.15)
        return total_cost

    def get_timeline_status(self):
        """Return status dict for logging."""
        if not self.event_timeline_loaded:
            return {'loaded': False}

        return {
            'loaded': True,
            'events': len(self.event_timeline),
            'segments': len(self.event_segments),
            'prep_points': len(self.event_preparation_points),
            'nav_covered': self.event_timeline_metadata.get('nav_targets_covered', []),
        }

    # =========================================================================
    # MENU DATA MANAGEMENT (Lua v3 "mu" field)
    # =========================================================================

    def update_menu_data(self, menu_data):
        """Update menu cursor state from Lua "mu" field. Called every frame."""
        self.prev_menu_data = self.menu_data.copy()
        self.menu_data = menu_data.copy()

    # =========================================================================
    # BAG DATA MANAGEMENT (Lua v3 "bg" field)
    # =========================================================================

    def update_bag_data(self, bag_data):
        """Update bag state from Lua "bg" field. Called every frame."""
        self.prev_bag_data = self.bag_data.copy()
        self.bag_data = {
            'pocket': bag_data.get('pocket', -1),
            'cursor': bag_data.get('cursor', -1),
            'active': bag_data.get('active', 0),
            'items': [item.copy() for item in bag_data.get('items', [])],
        }

    # =========================================================================
    # BAG THREAD — entry/exit detection + state management
    # =========================================================================

    def open_bag_thread(self, context):
        """Mark bag thread as active."""
        self.bag_thread_active = True
        self.bag_thread_context = context
        self.bag_thread_entered_at = self.timestep
        self.bag_thread_action_count = 0
        self.bag_thread_last_action = None
        self.bag_action_history.clear()

        ctx_emoji = {"overworld": "🎒", "battle": "🎒⚔️"}.get(context, "🎒")
        pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
        pocket_str = pocket_names.get(self.bag_data.get('pocket', -1), "?")
        n_items = len(self.bag_data.get('items', []))
        print(f"  {ctx_emoji} BAG THREAD OPEN: {context} | pocket={pocket_str} items={n_items}")

    def close_bag_thread(self, reason=""):
        """Mark bag thread as closed."""
        if self.bag_thread_active:
            duration = self.timestep - self.bag_thread_entered_at
            print(f"  🎒 BAG THREAD CLOSED: {reason} (was {self.bag_thread_context}, "
                  f"{duration} frames, {self.bag_thread_action_count} actions)")

        self.bag_thread_active = False
        self.bag_thread_context = "none"
        self.bag_thread_entered_at = 0
        self.bag_thread_action_count = 0
        self.bag_thread_last_action = None

    def is_bag_thread_active(self):
        """Check if bag thread should be in control."""
        return self.bag_thread_active

    def set_bag_thread_last_action(self, action_name):
        """Called from bag_action() to track what we just did."""
        self.bag_thread_last_action = action_name

    def update_bag_thread_state(self, context_state):
        """
        Called every frame. Detects bag entry and exit.
        """
        in_battle = context_state[3] > 0.5
        gs = self.game_state_raw
        bgd = self.bag_data
        prev_bgd = self.prev_bag_data

        if self.bag_thread_active:
            frames_in = self.timestep - self.bag_thread_entered_at
            if frames_in > self.BAG_THREAD_TIMEOUT:
                self.close_bag_thread("timeout")
                return

        if self.bag_thread_active:
            if self.bag_thread_context == "overworld":
                if gs != 14:
                    self.close_bag_thread("gs_left_14")
                    return
            elif self.bag_thread_context == "battle":
                if not in_battle:
                    self.close_bag_thread("battle_ended")
                    return
                if bgd.get('active', 0) == 0 and prev_bgd.get('active', 0) == 0:
                    self.close_bag_thread("bag_deactivated")
                    return
            return

        if self.party_menu_active:
            return

        if not in_battle and gs == 14:
            self.open_bag_thread("overworld")
            return

        if in_battle and bgd.get('active', 0) == 1:
            self.open_bag_thread("battle")
            return

    # =========================================================================
    # ITEM KNOWLEDGE — empirical learning through observation
    # =========================================================================

    def get_item_knowledge(self, item_id):
        """Get or create knowledge entry for an item."""
        if item_id not in self.item_knowledge:
            self.item_knowledge[item_id] = {
                'uses': 0,
                'category': 'unknown',
                'confidence': 0.0,
                'avg_hp_restored': 0.0,
                'total_hp_restored': 0,
                'status_cured': [],
                'catch_attempts': 0,
                'catch_successes': 0,
                'last_used_timestep': 0,
            }
        return self.item_knowledge[item_id]

    def get_item_at_cursor(self):
        """Get the item ID at the current bag cursor position."""
        items = self.bag_data.get('items', [])
        cursor = self.bag_data.get('cursor', -1)
        if 0 <= cursor < len(items):
            return items[cursor].get('id', -1)
        return -1

    def snapshot_party_for_item(self):
        """Take a snapshot of current party state for item effect detection."""
        slots_snapshot = []
        for slot in self.party_data.get('slots', []):
            slots_snapshot.append({
                'hp': slot.get('hp', 0),
                'max_hp': slot.get('max_hp', 0),
                'status': slot.get('status', 0),
            })
        return {
            'slots': slots_snapshot,
            'count': self.party_data.get('count', 0),
        }

    def start_item_observation(self, item_id, target_slot=-1):
        """Begin observing an item use."""
        in_battle = self.battle_data.get('battle_cursor', -1) != -1

        self.pending_item_observation = {
            'item_id': item_id,
            'item_pocket': self.bag_data.get('pocket', -1),
            'target_slot': target_slot,
            'party_snapshot': self.snapshot_party_for_item(),
            'party_count': self.party_data.get('count', 0),
            'in_battle': in_battle,
            'enemy_hp_before': self.battle_data.get('enemy_hp', -1) if in_battle else -1,
            'timestep': self.timestep,
            'frames_waiting': 0,
        }
        print(f"  🎒📝 ITEM OBSERVATION START: item {item_id} (pocket {self.bag_data.get('pocket', -1)}, "
              f"target slot {target_slot})")

    def check_item_observation(self):
        """
        Called every frame. Checks if a pending item observation has resolved.
        Returns True if resolved or timed out.
        """
        if self.pending_item_observation is None:
            return False

        obs = self.pending_item_observation
        obs['frames_waiting'] += 1

        if obs['frames_waiting'] > self.ITEM_OBSERVE_WAIT_FRAMES:
            print(f"  🎒📝 ITEM OBSERVATION TIMEOUT: item {obs['item_id']} — no effect detected")
            self.pending_item_observation = None
            return True

        snapshot = obs['party_snapshot']
        current_slots = self.party_data.get('slots', [])
        current_count = self.party_data.get('count', 0)

        hp_restored = 0
        hp_target_slot = -1
        for i, (snap_slot, curr_slot) in enumerate(zip(snapshot['slots'], current_slots)):
            hp_diff = curr_slot.get('hp', 0) - snap_slot['hp']
            if hp_diff > 0:
                hp_restored = hp_diff
                hp_target_slot = i
                break

        status_cured_value = 0
        status_target_slot = -1
        for i, (snap_slot, curr_slot) in enumerate(zip(snapshot['slots'], current_slots)):
            if snap_slot['status'] != 0 and curr_slot.get('status', 0) == 0:
                status_cured_value = snap_slot['status']
                status_target_slot = i
                break

        caught = current_count > obs['party_count'] and obs['party_count'] > 0

        effect_detected = hp_restored > 0 or status_cured_value != 0 or caught

        if not effect_detected:
            return False

        item_id = obs['item_id']
        target = hp_target_slot if hp_target_slot >= 0 else status_target_slot
        if target < 0:
            target = obs['target_slot']

        self.record_item_observation(
            item_id=item_id,
            hp_restored=hp_restored,
            status_cured=status_cured_value,
            caught=caught,
            target_slot=target,
            in_battle=obs['in_battle'],
        )

        self.pending_item_observation = None
        return True

    def record_item_observation(self, item_id, hp_restored=0, status_cured=0,
                                 caught=False, target_slot=-1, in_battle=False):
        """Record an observed item effect. Updates item knowledge."""
        if item_id <= 0:
            return

        ik = self.get_item_knowledge(item_id)
        ik['uses'] += 1
        ik['last_used_timestep'] = self.timestep

        if hp_restored > 0:
            ik['total_hp_restored'] += hp_restored
            ik['avg_hp_restored'] = ik['total_hp_restored'] / max(1, ik['uses'])

        if status_cured != 0:
            if status_cured not in ik['status_cured']:
                ik['status_cured'].append(status_cured)

        if in_battle:
            ik['catch_attempts'] += 1
            if caught:
                ik['catch_successes'] += 1

        ik['category'] = self._categorize_item(ik)
        ik['confidence'] = min(1.0, ik['uses'] / 5.0)

        self.item_knowledge_dirty = True

        cat_str = ik['category']
        conf_str = f"{ik['confidence']:.0%}"
        effect_parts = []
        if hp_restored > 0:
            effect_parts.append(f"+{hp_restored}HP")
        if status_cured != 0:
            effect_parts.append(f"cured status {status_cured}")
        if caught:
            effect_parts.append("CAUGHT!")
        effect_str = ', '.join(effect_parts) if effect_parts else "no visible effect"

        print(f"  🎒📖 ITEM LEARNED: item {item_id} → {cat_str} ({conf_str}) | {effect_str}")

    def _categorize_item(self, ik):
        """Determine item category from accumulated observations."""
        if ik['uses'] == 0:
            return 'unknown'

        has_hp = ik['total_hp_restored'] > 0
        has_status = len(ik['status_cured']) > 0
        has_catch = ik['catch_successes'] > 0 or (ik['catch_attempts'] >= 2)

        if has_catch:
            return 'catch'
        if has_hp and has_status:
            return 'heal_both'
        if has_hp:
            return 'heal_hp'
        if has_status:
            return 'heal_status'

        if ik['uses'] >= 3 and not has_hp and not has_status and not has_catch:
            return 'other'

        return 'unknown'

    def get_party_hp_ratios(self):
        """Get HP ratios for all party members."""
        ratios = []
        for slot in self.party_data.get('slots', []):
            max_hp = slot.get('max_hp', 0)
            if max_hp > 0:
                ratios.append(slot.get('hp', 0) / max_hp)
            else:
                ratios.append(0.0)
        return ratios

    def get_party_status_flags(self):
        """Get status condition flags for all party members."""
        return [slot.get('status', 0) for slot in self.party_data.get('slots', [])]

    def get_lowest_hp_ratio(self):
        """Get the lowest HP ratio across living party members."""
        ratios = self.get_party_hp_ratios()
        living_ratios = [r for r in ratios if r > 0.0]
        if not living_ratios:
            return 0.0
        return min(living_ratios)

    def has_status_condition_in_party(self):
        """Check if any living party member has a status condition."""
        for slot in self.party_data.get('slots', []):
            if slot.get('hp', 0) > 0 and slot.get('status', 0) != 0:
                return True
        return False

    def get_healing_items(self):
        """Get list of known healing items currently in the bag."""
        items = self.bag_data.get('items', [])
        healing = []
        for item in items:
            item_id = item.get('id', 0)
            qty = item.get('q', 0)
            if item_id <= 0 or qty <= 0:
                continue
            ik = self.item_knowledge.get(item_id)
            if ik and ik['category'] in ('heal_hp', 'heal_status', 'heal_both'):
                healing.append((item_id, qty, ik['category'], ik['confidence']))
        return healing

    def get_catch_items(self):
        """Get list of known catching items currently in the bag."""
        items = self.bag_data.get('items', [])
        catching = []
        for item in items:
            item_id = item.get('id', 0)
            qty = item.get('q', 0)
            if item_id <= 0 or qty <= 0:
                continue
            ik = self.item_knowledge.get(item_id)
            if ik and ik['category'] == 'catch':
                catching.append((item_id, qty, ik['confidence']))
        return catching

    # =========================================================================
    # ITEM KNOWLEDGE PERSISTENCE
    # =========================================================================

    def load_item_knowledge(self, filepath=None):
        """Load item knowledge from disk."""
        filepath = filepath or ITEM_KNOWLEDGE_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                self.item_knowledge = {}
                for k, v in data.items():
                    self.item_knowledge[int(k)] = v
                self.item_knowledge_dirty = False
                total = len(self.item_knowledge)
                known = sum(1 for v in self.item_knowledge.values() if v.get('category', 'unknown') != 'unknown')
                print(f"  🎒📖 Item knowledge loaded: {total} items ({known} categorized)")
                for iid, ik in self.item_knowledge.items():
                    if ik.get('category', 'unknown') != 'unknown':
                        print(f"     Item {iid}: {ik['category']} ({ik['confidence']:.0%} conf, "
                              f"{ik['uses']} uses, avg {ik.get('avg_hp_restored', 0):.0f} HP)")
            else:
                print(f"  🎒📖 No item knowledge file found, starting empty")
        except Exception as e:
            print(f"  ⚠️ Error loading item knowledge: {e}")
            self.item_knowledge = {}

    def save_item_knowledge(self, filepath=None):
        """Save item knowledge to disk."""
        filepath = filepath or ITEM_KNOWLEDGE_FILE
        try:
            save_data = {str(k): v for k, v in self.item_knowledge.items()}
            with open(filepath, 'w') as f:
                json.dump(save_data, f, indent=2)
            self.item_knowledge_dirty = False
        except Exception as e:
            print(f"  ⚠️ Error saving item knowledge: {e}")

    # =========================================================================
    # PARTY DATA MANAGEMENT
    # =========================================================================

    def update_party_data(self, party_data):
        """Update party state from Lua "pa" field. Called every frame."""
        self.prev_party_data = self.party_data.copy()
        self.party_data = party_data.copy()

    def get_party_slot_fingerprint(self, slot_index):
        """Get stat fingerprint for a party slot."""
        slots = self.party_data.get('slots', [])
        if slot_index < 0 or slot_index >= len(slots):
            return None
        s = slots[slot_index]
        return (s['atk'], s['def'], s['spd'], s['spatk'], s['spdef'])

    def get_living_party_slots(self):
        """Return list of (slot_index, slot_data) for party members with HP > 0."""
        living = []
        for i, slot in enumerate(self.party_data.get('slots', [])):
            if slot.get('hp', 0) > 0 and slot.get('max_hp', 0) > 0:
                living.append((i, slot))
        return living

    def get_active_slot_index(self):
        """Try to identify which party slot is currently active in battle."""
        bd = self.battle_data
        if bd['player_hp'] <= 0 and bd['player_max_hp'] <= 0:
            return -1

        for i, slot in enumerate(self.party_data.get('slots', [])):
            if (slot.get('hp', -1) == bd['player_hp'] and
                slot.get('max_hp', -1) == bd['player_max_hp'] and
                slot.get('level', -1) == bd['player_level']):
                return i
        return -1

    # =========================================================================
    # BATTLE DATA MANAGEMENT (EXPANDED)
    # =========================================================================

    def update_battle_data(self, battle_data):
        """Update battle state from Lua memory reads. Called every frame."""
        self.prev_battle_data = self.battle_data.copy()
        prev_pss = self.battle_data.get('player_stat_stages')
        prev_ess = self.battle_data.get('enemy_stat_stages')
        if isinstance(prev_pss, list):
            self.prev_battle_data['player_stat_stages'] = list(prev_pss)
        if isinstance(prev_ess, list):
            self.prev_battle_data['enemy_stat_stages'] = list(prev_ess)

        self.battle_data = battle_data.copy()
        pss = battle_data.get('player_stat_stages')
        ess = battle_data.get('enemy_stat_stages')
        if isinstance(pss, list):
            self.battle_data['player_stat_stages'] = list(pss)
        if isinstance(ess, list):
            self.battle_data['enemy_stat_stages'] = list(ess)

    def on_battle_start_with_data(self):
        """Called when battle starts AND we have battle data available."""
        bd = self.battle_data

        if bd['player_species'] > 0:
            self.battle_player_species = bd['player_species']
        if bd['enemy_species'] > 0:
            self.battle_enemy_species = bd['enemy_species']
        if bd['player_hp'] > 0:
            self.battle_start_hp = bd['player_hp']
        if bd['enemy_hp'] > 0:
            self.battle_enemy_start_hp = bd['enemy_hp']

        self.battle_menu_state = "unknown"
        self.battle_cursor_action_count = 0

        self.turn_count = 0
        self.turn_start_player_hp = bd['player_hp']
        self.turn_start_enemy_hp = bd['enemy_hp']
        self.turn_start_pp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        self.turn_start_enemy_pp = [bd['enemy_pp0'], bd['enemy_pp1'],
                                     bd['enemy_pp2'], bd['enemy_pp3']]
        self.turn_start_player_stats = list(bd.get('player_stat_stages', [-1]*7))
        self.turn_start_enemy_stats = list(bd.get('enemy_stat_stages', [-1]*7))
        self.turn_start_player_status = bd.get('player_status', 0)
        self.turn_start_enemy_status = bd.get('enemy_status', 0)
        self.last_move_used = -1
        self.last_move_slot = -1

        self.battle_no_damage_turns = 0
        self.battle_hp_trend.clear()

        self.close_party_menu("battle_start")
        self.close_bag_thread("battle_start")
        self.forced_switch_pending = False
        self.forced_switch_target_slot = -1

        self.update_roster_from_battle()

    def on_battle_end_with_data(self):
        """Called when battle ends. Returns outcome string."""
        outcome = 'unknown'

        if self.battle_start_hp > 0:
            last_player_hp = self.prev_battle_data.get('player_hp', -1)
            last_enemy_hp = self.prev_battle_data.get('enemy_hp', -1)

            if last_enemy_hp == 0:
                outcome = 'win'
            elif last_player_hp == 0:
                outcome = 'loss'
            elif last_player_hp > 0 and last_enemy_hp > 0:
                outcome = 'run'

        prev_count = self.prev_party_data.get('count', 0)
        curr_count = self.party_data.get('count', 0)
        if curr_count > prev_count and prev_count > 0:
            outcome = 'catch'
            print(f"  🎊 CATCH DETECTED! Party {prev_count} → {curr_count}")

        if self.battle_start_hp > 0:
            last_hp = self.prev_battle_data.get('player_hp', -1)
            if last_hp > 0 and self.battle_start_hp > 0:
                exit_ratio = last_hp / self.battle_start_hp
                if exit_ratio < 0.3:
                    self.battle_low_hp_exits += 1
                else:
                    self.battle_low_hp_exits = max(0, self.battle_low_hp_exits - 1)

        self.battle_player_species = -1
        self.battle_enemy_species = -1
        self.battle_start_hp = -1
        self.battle_enemy_start_hp = -1
        self.battle_menu_state = "unknown"
        self.battle_cursor_action_count = 0

        self.turn_count = 0
        self.turn_start_player_hp = -1
        self.turn_start_enemy_hp = -1
        self.turn_start_pp = [-1, -1, -1, -1]
        self.turn_start_enemy_pp = [-1, -1, -1, -1]
        self.turn_start_player_stats = [-1] * 7
        self.turn_start_enemy_stats = [-1] * 7
        self.turn_start_player_status = 0
        self.turn_start_enemy_status = 0
        self.last_move_used = -1
        self.last_move_slot = -1
        self.battle_no_damage_turns = 0

        self.close_party_menu("battle_end")
        self.close_bag_thread("battle_end")
        self.forced_switch_pending = False
        self.forced_switch_target_slot = -1

        return outcome

    def has_battle_data(self):
        """Check if battle data is available."""
        return (self.battle_data.get('battle_cursor', -1) != -1 or
                self.battle_data.get('player_hp', -1) != -1)

    def is_trainer_battle(self):
        """Check if current battle is a trainer battle."""
        bt = self.battle_data.get('battle_type', 0)
        return (bt & 8) != 0

    def infer_battle_menu_state(self):
        """Returns: 'main_menu', 'move_select', or 'unknown'."""
        bd = self.battle_data
        prev = self.prev_battle_data

        if not self.has_battle_data():
            return "unknown"
        if self.party_menu_active:
            return "unknown"
        if self.bag_thread_active:
            return "unknown"

        bc = bd['battle_cursor']
        mc = bd['move_cursor']
        prev_bc = prev['battle_cursor']
        prev_mc = prev['move_cursor']

        if 0 <= bc <= 3 and bc != prev_bc:
            return "main_menu"
        if 0 <= mc <= 3 and mc != prev_mc:
            return "move_select"
        if 0 <= bc <= 3:
            return "main_menu"

        return "unknown"

    # =========================================================================
    # PARTY MENU THREAD
    # =========================================================================

    def open_party_menu(self, context, target_slot=-1):
        """Mark party menu as active."""
        self.party_menu_active = True
        self.party_menu_context = context
        self.party_menu_entered_at = self.timestep
        self.party_menu_target_slot = target_slot
        self.party_menu_action_count = 0
        self.party_menu_last_action = None

        if context == "battle_forced":
            self.forced_switch_pending = True
            self.forced_switch_target_slot = target_slot

        ctx_emoji = {"battle_voluntary": "🔄", "battle_forced": "🔄⚠️", "overworld": "📋"}.get(context, "❓")
        target_str = f" → slot {target_slot}" if target_slot >= 0 else ""
        print(f"  {ctx_emoji} PARTY MENU OPEN: {context}{target_str}")

    def close_party_menu(self, reason=""):
        """Mark party menu as closed."""
        if self.party_menu_active:
            duration = self.timestep - self.party_menu_entered_at
            print(f"  📋 PARTY MENU CLOSED: {reason} (was {self.party_menu_context}, "
                  f"{duration} frames, {self.party_menu_action_count} actions)")

        self.party_menu_active = False
        self.party_menu_context = "none"
        self.party_menu_entered_at = 0
        self.party_menu_target_slot = -1
        self.party_menu_action_count = 0
        self.party_menu_awaiting_entry = False
        self.party_menu_battle_cursor_on_entry = -1
        self.party_menu_last_action = None

    def is_party_menu_active(self):
        return self.party_menu_active

    def update_party_menu_state(self, context_state):
        """Called every frame. Detects party menu entry and exit."""
        in_battle = context_state[3] > 0.5
        bd = self.battle_data
        prev_bd = self.prev_battle_data
        md = self.menu_data
        gs = self.game_state_raw

        if self.party_menu_active:
            frames_in = self.timestep - self.party_menu_entered_at
            if frames_in > self.PARTY_MENU_TIMEOUT:
                self.close_party_menu("timeout")
                return

        if self.party_menu_active:
            if in_battle:
                bc = bd.get('battle_cursor', -1)
                if 0 <= bc <= 3:
                    prev_bc = prev_bd.get('battle_cursor', -1)
                    if 0 <= prev_bc <= 3:
                        self.close_party_menu("returned_to_battle_menu")
                        return
            else:
                if self.party_menu_context in ("battle_voluntary", "battle_forced"):
                    self.close_party_menu("battle_ended")
                    return

                if self.party_menu_context == "overworld":
                    pc = md.get('pc', -1)
                    if gs != 1:
                        self.close_party_menu("gs_changed")
                        return
                    if not (0 <= pc <= 6):
                        self.close_party_menu("party_cursor_invalid")
                        return

                if context_state[4] <= 0.5 and self.party_menu_context == "overworld":
                    self.close_party_menu("menu_closed")
                    return
            return

        if in_battle and self.has_battle_data():
            bc = bd.get('battle_cursor', -1)
            prev_bc = prev_bd.get('battle_cursor', -1)

            if bd['player_hp'] == 0 and prev_bd.get('player_hp', -1) > 0:
                target = self.get_best_switch_slot()
                self.open_party_menu("battle_forced", target_slot=target)
                return

            if self.party_menu_awaiting_entry:
                self.party_menu_awaiting_entry = False
                if not (0 <= bc <= 3):
                    self.open_party_menu("battle_voluntary")
                    return
                return

            if (self.party_menu_last_action == "A" and
                prev_bc == 2 and
                not (0 <= bc <= 3)):
                self.open_party_menu("battle_voluntary")
                return

        if not in_battle:
            pc = md.get('pc', -1)
            prev_pc = self.prev_menu_data.get('pc', -1)

            if gs == 1 and 0 <= pc <= 5:
                prev_gs_was_party = (self.prev_menu_data.get('pc', -1) >= 0 and
                                     self.prev_menu_data.get('pc', -1) <= 5)
                if not prev_gs_was_party or prev_pc < 0 or prev_pc > 5:
                    self.open_party_menu("overworld")
                    return

    def set_party_menu_last_action(self, action_name):
        self.party_menu_last_action = action_name

    # =========================================================================
    # FORCED SWITCH
    # =========================================================================

    def detect_forced_switch(self):
        return (self.party_menu_active and
                self.party_menu_context == "battle_forced")

    def get_best_switch_slot(self):
        """Pick living party member with highest HP."""
        living = self.get_living_party_slots()
        if not living:
            return -1

        active_slot = self.get_active_slot_index()
        candidates = [(i, s) for i, s in living if i != active_slot]

        if not candidates:
            return living[0][0] if living else -1

        best_slot = max(candidates, key=lambda x: x[1].get('hp', 0))
        return best_slot[0]

    # =========================================================================
    # RUNNING DECISION
    # =========================================================================

    def should_run(self):
        """Determine if the AI should run from the current battle."""
        if not self.BATTLE_RUN_ENABLED:
            return False
        if self.is_trainer_battle():
            return False

        bd = self.battle_data

        if self.battle_no_damage_turns >= self.BATTLE_RUN_NO_DAMAGE_THRESHOLD:
            return True

        if bd['player_hp'] > 0 and bd['player_max_hp'] > 0:
            hp_ratio = bd['player_hp'] / bd['player_max_hp']
            if hp_ratio < self.BATTLE_RUN_HP_RATIO_THRESHOLD:
                return True

        if len(self.battle_hp_trend) >= 3 and self.battle_no_damage_turns >= 2:
            trend = list(self.battle_hp_trend)
            if all(trend[i] >= trend[i+1] for i in range(len(trend)-1)):
                return True

        if self.battle_low_hp_exits >= 3:
            return True

        return False

    # =========================================================================
    # TURN TRACKING
    # =========================================================================

    def detect_turn_resolved(self):
        """Detect if a battle turn just resolved."""
        bd = self.battle_data
        if bd['player_hp'] < 0 or self.turn_start_player_hp < 0:
            return False

        current_pp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        pp_decreased = False
        for i in range(4):
            if self.turn_start_pp[i] > 0 and current_pp[i] >= 0:
                if current_pp[i] < self.turn_start_pp[i]:
                    pp_decreased = True
                    break

        player_hp_changed = (bd['player_hp'] != self.turn_start_player_hp and
                             self.turn_start_player_hp >= 0)
        enemy_hp_changed = (bd['enemy_hp'] != self.turn_start_enemy_hp and
                            self.turn_start_enemy_hp >= 0)

        return pp_decreased or (player_hp_changed and enemy_hp_changed)

    def on_battle_turn_end(self):
        """Called when a turn resolves."""
        bd = self.battle_data
        self.turn_count += 1

        current_pp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        used_slot = -1
        for i in range(4):
            if self.turn_start_pp[i] > 0 and current_pp[i] >= 0:
                if current_pp[i] < self.turn_start_pp[i]:
                    used_slot = i
                    break

        move_id = -1
        if used_slot >= 0:
            move_keys = ['move0', 'move1', 'move2', 'move3']
            move_id = bd.get(move_keys[used_slot], -1)

        damage_dealt = 0
        if self.turn_start_enemy_hp >= 0 and bd['enemy_hp'] >= 0:
            damage_dealt = max(0, self.turn_start_enemy_hp - bd['enemy_hp'])

        missed = False
        if used_slot >= 0 and damage_dealt == 0:
            enemy_stat_changed = False
            curr_estats = bd.get('enemy_stat_stages', [-1]*7)
            for j in range(7):
                if (self.turn_start_enemy_stats[j] >= 0 and curr_estats[j] >= 0 and
                    curr_estats[j] != self.turn_start_enemy_stats[j]):
                    enemy_stat_changed = True
                    break

            status_inflicted = (bd.get('enemy_status', 0) != self.turn_start_enemy_status and
                                bd.get('enemy_status', 0) != 0)

            player_stat_changed = False
            curr_pstats = bd.get('player_stat_stages', [-1]*7)
            for j in range(7):
                if (self.turn_start_player_stats[j] >= 0 and curr_pstats[j] >= 0 and
                    curr_pstats[j] != self.turn_start_player_stats[j]):
                    player_stat_changed = True
                    break

            if not enemy_stat_changed and not status_inflicted and not player_stat_changed:
                missed = True

        if move_id > 0 and self.battle_enemy_species > 0:
            self_stat_changes = 0
            enemy_stat_changes = 0
            curr_pstats = bd.get('player_stat_stages', [-1]*7)
            curr_estats = bd.get('enemy_stat_stages', [-1]*7)
            for j in range(7):
                if (self.turn_start_player_stats[j] >= 0 and curr_pstats[j] >= 0 and
                    curr_pstats[j] != self.turn_start_player_stats[j]):
                    self_stat_changes += 1
                if (self.turn_start_enemy_stats[j] >= 0 and curr_estats[j] >= 0 and
                    curr_estats[j] != self.turn_start_enemy_stats[j]):
                    enemy_stat_changes += 1

            status_inflicted = (bd.get('enemy_status', 0) != self.turn_start_enemy_status and
                                bd.get('enemy_status', 0) != 0)

            self.record_move_result(
                move_id, self.battle_enemy_species,
                damage_dealt, missed,
                self_stat_changes, enemy_stat_changes,
                1 if status_inflicted else 0
            )

        enemy_move_id = self.detect_enemy_move()
        if enemy_move_id > 0:
            damage_taken = 0
            if self.turn_start_player_hp >= 0 and bd['player_hp'] >= 0:
                damage_taken = max(0, self.turn_start_player_hp - bd['player_hp'])

            player_got_status = (bd.get('player_status', 0) != self.turn_start_player_status and
                                 bd.get('player_status', 0) != 0)

            self.record_enemy_move_observation(
                enemy_move_id, damage_taken,
                1 if player_got_status else 0, 0
            )

        if damage_dealt == 0 and used_slot >= 0:
            self.battle_no_damage_turns += 1
        else:
            self.battle_no_damage_turns = 0

        if bd['player_hp'] > 0:
            self.battle_hp_trend.append(bd['player_hp'])

        if (self.battle_enemy_species > 0 and bd['enemy_species'] > 0 and
            bd['enemy_species'] != self.battle_enemy_species):
            print(f"  ⚔️ ENEMY SWITCHED: {self.battle_enemy_species} → {bd['enemy_species']}")
            self.battle_enemy_species = bd['enemy_species']
            self.battle_enemy_start_hp = bd['enemy_hp']

        self.turn_start_player_hp = bd['player_hp']
        self.turn_start_enemy_hp = bd['enemy_hp']
        self.turn_start_pp = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
        self.turn_start_enemy_pp = [bd['enemy_pp0'], bd['enemy_pp1'],
                                     bd['enemy_pp2'], bd['enemy_pp3']]
        self.turn_start_player_stats = list(bd.get('player_stat_stages', [-1]*7))
        self.turn_start_enemy_stats = list(bd.get('enemy_stat_stages', [-1]*7))
        self.turn_start_player_status = bd.get('player_status', 0)
        self.turn_start_enemy_status = bd.get('enemy_status', 0)

    def detect_enemy_move(self):
        """Detect which enemy move was used by checking PP decrements."""
        bd = self.battle_data
        curr_epp = [bd['enemy_pp0'], bd['enemy_pp1'], bd['enemy_pp2'], bd['enemy_pp3']]
        emove_keys = ['enemy_move0', 'enemy_move1', 'enemy_move2', 'enemy_move3']

        for i in range(4):
            if self.turn_start_enemy_pp[i] > 0 and curr_epp[i] >= 0:
                if curr_epp[i] < self.turn_start_enemy_pp[i]:
                    return bd.get(emove_keys[i], -1)
        return -1

    # =========================================================================
    # ROSTER SYSTEM
    # =========================================================================

    def update_roster_from_battle(self):
        """Match battle mirror data to party slot, update roster."""
        bd = self.battle_data
        if bd['player_species'] <= 0:
            return

        species = bd['player_species']
        moves = [bd['move0'], bd['move1'], bd['move2'], bd['move3']]
        level = bd.get('player_level', -1)

        active_slot = self.get_active_slot_index()

        if active_slot < 0:
            for i, slot in enumerate(self.party_data.get('slots', [])):
                if (slot.get('level', -1) == level and
                    slot.get('max_hp', -1) == bd['player_max_hp']):
                    active_slot = i
                    break

        if active_slot < 0:
            return

        fingerprint = self.get_party_slot_fingerprint(active_slot)

        self.roster[active_slot] = {
            'species': species,
            'moves': [m for m in moves if m > 0],
            'fingerprint': fingerprint,
            'level': level,
            'last_updated': self.timestep
        }
        self.roster_dirty = True

    def get_roster_moves_for_slot(self, slot_index):
        entry = self.roster.get(slot_index)
        if entry is None:
            return []
        return entry.get('moves', [])

    def get_move_slot_index(self, move_id):
        bd = self.battle_data
        move_keys = ['move0', 'move1', 'move2', 'move3']
        for i, key in enumerate(move_keys):
            if bd.get(key, -1) == move_id:
                return i
        return -1

    # =========================================================================
    # MOVE KNOWLEDGE SYSTEM
    # =========================================================================

    def record_move_result(self, move_id, enemy_species, damage, missed,
                           self_stat_changes, enemy_stat_changes, status_inflicted):
        if move_id <= 0:
            return

        mk = self.move_knowledge
        if move_id not in mk:
            mk[move_id] = {
                'total_uses': 0, 'total_damage': 0, 'avg_damage': 0.0,
                'misses': 0, 'vs_species': {}
            }

        entry = mk[move_id]
        entry['total_uses'] += 1
        entry['total_damage'] += damage
        entry['avg_damage'] = entry['total_damage'] / max(1, entry['total_uses'])
        if missed:
            entry['misses'] += 1

        sp_key = enemy_species
        if sp_key not in entry['vs_species']:
            entry['vs_species'][sp_key] = {
                'uses': 0, 'damage': 0, 'avg': 0.0, 'misses': 0,
                'stat_changes_self': 0, 'stat_changes_enemy': 0,
                'status_inflicted': 0
            }

        vs = entry['vs_species'][sp_key]
        vs['uses'] += 1
        vs['damage'] += damage
        vs['avg'] = vs['damage'] / max(1, vs['uses'])
        if missed:
            vs['misses'] += 1
        vs['stat_changes_self'] += self_stat_changes
        vs['stat_changes_enemy'] += enemy_stat_changes
        vs['status_inflicted'] += status_inflicted

        self.move_knowledge_dirty = True

    def record_enemy_move_observation(self, enemy_move_id, damage_to_us,
                                       status_inflicted, stat_changes):
        if enemy_move_id <= 0:
            return

        emk = self.enemy_move_knowledge
        if enemy_move_id not in emk:
            emk[enemy_move_id] = {
                'observed_uses': 0, 'damage_to_us': 0,
                'status_inflicted': 0, 'stat_changes': 0
            }

        entry = emk[enemy_move_id]
        entry['observed_uses'] += 1
        entry['damage_to_us'] += damage_to_us
        entry['status_inflicted'] += status_inflicted
        entry['stat_changes'] += stat_changes

    def get_best_move_for_enemy(self, enemy_species):
        bd = self.battle_data
        move_ids = [bd['move0'], bd['move1'], bd['move2'], bd['move3']]
        pp_vals = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]

        candidates = []
        for slot_idx in range(4):
            mid = move_ids[slot_idx]
            pp = pp_vals[slot_idx]
            if mid <= 0 or pp == 0:
                continue

            score = 1.0
            mk = self.move_knowledge.get(mid)
            if mk:
                vs = mk.get('vs_species', {}).get(enemy_species)
                if vs and vs['uses'] >= 2:
                    score = vs['avg'] * 10.0
                    if vs['uses'] > 0:
                        miss_rate = vs['misses'] / vs['uses']
                        score *= (1.0 - miss_rate * 0.5)
                    if vs['status_inflicted'] > 0:
                        score += 2.0
                    if vs['stat_changes_enemy'] > 0:
                        score += 1.0
                elif mk['total_uses'] >= 2:
                    score = mk['avg_damage'] * 8.0
                    if mk['total_uses'] > 0:
                        miss_rate = mk['misses'] / mk['total_uses']
                        score *= (1.0 - miss_rate * 0.5)

            if pp <= 3 and pp > 0:
                score *= 0.8
            candidates.append((mid, slot_idx, score))

        candidates.sort(key=lambda x: x[2], reverse=True)
        return candidates

    def get_moves_with_pp(self):
        bd = self.battle_data
        moves = []
        for i, (mk, pk) in enumerate(zip(
            ['move0', 'move1', 'move2', 'move3'],
            ['pp0', 'pp1', 'pp2', 'pp3']
        )):
            mid = bd.get(mk, -1)
            pp = bd.get(pk, -1)
            if mid > 0 and pp > 0:
                moves.append((i, mid, pp))
        return moves

    # =========================================================================
    # ROSTER & MOVE KNOWLEDGE PERSISTENCE
    # =========================================================================

    def load_roster(self, filepath=None):
        filepath = filepath or ROSTER_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                self.roster = {}
                for k, v in data.items():
                    self.roster[int(k)] = v
                    if isinstance(v.get('fingerprint'), list):
                        v['fingerprint'] = tuple(v['fingerprint'])
                self.roster_dirty = False
                print(f"  📋 Roster loaded: {len(self.roster)} slots")
                for slot, entry in self.roster.items():
                    moves_str = ','.join(str(m) for m in entry.get('moves', []))
                    print(f"     Slot {slot}: species {entry.get('species')} "
                          f"Lv{entry.get('level')} moves=[{moves_str}]")
            else:
                print(f"  📋 No roster file found, starting empty")
        except Exception as e:
            print(f"  ⚠️ Error loading roster: {e}")
            self.roster = {}

    def save_roster(self, filepath=None):
        filepath = filepath or ROSTER_FILE
        try:
            save_data = {}
            for k, v in self.roster.items():
                entry = v.copy()
                if isinstance(entry.get('fingerprint'), tuple):
                    entry['fingerprint'] = list(entry['fingerprint'])
                save_data[str(k)] = entry
            with open(filepath, 'w') as f:
                json.dump(save_data, f, indent=2)
            self.roster_dirty = False
        except Exception as e:
            print(f"  ⚠️ Error saving roster: {e}")

    def load_move_knowledge(self, filepath=None):
        filepath = filepath or MOVE_KNOWLEDGE_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                self.move_knowledge = {}
                for mk, mv in data.get('player_moves', {}).items():
                    mid = int(mk)
                    vs = {}
                    for sk, sv in mv.get('vs_species', {}).items():
                        vs[int(sk)] = sv
                    mv['vs_species'] = vs
                    self.move_knowledge[mid] = mv

                self.enemy_move_knowledge = {}
                for ek, ev in data.get('enemy_moves', {}).items():
                    self.enemy_move_knowledge[int(ek)] = ev

                self.move_knowledge_dirty = False
                total_moves = len(self.move_knowledge)
                total_uses = sum(m.get('total_uses', 0) for m in self.move_knowledge.values())
                print(f"  📖 Move knowledge loaded: {total_moves} moves, {total_uses} total uses")
                print(f"     Enemy moves observed: {len(self.enemy_move_knowledge)}")
            else:
                print(f"  📖 No move knowledge file found, starting empty")
        except Exception as e:
            print(f"  ⚠️ Error loading move knowledge: {e}")
            self.move_knowledge = {}
            self.enemy_move_knowledge = {}

    def save_move_knowledge(self, filepath=None):
        filepath = filepath or MOVE_KNOWLEDGE_FILE
        try:
            save_data = {
                'player_moves': {str(k): v for k, v in self.move_knowledge.items()},
                'enemy_moves': {str(k): v for k, v in self.enemy_move_knowledge.items()}
            }
            for mk, mv in save_data['player_moves'].items():
                vs = mv.get('vs_species', {})
                mv['vs_species'] = {str(sk): sv for sk, sv in vs.items()}

            with open(filepath, 'w') as f:
                json.dump(save_data, f, indent=2)
            self.move_knowledge_dirty = False
        except Exception as e:
            print(f"  ⚠️ Error saving move knowledge: {e}")


# ============================================================================
# CELL 3 PART 2: Taught Reference, Blend System, Markov (OW + Battle + Bag)
# ============================================================================
# CHANGES from v2:
# 1. NEW: load_taught_bag_transitions() — loads bag demonstration frames
# 2. NEW: compute_bag_markov_similarity() — bag-specific state matching
# 3. NEW: get_bag_markov_action() — returns best bag action from Markov
# 4. All overworld + battle Markov methods UNCHANGED
# 5. All blend system methods UNCHANGED
# ============================================================================

    # =========================================================================
    # TAUGHT MODEL REFERENCE
    # =========================================================================
    
    def load_taught_reference(self, filepath):
        """
        Load taught model as a READ-ONLY reference for stagnation blending.
        Does NOT overwrite AI's own utilities or weights.
        """
        try:
            if not Path(filepath).exists():
                print(f"  No taught reference model found at {filepath}")
                return
            
            with open(filepath, 'r') as f:
                model = json.load(f)
            
            if "perceptrons" not in model:
                print(f"  ⚠️ Taught reference model empty or invalid")
                return
            
            for saved_action in model["perceptrons"].get("actions", []):
                action_name = saved_action.get("action")
                if action_name:
                    self.taught_reference['utilities'][action_name] = saved_action.get("utility", 1.0)
                    
                    if saved_action.get("weights_nonzero"):
                        dim = saved_action.get("weights_shape", 1376)
                        w = np.zeros(dim)
                        for idx, val in saved_action["weights_nonzero"]:
                            if idx < dim:
                                w[idx] = val
                        self.taught_reference['weights'][action_name] = w
            
            self.taught_reference['loaded'] = True
            
            print(f"  📖 Taught reference loaded:")
            print(f"     Actions: {list(self.taught_reference['utilities'].keys())}")
            print(f"     Utilities: {', '.join(f'{k}:{v:.3f}' for k, v in self.taught_reference['utilities'].items())}")
            print(f"     Weights available: {list(self.taught_reference['weights'].keys())}")
            
        except Exception as e:
            print(f"  ⚠️ Error loading taught reference: {e}")
    
    def blend_from_taught(self, tier):
        """
        Blend AI's current utilities (and optionally weights) toward taught values.
        
        tier 1 (light):  80% AI / 20% taught — utilities only
        tier 2 (medium): 60% AI / 40% taught — utilities only
        tier 3 (hard):   40% AI / 60% taught — utilities + weights
        """
        if not self.taught_reference['loaded']:
            return
        
        if tier not in self.BLEND_RATIOS:
            return
        
        if self.timestep - self.last_blend_timestep < self.BLEND_COOLDOWN:
            return
        
        ai_weight, taught_weight = self.BLEND_RATIOS[tier]
        blend_weights = (tier == 3)
        
        blended_actions = []
        
        for a in self.actions():
            if a.action not in self.taught_reference['utilities']:
                continue
            
            taught_util = self.taught_reference['utilities'][a.action]
            old_util = a.utility
            
            a.utility = ai_weight * a.utility + taught_weight * taught_util
            
            if taught_util > 1.0:
                a.utility = max(a.utility, taught_util * 0.5)
            
            floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
            a.utility = max(a.utility, floor)
            a.utility = min(a.utility, 2.0)
            
            blended_actions.append(f"{a.action}:{old_util:.3f}→{a.utility:.3f}")
            
            if blend_weights and a.action in self.taught_reference['weights']:
                taught_w = self.taught_reference['weights'][a.action]
                if a.weights is not None:
                    min_dim = min(len(a.weights), len(taught_w))
                    a.weights[:min_dim] = (
                        ai_weight * a.weights[:min_dim] + 
                        taught_weight * taught_w[:min_dim]
                    )
        
        self.last_blend_timestep = self.timestep
        self.blend_tier = tier
        self.blend_count += 1
        
        tier_names = {1: "LIGHT", 2: "MEDIUM", 3: "HARD"}
        print(f"  🔀 BLEND [{tier_names.get(tier, '?')}] ({ai_weight:.0%} AI / {taught_weight:.0%} taught)"
              f" | Blend #{self.blend_count}")
        for ba in blended_actions:
            print(f"     {ba}")
        if blend_weights:
            print(f"     + Weights blended for: {list(self.taught_reference['weights'].keys())}")

    # =========================================================================
    # OVERWORLD MARKOV TRANSITION SYSTEM
    # =========================================================================
    
    def load_taught_transitions(self, filepath=None):
        filepath = filepath or TAUGHT_TRANSITIONS_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                self.taught_transitions = []
                self.taught_batches = data.get('batches', [])
                
                for batch in self.taught_batches:
                    batch_type = batch.get('batch_type', 'steady')
                    trigger_action = batch.get('trigger_action')
                    
                    for frame in batch.get('frames', []):
                        transition = {
                            'state': frame.get('state', {}),
                            'action': frame.get('action'),
                            'recent_actions': frame.get('recent_actions', []),
                            'frame_offset': frame.get('frame_offset', 0),
                            'batch_type': batch_type,
                            'trigger_action': trigger_action
                        }
                        self.taught_transitions.append(transition)
                
                self.taught_metadata = data.get('metadata', {})
                
                print(f"  📚 Loaded taught transitions:")
                print(f"     Batches: {len(self.taught_batches)}")
                print(f"     Frames: {len(self.taught_transitions)}")
                print(f"     Action changes: {self.taught_metadata.get('action_changes', 0)}")
                print(f"     Maps visited: {self.taught_metadata.get('maps_visited', [])}")
            else:
                self.taught_transitions = []
                self.taught_batches = []
                self.taught_metadata = {}
                print(f"  No taught transitions file found at {filepath}")
        except Exception as e:
            print(f"  Error loading taught transitions: {e}")
            self.taught_transitions = []
            self.taught_batches = []
            self.taught_metadata = {}
    
    def extract_partial_context(self, context_state, raw_position=None):
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_map = int(context_state[2])
        
        movement_blocked = self.get_position_stagnation() > 3
        
        near_transition = False
        memory = self.get_current_map_memory(current_map)
        for t in memory.get('transitions', []):
            t_pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
            if abs(raw_x - t_pos[0]) + abs(raw_y - t_pos[1]) <= 2:
                near_transition = True
                break
        
        tile_probed = not self.should_interact_at_tile(raw_x, raw_y, current_map)
        
        return {
            'in_battle': context_state[3] > 0.5,
            'in_menu': context_state[4] > 0.5,
            'movement_blocked': movement_blocked,
            'near_transition': near_transition,
            'tile_probed': tile_probed
        }
    
    def compute_markov_similarity(self, context_state, raw_position=None, taught_frames=None):
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        skip_map_check = taught_frames is not None
        
        if not frames:
            return 0.0, None, -1
        
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_map = int(context_state[2])
        current_dir = int(context_state[5])
        in_battle = context_state[3] > 0.5
        in_menu = context_state[4] > 0.5
        
        current_actions = list(self.action_history)
        current_partial = self.extract_partial_context(context_state, raw_position)
        
        best_score = 0.0
        best_action = None
        best_idx = -1
        
        for idx, transition in enumerate(frames):
            t_state = transition.get('state', {})
            t_action = transition.get('action')
            t_recent = transition.get('recent_actions', [])
            batch_type = transition.get('batch_type', 'steady')
            
            if not t_action or t_action == "NONE":
                continue
            
            immediate_score = 0.0
            
            if not skip_map_check:
                if t_state.get('map_id') != current_map:
                    continue
            immediate_score += 0.25
            
            t_x = t_state.get('x', 0)
            t_y = t_state.get('y', 0)
            pos_dist = abs(raw_x - t_x) + abs(raw_y - t_y)
            
            if pos_dist == 0:
                immediate_score += MARKOV_POS_EXACT_BONUS
            elif pos_dist <= 2:
                immediate_score += MARKOV_POS_NEAR_BONUS
            elif pos_dist <= MARKOV_POS_MAX_DIST:
                immediate_score += MARKOV_POS_FAR_BONUS
            else:
                continue
            
            if t_state.get('direction') == current_dir:
                immediate_score += 0.2
            
            t_in_battle = t_state.get('in_battle', 0) == 1
            t_in_menu = t_state.get('in_menu', 0) == 1
            
            if t_in_battle == in_battle:
                immediate_score += 0.1
            if t_in_menu == in_menu:
                immediate_score += 0.1
            
            sequential_score = 0.0
            
            if t_recent and current_actions:
                if len(current_actions) >= 8 and len(t_recent) >= 8:
                    if list(current_actions)[-8:] == t_recent[-8:]:
                        sequential_score = MARKOV_SEQ_FULL_WEIGHT
                
                if sequential_score < MARKOV_SEQ_MEDIUM_WEIGHT:
                    if len(current_actions) >= 5 and len(t_recent) >= 5:
                        if list(current_actions)[-5:] == t_recent[-5:]:
                            sequential_score = MARKOV_SEQ_MEDIUM_WEIGHT
                
                if sequential_score < MARKOV_SEQ_SHORT_WEIGHT:
                    if len(current_actions) >= 3 and len(t_recent) >= 3:
                        if list(current_actions)[-3:] == t_recent[-3:]:
                            sequential_score = MARKOV_SEQ_SHORT_WEIGHT
            
            partial_score = 0.0
            partial_matches = 0
            partial_total = 2
            
            if t_in_battle == current_partial['in_battle']:
                partial_matches += 1
            if t_in_menu == current_partial['in_menu']:
                partial_matches += 1
            
            partial_score = partial_matches / partial_total
            
            total_score = (
                MARKOV_IMMEDIATE_WEIGHT * immediate_score +
                MARKOV_SEQUENTIAL_WEIGHT * sequential_score +
                MARKOV_PARTIAL_WEIGHT * partial_score
            )
            
            if batch_type == "action_change":
                total_score *= 1.2
            
            if transition.get('frame_offset', 0) == 0:
                total_score *= 1.1
            
            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx
        
        return best_score, best_action, best_idx
    
    def get_markov_action(self, context_state, raw_position=None, taught_frames=None):
        if not self.markov_enabled:
            return False, None, 0.0
        
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        if not frames:
            return False, None, 0.0
        
        score, action, idx = self.compute_markov_similarity(
            context_state, raw_position, taught_frames=frames
        )
        
        self.last_markov_score = score
        
        if score >= MARKOV_FAMILIARITY_THRESHOLD:
            self.last_markov_action = action
            return True, action, score
        
        return False, None, score

    # =========================================================================
    # BATTLE MARKOV SYSTEM
    # =========================================================================
    
    def load_taught_battle_transitions(self, filepath=None):
        """
        Load battle demonstration frames from taught_battle_transitions.json.
        """
        filepath = filepath or TAUGHT_BATTLE_TRANSITIONS_FILE
        try:
            if not Path(filepath).exists():
                self.battle_transitions = []
                self.battle_sequences = []
                self.battle_metadata = {}
                self.battle_loaded = False
                print(f"  ⚔️ No battle transitions file found at {filepath}")
                print(f"     Battle fallback: A-button only")
                return
            
            with open(filepath, 'r') as f:
                data = json.load(f)
            
            self.battle_transitions = data.get('flat_frames', [])
            self.battle_sequences = data.get('battle_sequences', [])
            self.battle_metadata = data.get('metadata', {})
            self.battle_loaded = True
            
            print(f"  ⚔️ Loaded battle transitions:")
            print(f"     Flat frames: {len(self.battle_transitions)}")
            print(f"     Battle sequences: {len(self.battle_sequences)}")
            print(f"     Total battle frames: {self.battle_metadata.get('total_battle_frames', 0)}")
            print(f"     Battles recorded: {self.battle_metadata.get('battles_recorded', 0)}")
            print(f"     Avg battle length: {self.battle_metadata.get('avg_battle_length', 0)}")
            outcomes = self.battle_metadata.get('outcomes', {})
            if outcomes:
                print(f"     Outcomes: {outcomes}")
            common_seqs = self.battle_metadata.get('most_common_sequences', [])
            if common_seqs:
                for seq in common_seqs[:3]:
                    print(f"     Common seq: {seq.get('sequence', [])} x{seq.get('count', 0)} ({seq.get('context', '')})")
        
        except Exception as e:
            print(f"  ⚠️ Error loading battle transitions: {e}")
            self.battle_transitions = []
            self.battle_sequences = []
            self.battle_metadata = {}
            self.battle_loaded = False
    
    def compute_battle_markov_similarity(self, context_state, palette_state=None):
        """
        Battle-specific Markov matching against taught battle frames.
        """
        if not self.battle_transitions:
            return 0.0, None, -1
        
        current_actions = list(self.battle_action_history)
        in_menu = context_state[4] > 0.5
        
        best_score = 0.0
        best_action = None
        best_idx = -1
        
        for idx, frame in enumerate(self.battle_transitions):
            t_action = frame.get('action')
            t_recent = frame.get('recent_actions', [])
            t_state = frame.get('state', {})
            batch_type = frame.get('batch_type', 'steady')
            
            if not t_action or t_action == "NONE":
                continue
            
            seq_score = 0.0
            
            if t_recent and current_actions:
                if len(current_actions) >= 8 and len(t_recent) >= 8:
                    if list(current_actions)[-8:] == t_recent[-8:]:
                        seq_score = 1.0
                
                if seq_score < 0.6:
                    if len(current_actions) >= 5 and len(t_recent) >= 5:
                        if list(current_actions)[-5:] == t_recent[-5:]:
                            seq_score = 0.6
                
                if seq_score < 0.3:
                    if len(current_actions) >= 3 and len(t_recent) >= 3:
                        if list(current_actions)[-3:] == t_recent[-3:]:
                            seq_score = 0.3
            
            palette_score = 0.0
            
            if palette_state is not None and 'palette_snapshot' in frame:
                t_palette = np.array(frame['palette_snapshot'], dtype=float)
                if len(t_palette) > 0 and len(palette_state) > 0:
                    min_dim = min(len(t_palette), len(palette_state))
                    diff = np.linalg.norm(t_palette[:min_dim] - palette_state[:min_dim])
                    palette_score = 1.0 / (1.0 + diff * 0.01)
            else:
                palette_score = 0.5
            
            menu_score = 0.0
            t_in_menu = t_state.get('in_menu', 0) == 1
            if t_in_menu == in_menu:
                menu_score = 1.0
            
            total_score = (
                BATTLE_MARKOV_ACTION_SEQ_WEIGHT * seq_score +
                BATTLE_MARKOV_PALETTE_WEIGHT * palette_score +
                BATTLE_MARKOV_MENU_STATE_WEIGHT * menu_score
            )
            
            if batch_type == "action_change":
                total_score *= 1.15
            
            if frame.get('frame_offset', 0) == 0:
                total_score *= 1.1
            
            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx
        
        return best_score, best_action, best_idx
    
    def get_battle_markov_action(self, context_state, palette_state=None):
        """
        Get battle action from Markov matching.
        """
        if not self.battle_loaded or not self.battle_transitions:
            return False, None, 0.0
        
        score, action, idx = self.compute_battle_markov_similarity(
            context_state, palette_state
        )
        
        self.last_battle_markov_score = score
        
        threshold = BATTLE_MARKOV_THRESHOLD_LOW
        if len(self.battle_transitions) > 200:
            threshold = BATTLE_MARKOV_THRESHOLD_HIGH
        
        if score >= threshold and action:
            self.last_battle_markov_action = action
            return True, action, score
        
        return False, None, score

    # =========================================================================
    # BAG MARKOV SYSTEM
    # =========================================================================

    def load_taught_bag_transitions(self, filepath=None):
        """
        Load bag demonstration frames from taught_bag_transitions.json.
        This is read-only — the AI never writes to this file.

        Expected format:
        {
            "bag_frames": [
                {
                    "action": "A",
                    "recent_actions": ["DOWN", "DOWN", "A", ...],
                    "state": {
                        "gs": 14,
                        "pocket": 0,
                        "cursor": 2,
                        "item_id": 13,
                        "mc": 0,
                        "mm": 3,
                        "in_battle": false
                    },
                    "party_context": {
                        "hp_ratios": [0.4, 1.0, 0.8, ...],
                        "status_flags": [0, 0, 4, ...],
                        "lowest_hp_ratio": 0.4,
                        "has_status": true,
                        "party_count": 3
                    },
                    "batch_type": "action_change",
                    "frame_offset": 0
                },
                ...
            ],
            "metadata": {
                "total_bag_frames": 150,
                "bag_sessions_recorded": 12,
                "items_used": [13, 17, 4],
                "pockets_visited": [0, 2]
            }
        }
        """
        filepath = filepath or TAUGHT_BAG_TRANSITIONS_FILE
        try:
            if not Path(filepath).exists():
                self.bag_transitions = []
                self.bag_metadata = {}
                self.bag_loaded = False
                print(f"  🎒 No bag transitions file found at {filepath}")
                print(f"     Bag fallback: B-button exit only")
                return

            with open(filepath, 'r') as f:
                data = json.load(f)

            self.bag_transitions = data.get('bag_frames', [])
            self.bag_metadata = data.get('metadata', {})
            self.bag_loaded = True

            print(f"  🎒 Loaded bag transitions:")
            print(f"     Bag frames: {len(self.bag_transitions)}")
            print(f"     Sessions recorded: {self.bag_metadata.get('bag_sessions_recorded', 0)}")
            items_used = self.bag_metadata.get('items_used', [])
            if items_used:
                print(f"     Items used in demos: {items_used}")
            pockets = self.bag_metadata.get('pockets_visited', [])
            if pockets:
                pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                pk_strs = [pocket_names.get(p, f"?{p}") for p in pockets]
                print(f"     Pockets visited: {', '.join(pk_strs)}")

        except Exception as e:
            print(f"  ⚠️ Error loading bag transitions: {e}")
            self.bag_transitions = []
            self.bag_metadata = {}
            self.bag_loaded = False

    def compute_bag_markov_similarity(self):
        """
        Bag-specific Markov matching against taught bag frames.

        Unlike overworld (spatial) or battle (visual/sequential), the bag
        Markov operates primarily on MENU STATE:
        - Which pocket is open
        - Where the cursor is
        - What item is at the cursor
        - What the submenu cursor (mc) shows
        - Whether we're in battle or overworld

        Secondary signal: recent action sequence (navigating menus produces
        characteristic UP/DOWN/LEFT/RIGHT/A/B sequences).

        Tertiary signal: party context (HP ratios, status conditions).
        This ensures "heal when HP is low" demonstrations match when HP
        is actually low, not when party is full health.

        Returns: (score, action, index)
        """
        if not self.bag_transitions:
            return 0.0, None, -1

        # Current bag state
        bgd = self.bag_data
        md = self.menu_data
        gs = self.game_state_raw
        in_battle = self.battle_data.get('battle_cursor', -1) != -1

        current_pocket = bgd.get('pocket', -1)
        current_cursor = bgd.get('cursor', -1)
        current_item_id = self.get_item_at_cursor()
        current_mc = md.get('mc', -1)
        current_mm = md.get('mm', -1)

        # Current party context
        hp_ratios = self.get_party_hp_ratios()
        lowest_hp = self.get_lowest_hp_ratio()
        has_status = self.has_status_condition_in_party()
        party_count = self.party_data.get('count', 0)

        # Current bag action history
        current_actions = list(self.bag_action_history)

        best_score = 0.0
        best_action = None
        best_idx = -1

        for idx, frame in enumerate(self.bag_transitions):
            t_action = frame.get('action')
            t_state = frame.get('state', {})
            t_recent = frame.get('recent_actions', [])
            t_party = frame.get('party_context', {})
            batch_type = frame.get('batch_type', 'steady')

            if not t_action or t_action == "NONE":
                continue

            # === PRIMARY: Menu state matching (40% weight) ===
            menu_score = 0.0
            menu_matches = 0
            menu_total = 5  # pocket, cursor, item_id, mc, in_battle

            # Pocket match
            t_pocket = t_state.get('pocket', -1)
            if t_pocket == current_pocket and current_pocket >= 0:
                menu_matches += 1

            # Cursor proximity (exact or within 1)
            t_cursor = t_state.get('cursor', -1)
            if t_cursor >= 0 and current_cursor >= 0:
                cursor_dist = abs(t_cursor - current_cursor)
                if cursor_dist == 0:
                    menu_matches += 1
                elif cursor_dist == 1:
                    menu_matches += 0.5

            # Item ID match (strong signal — same item at cursor)
            t_item_id = t_state.get('item_id', -1)
            if t_item_id > 0 and t_item_id == current_item_id:
                menu_matches += 1

            # Submenu cursor match (mc)
            t_mc = t_state.get('mc', -1)
            if t_mc >= 0 and t_mc == current_mc:
                menu_matches += 1

            # Battle context match
            t_in_battle = t_state.get('in_battle', False)
            if t_in_battle == in_battle:
                menu_matches += 1

            menu_score = menu_matches / menu_total

            # Skip frames from completely different bag contexts
            # (wrong pocket AND wrong item AND different battle context)
            if menu_matches < 1.0:
                continue

            # === SECONDARY: Action sequence matching (35% weight) ===
            seq_score = 0.0

            if t_recent and current_actions:
                # 6-action match = high confidence for menu navigation
                if len(current_actions) >= 6 and len(t_recent) >= 6:
                    if list(current_actions)[-6:] == t_recent[-6:]:
                        seq_score = 1.0

                # 4-action match
                if seq_score < 0.6:
                    if len(current_actions) >= 4 and len(t_recent) >= 4:
                        if list(current_actions)[-4:] == t_recent[-4:]:
                            seq_score = 0.6

                # 2-action match (short sequences common in menu nav)
                if seq_score < 0.3:
                    if len(current_actions) >= 2 and len(t_recent) >= 2:
                        if list(current_actions)[-2:] == t_recent[-2:]:
                            seq_score = 0.3

            # === TERTIARY: Party context matching (25% weight) ===
            party_score = 0.0
            party_matches = 0
            party_total = 3

            # HP ratio similarity
            t_lowest_hp = t_party.get('lowest_hp_ratio', 1.0)
            if t_lowest_hp < 1.0 or lowest_hp < 1.0:
                # Both have damage — check if similar urgency
                hp_diff = abs(t_lowest_hp - lowest_hp)
                if hp_diff < 0.15:
                    party_matches += 1  # Similar HP state
                elif hp_diff < 0.3:
                    party_matches += 0.5  # Somewhat similar
                # Bonus: both are critically low
                if t_lowest_hp < 0.3 and lowest_hp < 0.3:
                    party_matches += 0.5
            else:
                # Both full HP
                party_matches += 1

            # Status condition match
            t_has_status = t_party.get('has_status', False)
            if t_has_status == has_status:
                party_matches += 1

            # Party count match (catches change when catching)
            t_party_count = t_party.get('party_count', 0)
            if t_party_count > 0 and t_party_count == party_count:
                party_matches += 1
            elif t_party_count > 0:
                party_matches += 0.5  # Different count but close

            party_score = party_matches / party_total

            # === COMBINE ===
            total_score = (
                BAG_MARKOV_MENU_STATE_WEIGHT * menu_score +
                BAG_MARKOV_ACTION_SEQ_WEIGHT * seq_score +
                BAG_MARKOV_PARTY_CONTEXT_WEIGHT * party_score
            )

            # Boost action_change frames (human actively chose this)
            if batch_type == "action_change":
                total_score *= 1.2

            # Boost frame_offset 0 (start of new action)
            if frame.get('frame_offset', 0) == 0:
                total_score *= 1.1

            if total_score > best_score:
                best_score = total_score
                best_action = t_action
                best_idx = idx

        return best_score, best_action, best_idx

    def get_bag_markov_action(self):
        """
        Get bag action from Markov matching.

        Returns: (matched, action, score)
          matched: True if score >= threshold
          action: the suggested action string (or None)
          score: the similarity score
        """
        if not self.bag_loaded or not self.bag_transitions:
            return False, None, 0.0

        score, action, idx = self.compute_bag_markov_similarity()

        self.last_bag_markov_score = score

        if score >= BAG_MARKOV_THRESHOLD and action:
            self.last_bag_markov_action = action
            return True, action, score

        return False, None, score


# ============================================================================
# CELL 3 PART 3: Action Confirmation, Exploration Memory, Tile Probing
# ============================================================================
# NO CHANGES from original Cell 3 Part 1/2 — just reorganized boundary
# ============================================================================

    # =========================================================================
    # ACTION EXECUTION CONFIRMATION
    # =========================================================================
    
    def set_pending_action(self, action_name):
        self.pending_action = action_name
        self.pending_action_frames = 0
    
    def confirm_action_executed(self, context_state, prev_context_state):
        if self.pending_action is None:
            return True
        self.pending_action_frames += 1
        action_executed = False
        if prev_context_state is not None:
            if self.pending_action in ["UP", "DOWN", "LEFT", "RIGHT"]:
                pos_changed = (context_state[0] != prev_context_state[0] or 
                              context_state[1] != prev_context_state[1])
                dir_changed = context_state[5] != prev_context_state[5]
                action_executed = pos_changed or dir_changed
            elif self.pending_action in ["A", "B", "Start", "Select"]:
                menu_changed = abs(context_state[4] - prev_context_state[4]) > 0.1
                battle_changed = context_state[3] != prev_context_state[3]
                map_changed = context_state[2] != prev_context_state[2]
                action_executed = menu_changed or battle_changed or map_changed
        if action_executed or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES:
            self.last_confirmed_action = self.pending_action
            self.pending_action = None
            self.pending_action_frames = 0
            return True
        return False
    
    def should_send_new_action(self):
        return self.pending_action is None or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES

    # =========================================================================
    # EXPLORATION MEMORY PERSISTENCE
    # =========================================================================
    
    def load_exploration_memory(self):
        try:
            if self.EXPLORATION_MEMORY_FILE.exists():
                with open(self.EXPLORATION_MEMORY_FILE, 'r') as f:
                    data = json.load(f)
                    self.exploration_memory = {}
                    for map_key, map_data in data.items():
                        map_id = int(map_key.replace('map_', ''))
                        self.exploration_memory[map_id] = self._deserialize_map_memory(map_data)
                print(f"  Loaded exploration memory: {len(self.exploration_memory)} maps")
            else:
                self.exploration_memory = {}
        except Exception as e:
            print(f"  Error loading exploration memory: {e}")
            self.exploration_memory = {}

    def _deserialize_map_memory(self, map_data):
        memory = {
            'visited_tiles': set(tuple(t) for t in map_data.get('visited_tiles', [])),
            'obstructions': set(tuple(t) for t in map_data.get('obstructions', [])),
            'interactable_objects': map_data.get('interactable_objects', []),
            'last_visited_timestep': map_data.get('last_visited_timestep', 0),
            'transitions': map_data.get('transitions', []),
            'temp_debt': map_data.get('temp_debt', 0.0),
            'tile_interactions': {}
        }
        for tile_key, tile_data in map_data.get('tile_interactions', {}).items():
            memory['tile_interactions'][tile_key] = {
                'directions_tried': set(tile_data.get('directions_tried', [])),
                'direction_attempts': {int(k): v for k, v in tile_data.get('direction_attempts', {}).items()},
                'direction_successes': {int(k): v for k, v in tile_data.get('direction_successes', {}).items()},
                'exhausted': tile_data.get('exhausted', False)
            }
        return memory

    def save_exploration_memory(self):
        try:
            data = {f'map_{mid}': self._serialize_map_memory(md) for mid, md in self.exploration_memory.items()}
            with open(self.EXPLORATION_MEMORY_FILE, 'w') as f:
                json.dump(data, f, indent=2)
        except Exception as e:
            print(f"  Error saving exploration memory: {e}")

    def _serialize_map_memory(self, map_data):
        serialized_ti = {}
        for tile_key, td in map_data.get('tile_interactions', {}).items():
            serialized_ti[tile_key] = {
                'directions_tried': list(td.get('directions_tried', set())),
                'direction_attempts': {str(k): v for k, v in td.get('direction_attempts', {}).items()},
                'direction_successes': {str(k): v for k, v in td.get('direction_successes', {}).items()},
                'exhausted': td.get('exhausted', False)
            }
        return {
            'visited_tiles': list(map_data['visited_tiles']),
            'obstructions': list(map_data['obstructions']),
            'interactable_objects': map_data['interactable_objects'],
            'last_visited_timestep': map_data['last_visited_timestep'],
            'transitions': map_data.get('transitions', []),
            'temp_debt': map_data.get('temp_debt', 0.0),
            'tile_interactions': serialized_ti
        }

    def get_current_map_memory(self, map_id):
        if map_id not in self.exploration_memory:
            self.exploration_memory[map_id] = {
                'visited_tiles': set(), 'obstructions': set(), 'interactable_objects': [],
                'last_visited_timestep': self.timestep, 'transitions': [], 'temp_debt': 0.0,
                'tile_interactions': {}
            }
        return self.exploration_memory[map_id]

    def record_visited_tile(self, x, y, map_id):
        memory = self.get_current_map_memory(map_id)
        memory['visited_tiles'].add((int(x), int(y)))
        memory['last_visited_timestep'] = self.timestep

    def record_obstruction(self, x, y, map_id, direction):
        dx, dy = self.DIRECTION_DELTAS_INT.get(direction, (0, 0))
        memory = self.get_current_map_memory(map_id)
        memory['obstructions'].add((int(x + dx), int(y + dy)))

    # =========================================================================
    # TILE-BASED INTERACTION PROBING
    # =========================================================================
    
    def get_tile_interaction_key(self, x, y):
        return f"{int(x)}_{int(y)}"
    
    def get_tile_interaction_state(self, x, y, map_id):
        memory = self.get_current_map_memory(map_id)
        tile_key = self.get_tile_interaction_key(x, y)
        if tile_key not in memory['tile_interactions']:
            memory['tile_interactions'][tile_key] = {
                'directions_tried': set(),
                'direction_attempts': {0: 0, 1: 0, 2: 0, 3: 0},
                'direction_successes': {0: 0, 1: 0, 2: 0, 3: 0},
                'exhausted': False
            }
        return memory['tile_interactions'][tile_key]
    
    def should_interact_at_tile(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        if tile_state['exhausted']:
            return False
        if len(tile_state['directions_tried']) < 4:
            return True
        for d in range(4):
            attempts = tile_state['direction_attempts'].get(d, 0)
            successes = tile_state['direction_successes'].get(d, 0)
            if attempts > 0 and successes / attempts >= self.MIN_SUCCESS_RATE_THRESHOLD:
                return True
        return False
    
    def get_untried_directions(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        return [d for d in range(4) if d not in tile_state['directions_tried']]
    
    def get_best_interaction_direction(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        untried = self.get_untried_directions(x, y, map_id)
        if untried:
            return untried[0]
        best_dir, best_rate = None, 0.0
        for d in range(4):
            attempts = tile_state['direction_attempts'].get(d, 0)
            if attempts > 0:
                rate = tile_state['direction_successes'].get(d, 0) / attempts
                if rate > best_rate:
                    best_rate, best_dir = rate, d
        return best_dir
    
    def get_best_probe_action(self, raw_x, raw_y, current_map, current_dir):
        cache_key = (raw_x, raw_y, current_map, current_dir)
        
        if self._probe_cache_position == cache_key:
            return self._cached_probe_action, self._cached_probe_dir
        
        if not self.should_interact_at_tile(raw_x, raw_y, current_map):
            result = (None, None)
        else:
            untried = self.get_untried_directions(raw_x, raw_y, current_map)
            if not untried:
                best_dir = self.get_best_interaction_direction(raw_x, raw_y, current_map)
                if best_dir is not None:
                    result = ('A', current_dir) if current_dir == best_dir else (self.INT_TO_ACTION[best_dir], best_dir)
                else:
                    result = (None, None)
            elif current_dir in untried:
                result = ('A', current_dir)
            else:
                target_dir = untried[0]
                result = (self.INT_TO_ACTION[target_dir], target_dir)
        
        self._probe_cache_position = cache_key
        self._cached_probe_action, self._cached_probe_dir = result
        return result
    
    def record_tile_interaction_attempt(self, x, y, map_id, direction, success):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        tile_state['directions_tried'].add(direction)
        tile_state['direction_attempts'][direction] = tile_state['direction_attempts'].get(direction, 0) + 1
        if success:
            tile_state['direction_successes'][direction] = tile_state['direction_successes'].get(direction, 0) + 1
            memory = self.get_current_map_memory(map_id)
            dir_name = self.DIRECTION_NAMES.get(direction, str(direction))
            interactable = [int(x), int(y), dir_name]
            if interactable not in memory['interactable_objects']:
                memory['interactable_objects'].append(interactable)
                print(f"  🎯 INTERACTABLE FOUND: ({x}, {y}) facing {dir_name}")
        self._check_tile_exhaustion(x, y, map_id)
    
    def _check_tile_exhaustion(self, x, y, map_id):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        if len(tile_state['directions_tried']) < 4:
            return
        if not any(tile_state['direction_successes'].get(d, 0) > 0 for d in range(4)):
            tile_state['exhausted'] = True
            print(f"  ✓ Tile ({x}, {y}) exhausted - no interactions found")
    
    def get_direction_success_rate(self, x, y, map_id, direction):
        tile_state = self.get_tile_interaction_state(x, y, map_id)
        attempts = tile_state['direction_attempts'].get(direction, 0)
        if attempts == 0:
            return None
        return tile_state['direction_successes'].get(direction, 0) / attempts
    
    def start_interaction_verification(self, x, y, map_id, direction):
        self.pending_interaction_verify = {'x': x, 'y': y, 'map_id': map_id, 'direction': direction}
        self.interaction_verify_countdown = self.INTERACTION_VERIFY_FRAMES
    
    def check_interaction_verification(self, context_state, prev_context_state):
        if self.pending_interaction_verify is None:
            return
        self.interaction_verify_countdown -= 1
        success = False
        if prev_context_state is not None:
            in_overworld = prev_context_state[3] <= 0.5 and prev_context_state[4] <= 0.5
            if in_overworld:
                menu_changed = abs(context_state[4] - prev_context_state[4]) > 0.1
                battle_started = context_state[3] > 0.5 and prev_context_state[3] <= 0.5
                map_changed = int(context_state[2]) != int(prev_context_state[2])
                success = menu_changed or battle_started or map_changed
        if success or self.interaction_verify_countdown <= 0:
            info = self.pending_interaction_verify
            self.record_tile_interaction_attempt(info['x'], info['y'], info['map_id'], info['direction'], success)
            self.pending_interaction_verify = None


# ============================================================================
# CELL 3 PART 4: Transitions, Debt, Menu Trap, Navigation (with Cross-Map)
# ============================================================================
# CHANGES from v2:
# 1. UPDATED: update_menu_trap_tracking() — exempts bag (gs==14) and
#    start menu (gs==1 with valid mc) from B-boost spam. These menus
#    are now handled by their own threads/handlers instead of blind B.
#    PC menus, dialogue, and other unknown menus still get B-boosted.
# 2. All other methods UNCHANGED
# ============================================================================

    # =========================================================================
    # TRANSITION SYSTEM
    # =========================================================================
    
    def record_transition(self, from_pos, from_map, to_map, direction, action_type):
        memory = self.get_current_map_memory(from_map)
        for t in memory['transitions']:
            if t['position'] == from_pos and t['direction'] == direction:
                t['use_count'] += 1
                t['last_used'] = self.timestep
                return
        memory['transitions'].append({
            'position': from_pos, 'direction': direction, 'action': action_type,
            'destination_map': to_map, 'use_count': 1, 'last_used': self.timestep
        })
        self._map_graph_dirty = True
        print(f"  🚪 TRANSITION FOUND: Map {from_map} ({from_pos}) → Map {to_map}")

    def get_transition_attraction(self, current_map):
        memory = self.get_current_map_memory(current_map)
        transitions = memory.get('transitions', [])
        if not transitions:
            return 0.0, None
        current_debt = self.map_novelty_debt.get(current_map, 0.0)
        current_temp_debt = self.get_temp_debt(current_map)
        current_coverage = self.get_exploration_coverage(current_map)
        best_attraction, best_transition = 0.0, None
        for t in transitions:
            if self.is_transition_banned(current_map, t['position'], t['direction']):
                continue
            dest_map = t['destination_map']
            dest_debt = self.map_novelty_debt.get(dest_map, 0.0)
            dest_temp_debt = self.get_temp_debt(dest_map)
            dest_coverage = self.get_exploration_coverage(dest_map)
            debt_diff = (current_debt + current_temp_debt * 2.0) - (dest_debt + dest_temp_debt * 2.0)
            coverage_diff = current_coverage - dest_coverage
            attraction = debt_diff * 0.5 + coverage_diff * 0.5
            if t['use_count'] < 3:
                attraction *= 1.5
            if attraction > best_attraction:
                best_attraction, best_transition = attraction, t
        return best_attraction * self.TRANSITION_ATTRACTION_WEIGHT, best_transition

    # =========================================================================
    # TRANSITION BAN SYSTEM
    # =========================================================================
    
    def create_transition_ban(self, map_id, tile_pos, direction_back):
        self.transition_bans[map_id] = {
            'banned_tile': tile_pos, 'banned_direction': direction_back,
            'vicinity_radius': self.BAN_VICINITY_RADIUS, 'vicinity_active': False,
            'created_at': self.timestep
        }
        print(f"  🚫 TRANSITION BAN: Map {map_id} at {tile_pos} facing {self.DIRECTION_NAMES.get(direction_back, '?')}")
    
    def is_transition_banned(self, map_id, position, direction):
        if map_id not in self.transition_bans:
            return False
        ban = self.transition_bans[map_id]
        banned_tile = tuple(ban['banned_tile']) if isinstance(ban['banned_tile'], list) else ban['banned_tile']
        position = tuple(position) if isinstance(position, list) else position
        if position == banned_tile and direction == ban['banned_direction']:
            return True
        if ban['vicinity_active']:
            dist = abs(position[0] - banned_tile[0]) + abs(position[1] - banned_tile[1])
            if dist <= ban['vicinity_radius'] and direction == ban['banned_direction']:
                return True
        return False
    
    def is_position_banned(self, map_id, x, y, direction):
        return self.is_transition_banned(map_id, (x, y), direction)
    
    def update_transition_ban(self, map_id, current_pos):
        if map_id not in self.transition_bans:
            return
        ban = self.transition_bans[map_id]
        banned_tile = tuple(ban['banned_tile']) if isinstance(ban['banned_tile'], list) else ban['banned_tile']
        if not ban['vicinity_active'] and abs(current_pos[0] - banned_tile[0]) + abs(current_pos[1] - banned_tile[1]) >= 3:
            ban['vicinity_active'] = True
            print(f"  🚫 VICINITY BAN ACTIVE: Map {map_id}")
    
    def check_ban_lift_conditions(self, map_id):
        if map_id not in self.transition_bans:
            return
        ban = self.transition_bans[map_id]
        should_lift, reason = False, ""
        memory = self.get_current_map_memory(map_id)
        non_banned = [t for t in memory.get('transitions', []) if not self.is_transition_banned(map_id, t['position'], t['direction'])]
        if non_banned:
            should_lift, reason = True, "alternative transition found"
        elif self.get_exploration_coverage(map_id) >= self.BAN_COVERAGE_LIFT_THRESHOLD:
            should_lift, reason = True, f"coverage reached"
        elif self.timestep - ban['created_at'] >= self.BAN_TIMEOUT_STEPS:
            should_lift, reason = True, "timeout"
        if should_lift:
            del self.transition_bans[map_id]
            print(f"  ✅ BAN LIFTED: Map {map_id} - {reason}")

    # =========================================================================
    # DEBT SYSTEMS
    # =========================================================================
    
    def get_temp_debt(self, map_id):
        memory = self.get_current_map_memory(map_id)
        raw_debt = memory.get('temp_debt', 0.0)
        if map_id != self.current_map_id:
            steps_away = self.timestep - memory.get('last_visited_timestep', 0)
            return max(0.0, raw_debt - steps_away * self.TEMP_DEBT_DECAY)
        return raw_debt

    def accumulate_temp_debt(self, map_id):
        memory = self.get_current_map_memory(map_id)
        memory['temp_debt'] = min(self.TEMP_DEBT_MAX, memory.get('temp_debt', 0.0) + self.TEMP_DEBT_ACCUMULATION)

    def decay_all_debts(self):
        for map_id in list(self.map_novelty_debt.keys()):
            if map_id != self.current_map_id:
                self.map_novelty_debt[map_id] *= (1.0 - self.DEBT_DECAY_RATE)
                if self.map_novelty_debt[map_id] < 0.1:
                    del self.map_novelty_debt[map_id]
        
        current_loc = None
        if self.current_map_id is not None and len(self.last_positions) > 0:
            pos = self.last_positions[-1]
            current_loc = self.get_location_key(pos[0], pos[1], self.current_map_id)
        
        for loc in list(self.location_novelty.keys()):
            if loc != current_loc:
                self.location_novelty[loc] *= (1.0 - self.DEBT_DECAY_RATE)
                if self.location_novelty[loc] < 0.1:
                    del self.location_novelty[loc]

    def get_exploration_coverage(self, map_id):
        memory = self.get_current_map_memory(map_id)
        visited = len(memory['visited_tiles'])
        obstructions = len(memory['obstructions'])
        if visited == 0 or visited + obstructions < 10:
            return 0.0
        return visited / (visited + obstructions)

    def detect_obstruction(self, prev_context, context_state, raw_position, prev_raw_position):
        if prev_context is None or prev_raw_position is None:
            return False
        if self.last_action not in ['UP', 'DOWN', 'LEFT', 'RIGHT']:
            return False
        if raw_position == prev_raw_position:
            self.record_obstruction(raw_position[0], raw_position[1], int(context_state[2]), int(context_state[5]))
            return True
        return False

    # =========================================================================
    # MENU TRAP B-BOOST
    # =========================================================================
    
    def update_menu_trap_tracking(self, context_state, action_taken, raw_position=None):
        """
        Track when the AI is stuck in a menu and boost B utility to escape.

        EXCEPTIONS (menus with their own handlers — do NOT B-boost):
        - Bag (gs == 14): handled by bag thread
        - Start menu (gs == 1, mc in valid range): future start menu handler
        - Party menu: already handled by party_menu thread
        - Battle: not a menu trap situation

        STILL B-BOOSTED (no handler yet):
        - PC menus
        - Dialogue boxes
        - Any other unknown menu state
        """
        # Don't track during battle — battle has its own logic
        if context_state[3] > 0.5:
            self.reset_menu_trap_boost()
            return

        # Don't track if a dedicated thread is handling this menu
        if self.party_menu_active:
            self.reset_menu_trap_boost()
            return

        if self.bag_thread_active:
            self.reset_menu_trap_boost()
            return

        # === EXEMPT: Bag screen (gs == 14) ===
        # Even if bag thread hasn't activated yet (e.g. first frame),
        # don't B-boost — let bag thread detection handle it next frame
        gs = self.game_state_raw
        if gs == 14:
            self.reset_menu_trap_boost()
            return

        # === EXEMPT: Start menu (gs == 1, valid menu cursor) ===
        # Start menu has mc in range 0-6 (Pokedex through Exit)
        # with mm typically == 6. Don't B-boost — future start menu
        # handler will navigate this. For now the AI may get stuck
        # briefly but won't spam B out of a menu it should learn to use.
        if gs == 1:
            mc = self.menu_data.get('mc', -1)
            mm = self.menu_data.get('mm', -1)
            pc = self.menu_data.get('pc', -1)

            # If party cursor is active (0-6), party menu thread handles it
            # (already checked above via party_menu_active, but double-safe)
            if 0 <= pc <= 6:
                self.reset_menu_trap_boost()
                return

            # If menu cursor is in valid start menu range, exempt
            if 0 <= mc <= 6 and mm >= 4:
                self.reset_menu_trap_boost()
                return

        # === STANDARD B-BOOST for unhandled menus ===
        # PC menus, dialogue, unknown menu states, etc.
        current_pos = raw_position if raw_position else (round(context_state[0] * 255), round(context_state[1] * 255))
        if self.menu_trap_position is not None and current_pos != self.menu_trap_position:
            self.reset_menu_trap_boost()
            return
        if self.get_context_state_hash(context_state) == self.last_context_state_hash:
            if action_taken in ["A", "B", "Start", "Select"]:
                self.menu_trap_frames += 1
                self.menu_trap_position = current_pos
                if self.menu_trap_frames > self.MENU_TRAP_THRESHOLD:
                    if self.original_b_utility is None:
                        for a in self.actions():
                            if a.action == 'B':
                                self.original_b_utility = a.utility
                                break
                    self.menu_trap_b_boost = min(self.B_BOOST_MAX, self.menu_trap_b_boost + self.B_BOOST_INCREMENT)
        elif current_pos != self.menu_trap_position:
            self.reset_menu_trap_boost()

    def reset_menu_trap_boost(self):
        if self.menu_trap_b_boost > 1.0 and self.original_b_utility is not None:
            for a in self.actions():
                if a.action == 'B':
                    a.utility = self.original_b_utility
                    break
        self.menu_trap_frames = 0
        self.menu_trap_b_boost = 1.0
        self.menu_trap_position = None
        self.original_b_utility = None

    # =========================================================================
    # STANDARD METHODS
    # =========================================================================
    
    def add(self, p):
        self.perceptrons.append(p)
        self._cache_valid = False

    def actions(self):
        return [p for p in self.perceptrons if p.kind == "action"]

    def entities(self):
        return [p for p in self.perceptrons if p.kind == "entity"]

    def get_location_key(self, x, y, map_id, bin_size=5):
        return (int(map_id), int(x // bin_size) * bin_size, int(y // bin_size) * bin_size)

    def is_near_map_edge(self, x, y):
        return x < 10 or x > 245 or y < 10 or y > 245

    def record_action_execution(self, action_name):
        if action_name:
            self.action_execution_count[action_name] = self.action_execution_count.get(action_name, 0) + 1

    def get_position_stagnation(self):
        if len(self.last_positions) < 2:
            return 0
        current_pos = self.last_positions[-1]
        return sum(1 for pos in reversed(list(self.last_positions)[:-1]) if pos == current_pos)

    def get_group_weight(self, group):
        return sum(a.utility for a in self.actions() if a.group == group)

    # =========================================================================
    # MAP CONNECTIVITY GRAPH
    # =========================================================================

    def build_map_graph(self):
        """
        Build a connectivity graph from all known transitions across all maps.
        """
        if not self._map_graph_dirty:
            return self._map_graph
        
        graph = {}
        
        for map_id, memory in self.exploration_memory.items():
            edges = []
            for t in memory.get('transitions', []):
                dest = t.get('destination_map')
                if dest is not None:
                    edges.append((dest, t))
            if edges:
                graph[map_id] = edges
        
        self._map_graph = graph
        self._map_graph_dirty = False
        
        return graph

    def find_map_path(self, from_map, to_map):
        """
        BFS on the map connectivity graph to find shortest chain of maps.
        """
        if from_map == to_map:
            return [from_map]
        
        graph = self.build_map_graph()
        
        if from_map not in graph:
            return []
        
        from collections import deque as bfs_deque
        queue = bfs_deque([(from_map, [from_map])])
        visited = {from_map}
        
        while queue:
            current, path = queue.popleft()
            
            for dest_map, _ in graph.get(current, []):
                if dest_map == to_map:
                    return path + [dest_map]
                
                if dest_map not in visited:
                    visited.add(dest_map)
                    queue.append((dest_map, path + [dest_map]))
        
        return []

    def get_transition_to_map(self, from_map, to_map):
        """
        Find the best transition tile on from_map that leads to to_map.
        """
        memory = self.get_current_map_memory(from_map)
        
        best_transition = None
        best_use_count = -1
        
        for t in memory.get('transitions', []):
            if t.get('destination_map') == to_map:
                pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
                if self.is_transition_banned(from_map, pos, t['direction']):
                    continue
                
                use_count = t.get('use_count', 0)
                if use_count > best_use_count:
                    best_use_count = use_count
                    best_transition = t
        
        return best_transition

    def get_cross_map_status(self):
        """Return status dict for logging."""
        if not self.nav_map_chain:
            return {'active': False}
        
        return {
            'active': True,
            'chain': self.nav_map_chain,
            'chain_index': self.nav_chain_index,
            'chain_length': len(self.nav_map_chain),
            'current_map': self.nav_map_chain[self.nav_chain_index] if self.nav_chain_index < len(self.nav_map_chain) else None,
            'target_map': self.nav_map_chain[-1] if self.nav_map_chain else None,
            'final_target': self.nav_cross_map_target,
            'paused': self.nav_paused,
            'paused_reason': self.nav_paused_reason
        }

    # =========================================================================
    # NAVIGATION SYSTEM - A* Pathfinding + Cross-Map + Taught Targets
    # =========================================================================
    
    def load_taught_nav_targets(self, filepath=None):
        """Load human-curated navigation targets from taught_nav_targets.json."""
        filepath = filepath or TAUGHT_NAV_TARGETS_FILE
        try:
            if Path(filepath).exists():
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                self.taught_nav_targets = {}
                for map_key, targets in data.get('targets_by_map', {}).items():
                    map_id = int(map_key)
                    self.taught_nav_targets[map_id] = targets
                
                self.taught_nav_global_order = data.get('global_order', [])
                self.taught_nav_loaded = True
                
                total = sum(len(t) for t in self.taught_nav_targets.values())
                maps = list(self.taught_nav_targets.keys())
                print(f"  🎯 Loaded taught nav targets:")
                print(f"     Total targets: {total}")
                print(f"     Maps with targets: {maps}")
                print(f"     Global order entries: {len(self.taught_nav_global_order)}")
            else:
                self.taught_nav_targets = {}
                self.taught_nav_global_order = []
                self.taught_nav_loaded = False
                print(f"  No taught nav targets found at {filepath}")
        except Exception as e:
            print(f"  Error loading taught nav targets: {e}")
            self.taught_nav_targets = {}
            self.taught_nav_global_order = []
            self.taught_nav_loaded = False

    def _astar(self, start, goal, map_id):
        """A* pathfinding on visited_tiles grid, avoiding obstructions."""
        import heapq
        
        memory = self.get_current_map_memory(map_id)
        visited_tiles = memory['visited_tiles']
        obstructions = memory['obstructions']
        
        start = (int(start[0]), int(start[1]))
        goal = (int(goal[0]), int(goal[1]))
        
        if start not in visited_tiles:
            return []
        
        if goal not in visited_tiles:
            best_adj = None
            best_dist = float('inf')
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                adj = (goal[0] + dx, goal[1] + dy)
                if adj in visited_tiles and adj not in obstructions:
                    d = abs(adj[0] - start[0]) + abs(adj[1] - start[1])
                    if d < best_dist:
                        best_dist = d
                        best_adj = adj
            if best_adj is None:
                return []
            goal = best_adj
        
        if start == goal:
            return [start]
        
        open_set = [(abs(goal[0] - start[0]) + abs(goal[1] - start[1]), 0, start)]
        came_from = {}
        g_score = {start: 0}
        closed = set()
        
        while open_set:
            f, g, current = heapq.heappop(open_set)
            
            if current == goal:
                path = [current]
                while current in came_from:
                    current = came_from[current]
                    path.append(current)
                path.reverse()
                return path
            
            if current in closed:
                continue
            closed.add(current)
            
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                neighbor = (current[0] + dx, current[1] + dy)
                if neighbor in closed or neighbor not in visited_tiles or neighbor in obstructions:
                    continue
                new_g = g + 1
                if new_g < g_score.get(neighbor, float('inf')):
                    g_score[neighbor] = new_g
                    h = abs(goal[0] - neighbor[0]) + abs(goal[1] - neighbor[1])
                    came_from[neighbor] = current
                    heapq.heappush(open_set, (new_g + h, new_g, neighbor))
        
        return []

    def _get_frontier_tiles(self, map_id):
        """Find unvisited tiles adjacent to visited tiles (fallback targets)."""
        memory = self.get_current_map_memory(map_id)
        visited = memory['visited_tiles']
        obstructions = memory['obstructions']
        
        frontier = set()
        for vx, vy in visited:
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                neighbor = (vx + dx, vy + dy)
                if neighbor not in visited and neighbor not in obstructions:
                    if 0 <= neighbor[0] <= 255 and 0 <= neighbor[1] <= 255:
                        frontier.add(neighbor)
        return list(frontier)

    def _score_nav_target(self, target, current_pos, map_id):
        """Score a fallback navigation target by novelty potential."""
        memory = self.get_current_map_memory(map_id)
        visited = memory['visited_tiles']
        obstructions = memory['obstructions']
        tx, ty = target
        
        unvisited_neighbors = 0
        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            n = (tx + dx, ty + dy)
            if n not in visited and n not in obstructions:
                if 0 <= n[0] <= 255 and 0 <= n[1] <= 255:
                    unvisited_neighbors += 1
        
        score = unvisited_neighbors * 2.0
        dist = abs(current_pos[0] - tx) + abs(current_pos[1] - ty)
        score -= dist * 0.05
        if target in self.nav_struck_targets:
            score -= 100.0
        return score

    def _get_taught_targets_for_map(self, map_id, current_pos):
        """Get unvisited taught nav targets for current map, sorted by nearest."""
        if not self.taught_nav_loaded:
            return []
        
        map_targets = self.taught_nav_targets.get(map_id, [])
        if not map_targets:
            return []
        
        candidates = []
        for t in map_targets:
            order = t.get('order', 0)
            if order in self.nav_visited_targets:
                continue
            pos = tuple(t['position'])
            if pos in self.nav_struck_targets:
                continue
            dist = abs(current_pos[0] - pos[0]) + abs(current_pos[1] - pos[1])
            candidates.append((pos, t, dist))
        
        candidates.sort(key=lambda x: x[2])
        return [(pos, t) for pos, t, dist in candidates]

    def _get_next_taught_target_any_map(self, current_pos, current_map):
        """
        Find the next unvisited taught target across ALL maps.
        """
        if not self.taught_nav_loaded or not self.taught_nav_global_order:
            return None, None, None
        
        for entry in self.taught_nav_global_order:
            order = entry.get('order', -1)
            if order in self.nav_visited_targets:
                continue
            
            target_map = entry.get('map_id')
            pos = tuple(entry.get('position', [0, 0]))
            
            if pos in self.nav_struck_targets:
                continue
            
            target_data = entry
            for t in self.taught_nav_targets.get(target_map, []):
                if t.get('order') == order:
                    target_data = t
                    break
            
            return pos, target_data, target_map
        
        return None, None, None

    def build_nav_target_list(self, current_pos, map_id):
        """
        Build ranked target list.
        Priority: 
          1. Taught targets on current map (nearest unvisited)
          2. Taught targets on other maps (via cross-map nav)
          3. Fallback: frontier tiles + transitions on current map
        """
        targets = []
        
        taught_targets = self._get_taught_targets_for_map(map_id, current_pos)
        
        if taught_targets:
            for pos, t_data in taught_targets:
                dist = abs(current_pos[0] - pos[0]) + abs(current_pos[1] - pos[1])
                score = t_data.get('forward_progress_score', 0.5) * 10.0
                score -= dist * 0.02
                progress_type = t_data.get('progress_type', 'unknown')
                targets.append((pos, score, f"taught_{progress_type}", map_id))
            
            targets.sort(key=lambda x: x[1], reverse=True)
            return targets
        
        cross_pos, cross_data, cross_map = self._get_next_taught_target_any_map(current_pos, map_id)
        
        if cross_pos is not None and cross_map is not None and cross_map != map_id:
            map_path = self.find_map_path(map_id, cross_map)
            
            if map_path and len(map_path) > 1:
                next_map = map_path[1]
                transition = self.get_transition_to_map(map_id, next_map)
                
                if transition:
                    t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
                    score = 15.0
                    score -= len(map_path) * 0.5
                    targets.append((t_pos, score, f"cross_map_to_{cross_map}", cross_map))
                    
                    self._pending_cross_map = {
                        'final_target': cross_pos,
                        'final_map': cross_map,
                        'target_data': cross_data,
                        'map_chain': map_path,
                        'transition': transition
                    }
                    
                    targets.sort(key=lambda x: x[1], reverse=True)
                    return targets
            
            elif not map_path:
                targets.append(((0, 0), 5.0, f"cross_map_need_{cross_map}", cross_map))
                self._pending_cross_map = {
                    'final_target': cross_pos,
                    'final_map': cross_map,
                    'target_data': cross_data,
                    'map_chain': [],
                    'transition': None
                }
                return targets
        
        frontier = self._get_frontier_tiles(map_id)
        for ft in frontier:
            score = self._score_nav_target(ft, current_pos, map_id)
            targets.append((ft, score, 'frontier', map_id))
        
        memory = self.get_current_map_memory(map_id)
        for t in memory.get('transitions', []):
            t_pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
            if self.is_transition_banned(map_id, t_pos, t['direction']):
                continue
            score = self._score_nav_target(t_pos, current_pos, map_id)
            dest_coverage = self.get_exploration_coverage(t['destination_map'])
            score += 3.0 * (1.0 - dest_coverage)
            if t_pos not in self.nav_struck_targets:
                targets.append((t_pos, score, 'transition', map_id))
        
        targets.sort(key=lambda x: x[1], reverse=True)
        return targets

    def start_navigation(self, current_pos, map_id):
        """
        Initialize navigation mode.
        """
        self._pending_cross_map = None
        self.nav_target_list = self.build_nav_target_list(current_pos, map_id)
        
        if not self.nav_target_list:
            return False
        
        if self._pending_cross_map:
            cross = self._pending_cross_map
            self._pending_cross_map = None
            
            if cross['map_chain'] and cross['transition']:
                self.nav_map_chain = cross['map_chain']
                self.nav_chain_index = 0
                self.nav_cross_map_target = cross['final_target']
                self.nav_cross_map_target_data = cross['target_data']
                self.nav_paused = False
                
                t_pos = tuple(cross['transition']['position']) if isinstance(cross['transition']['position'], list) else cross['transition']['position']
                path = self._astar(current_pos, t_pos, map_id)
                
                if path and len(path) > 1:
                    self.nav_active = True
                    self.nav_path = path
                    self.nav_path_index = 1
                    self.nav_target = t_pos
                    self.nav_steps_taken = 0
                    self.nav_stagnation_count = 0
                    self.nav_last_position = current_pos
                    self.nav_curiosity_countdown = 0
                    
                    chain_str = ' → '.join(str(m) for m in self.nav_map_chain)
                    print(f"  🧭🌍 CROSS-MAP NAV START: {chain_str}")
                    print(f"     Final target: ({cross['final_target'][0]}, {cross['final_target'][1]}) on map {cross['final_map']}")
                    print(f"     Immediate: → transition at ({t_pos[0]}, {t_pos[1]}) → map {self.nav_map_chain[1]}")
                    self.nav_cross_map_refresh_countdown = self.NAV_CROSS_MAP_REFRESH_INTERVAL
                    return True
                else:
                    self._clear_cross_map_state()
                    
            elif not cross['map_chain']:
                self.nav_map_chain = []
                self.nav_chain_index = 0
                self.nav_cross_map_target = cross['final_target']
                self.nav_cross_map_target_data = cross['target_data']
                self.nav_paused = True
                self.nav_paused_reason = f"no path to map {cross['final_map']}"
                self.nav_paused_target_map = cross['final_map']
                self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
                self.nav_active = True
                
                print(f"  🧭⏸️ CROSS-MAP NAV PAUSED: no path to map {cross['final_map']}")
                print(f"     Exploring to find transitions...")
                return True
        
        self.nav_target_index = 0
        return self._navigate_to_next_target(current_pos, map_id)

    def advance_map_chain(self, new_map_id, current_pos):
        """
        Called when map changes during cross-map navigation.
        """
        if not self.nav_map_chain:
            return False
        
        new_index = None
        for i, chain_map in enumerate(self.nav_map_chain):
            if chain_map == new_map_id:
                new_index = i
                break
        
        if new_index is None:
            print(f"  🧭🌍 CROSS-MAP: landed on unexpected map {new_map_id}, aborting chain")
            self._clear_cross_map_state()
            return False
        
        self.nav_chain_index = new_index
        
        if new_map_id == self.nav_map_chain[-1]:
            if self.nav_cross_map_target:
                target_pos = self.nav_cross_map_target
                path = self._astar(current_pos, target_pos, new_map_id)
                
                if path and len(path) > 1:
                    self.nav_path = path
                    self.nav_path_index = 1
                    self.nav_target = target_pos
                    self.nav_steps_taken = 0
                    self.nav_stagnation_count = 0
                    self.nav_last_position = current_pos
                    
                    print(f"  🧭🌍 CROSS-MAP FINAL: arrived at map {new_map_id}, "
                          f"pathfinding to ({target_pos[0]}, {target_pos[1]})")
                    return True
                else:
                    print(f"  🧭🌍 CROSS-MAP: can't reach target on final map {new_map_id}")
                    self._clear_cross_map_state()
                    return False
            else:
                self._clear_cross_map_state()
                return False
        
        next_map = self.nav_map_chain[new_index + 1]
        transition = self.get_transition_to_map(new_map_id, next_map)
        
        if transition:
            t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
            path = self._astar(current_pos, t_pos, new_map_id)
            
            if path and len(path) > 1:
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = t_pos
                self.nav_steps_taken = 0
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                
                remaining = len(self.nav_map_chain) - new_index - 1
                print(f"  🧭🌍 CROSS-MAP STEP: map {new_map_id} → transition to map {next_map} "
                      f"({remaining} maps remaining)")
                return True
            else:
                print(f"  🧭🌍 CROSS-MAP: can't reach transition on map {new_map_id}")
                self.nav_paused = True
                self.nav_paused_reason = f"can't reach transition to map {next_map}"
                self.nav_paused_target_map = next_map
                self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
                return True
        else:
            print(f"  🧭🌍 CROSS-MAP: no transition from map {new_map_id} to map {next_map}")
            self.nav_paused = True
            self.nav_paused_reason = f"no transition to map {next_map}"
            self.nav_paused_target_map = next_map
            self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
            return True

    def check_nav_pause_resume(self, current_pos, map_id):
        """
        Called periodically when nav is paused.
        """
        if not self.nav_paused:
            return False
        
        self.nav_pause_check_countdown -= 1
        if self.nav_pause_check_countdown > 0:
            return False
        
        self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
        
        target_map = self.nav_paused_target_map
        if target_map is None:
            return False
        
        self._map_graph_dirty = True
        
        final_map = self.nav_map_chain[-1] if self.nav_map_chain else target_map
        if self.nav_cross_map_target_data:
            final_map_from_data = None
            for entry in self.taught_nav_global_order:
                if tuple(entry.get('position', [])) == self.nav_cross_map_target:
                    final_map_from_data = entry.get('map_id')
                    break
            if final_map_from_data:
                final_map = final_map_from_data
        
        new_path = self.find_map_path(map_id, final_map)
        
        if new_path and len(new_path) > 1:
            self.nav_map_chain = new_path
            self.nav_chain_index = 0
            self.nav_paused = False
            self.nav_paused_reason = ""
            self.nav_paused_target_map = None
            
            next_map = new_path[1]
            transition = self.get_transition_to_map(map_id, next_map)
            
            if transition:
                t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
                path = self._astar(current_pos, t_pos, map_id)
                
                if path and len(path) > 1:
                    self.nav_path = path
                    self.nav_path_index = 1
                    self.nav_target = t_pos
                    self.nav_steps_taken = 0
                    self.nav_stagnation_count = 0
                    self.nav_last_position = current_pos
                    
                    chain_str = ' → '.join(str(m) for m in new_path)
                    print(f"  🧭▶️ CROSS-MAP NAV RESUMED: {chain_str}")
                    return True
            
            self.nav_paused = True
            self.nav_paused_reason = f"can't reach transition to map {next_map}"
        
        return False

    def refresh_cross_map_navigation(self, current_pos, current_map):
        """
        Periodic recalculation of cross-map path.
        """
        if not self.nav_cross_map_target:
            return False
        
        final_target = self.nav_cross_map_target
        
        final_map = self.nav_map_chain[-1] if self.nav_map_chain else None
        if final_map is None:
            return False
        
        if current_map == final_map:
            path = self._astar(current_pos, final_target, current_map)
            if path and len(path) > 1:
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = final_target
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                self.nav_paused = False
                print(f"  🧭🔄 CROSS-MAP REFRESH: on final map {current_map}, re-pathing to target")
                return True
            else:
                print(f"  🧭🔄 CROSS-MAP REFRESH: on final map but can't reach target")
                return False
        
        self._map_graph_dirty = True
        new_chain = self.find_map_path(current_map, final_map)
        
        if not new_chain:
            self.nav_map_chain = [current_map]
            self.nav_chain_index = 0
            self.nav_paused = True
            self.nav_paused_reason = f"refresh: no path from map {current_map} to map {final_map}"
            self.nav_paused_target_map = final_map
            self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
            print(f"  🧭🔄 CROSS-MAP REFRESH: no path from {current_map} to {final_map}, pausing")
            return True
        
        old_chain = self.nav_map_chain
        self.nav_map_chain = new_chain
        self.nav_chain_index = 0
        
        next_map = new_chain[1]
        transition = self.get_transition_to_map(current_map, next_map)
        
        if transition:
            t_pos = tuple(transition['position']) if isinstance(transition['position'], list) else transition['position']
            path = self._astar(current_pos, t_pos, current_map)
            
            if path and len(path) > 1:
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = t_pos
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                self.nav_paused = False
                
                old_str = ' → '.join(str(m) for m in old_chain)
                new_str = ' → '.join(str(m) for m in new_chain)
                if old_chain != new_chain:
                    print(f"  🧭🔄 CROSS-MAP REFRESH: chain updated {old_str} → {new_str}")
                else:
                    print(f"  🧭🔄 CROSS-MAP REFRESH: re-pathed on map {current_map}")
                return True
            else:
                self.nav_paused = True
                self.nav_paused_reason = f"refresh: can't reach transition to map {next_map}"
                self.nav_paused_target_map = next_map
                self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
                print(f"  🧭🔄 CROSS-MAP REFRESH: can't reach transition to {next_map}, pausing")
                return True
        else:
            self.nav_paused = True
            self.nav_paused_reason = f"refresh: no transition to map {next_map}"
            self.nav_paused_target_map = next_map
            self.nav_pause_check_countdown = self.NAV_PAUSE_CHECK_INTERVAL
            print(f"  🧭🔄 CROSS-MAP REFRESH: no transition to {next_map}, pausing")
            return True

    def _clear_cross_map_state(self):
        """Reset all cross-map navigation state."""
        self.nav_map_chain = []
        self.nav_chain_index = 0
        self.nav_cross_map_target = None
        self.nav_cross_map_target_data = None
        self.nav_paused = False
        self.nav_paused_reason = ""
        self.nav_paused_target_map = None
        self.nav_pause_check_countdown = 0
        self.nav_cross_map_refresh_countdown = 0

    def _navigate_to_next_target(self, current_pos, map_id):
        """Try to path to the next target in the list."""
        while self.nav_target_index < len(self.nav_target_list):
            entry = self.nav_target_list[self.nav_target_index]
            target = entry[0]
            score = entry[1]
            target_type = entry[2]
            target_map = entry[3] if len(entry) > 3 else map_id
            
            if target in self.nav_struck_targets:
                self.nav_target_index += 1
                continue
            
            if target_type.startswith('cross_map_need'):
                self.nav_target_index += 1
                continue
            
            path = self._astar(current_pos, target, map_id)
            
            if path and len(path) > 1:
                self.nav_active = True
                self.nav_path = path
                self.nav_path_index = 1
                self.nav_target = target
                self.nav_steps_taken = 0
                self.nav_stagnation_count = 0
                self.nav_last_position = current_pos
                self.nav_curiosity_countdown = 0
                
                print(f"  🧭 NAV START: → ({target[0]}, {target[1]}) [{target_type}] "
                      f"score={score:.1f} path={len(path)} steps")
                return True
            
            self.nav_target_index += 1
        
        self.abort_navigation("no valid targets")
        return False

    def get_nav_action(self, current_pos):
        """Get next action from A* path."""
        if not self.nav_active or not self.nav_path:
            return None
        
        if self.nav_paused:
            return None
        
        if self.nav_path_index < len(self.nav_path):
            next_tile = self.nav_path[self.nav_path_index]
            
            if current_pos == next_tile:
                self.nav_path_index += 1
                if self.nav_path_index >= len(self.nav_path):
                    return None
                next_tile = self.nav_path[self.nav_path_index]
            
            dx = next_tile[0] - current_pos[0]
            dy = next_tile[1] - current_pos[1]
            
            if dx > 0: return "RIGHT"
            elif dx < 0: return "LEFT"
            elif dy > 0: return "DOWN"
            elif dy < 0: return "UP"
        
        return None

    def update_nav_state(self, current_pos, map_id):
        """Track navigation progress."""
        if not self.nav_active:
            return False
        
        if self.nav_paused:
            self.check_nav_pause_resume(current_pos, map_id)
            return True
        
        self.nav_steps_taken += 1
        
        if self.nav_map_chain:
            self.nav_cross_map_refresh_countdown -= 1
            if self.nav_cross_map_refresh_countdown <= 0:
                self.nav_cross_map_refresh_countdown = self.NAV_CROSS_MAP_REFRESH_INTERVAL
                self.refresh_cross_map_navigation(current_pos, map_id)
        
        if self.nav_steps_taken >= self.NAV_MAX_STEPS:
            self.abort_navigation("max steps reached")
            return False
        
        if current_pos == self.nav_last_position:
            self.nav_stagnation_count += 1
            if self.nav_stagnation_count >= self.NAV_STAGNATION_LIMIT:
                self.abort_navigation("stuck during pathfinding")
                return False
        else:
            self.nav_stagnation_count = 0
        self.nav_last_position = current_pos
        
        if self.nav_target:
            dist_to_target = abs(current_pos[0] - self.nav_target[0]) + abs(current_pos[1] - self.nav_target[1])
            if dist_to_target <= 1:
                if self.nav_map_chain and self.nav_chain_index < len(self.nav_map_chain) - 1:
                    print(f"  🧭🌍 Approaching transition at ({self.nav_target[0]}, {self.nav_target[1]})")
                    return True
                
                if self.nav_curiosity_countdown == 0:
                    self.nav_curiosity_countdown = self.NAV_CURIOSITY_WINDOW
                    self._mark_taught_target_visited(self.nav_target)
                    print(f"  🧭 NAV ARRIVED: ({self.nav_target[0]}, {self.nav_target[1]}) "
                          f"— curiosity window {self.NAV_CURIOSITY_WINDOW} steps")
                    return False
        
        return True

    def _mark_taught_target_visited(self, position):
        """Mark a taught nav target as visited by its order number."""
        if not self.taught_nav_loaded:
            return
        pos_tuple = tuple(position)
        maps_to_check = [self.current_map_id] if self.current_map_id is not None else []
        maps_to_check += [m for m in self.taught_nav_targets.keys() if m != self.current_map_id]
        
        for check_map in maps_to_check:
            for t in self.taught_nav_targets.get(check_map, []):
                t_pos = tuple(t['position'])
                if t_pos == pos_tuple:
                    order = t.get('order', -1)
                    if order >= 0:
                        self.nav_visited_targets.add(order)
                        print(f"  🧭 TARGET VISITED: order #{order} at ({pos_tuple[0]}, {pos_tuple[1]}) on map {check_map}")
                    return

    def complete_nav_target(self, found_novelty):
        """After curiosity window: if novelty found, done. If not, strike and try next."""
        if found_novelty:
            print(f"  🧭 NAV SUCCESS: Novelty found at ({self.nav_target[0]}, {self.nav_target[1]})")
            self.abort_navigation("novelty found")
            return
        
        if self.nav_target:
            self.nav_struck_targets.add(self.nav_target)
            print(f"  🧭 NAV STRIKE: ({self.nav_target[0]}, {self.nav_target[1]}) — no novelty, trying next")
        
        if self.nav_map_chain and self.nav_chain_index >= len(self.nav_map_chain) - 1:
            self._clear_cross_map_state()
        
        self.nav_target_index += 1
        current_pos = self.nav_last_position or (0, 0)
        map_id = self.current_map_id
        
        if not self._navigate_to_next_target(current_pos, map_id):
            self.abort_navigation("all targets exhausted")

    def abort_navigation(self, reason=""):
        """End navigation mode."""
        if self.nav_active:
            cross_info = ""
            if self.nav_map_chain:
                cross_info = f" [cross-map chain: {' → '.join(str(m) for m in self.nav_map_chain)}]"
            print(f"  🧭 NAV END: {reason} (took {self.nav_steps_taken} steps){cross_info}")
        
        self.nav_active = False
        self.nav_path = []
        self.nav_path_index = 0
        self.nav_target = None
        self.nav_steps_taken = 0
        self.nav_stagnation_count = 0
        self.nav_curiosity_countdown = 0
        self._clear_cross_map_state()

    def update_known_area_counter(self, raw_x, raw_y, map_id):
        """Track consecutive steps in visited tiles."""
        memory = self.get_current_map_memory(map_id)
        pos = (int(raw_x), int(raw_y))
        if pos in memory['visited_tiles']:
            self.known_area_counter += 1
        else:
            self.known_area_counter = 0

    def should_start_navigation(self):
        if self.nav_active:
            return False
        if self.known_area_counter < self.KNOWN_AREA_TRIGGER:
            return False
        return True

    def is_nav_active(self):
        return self.nav_active

    def is_nav_paused(self):
        return self.nav_paused

    def is_in_nav_curiosity_window(self):
        return self.nav_curiosity_countdown > 0

    def tick_nav_curiosity_window(self):
        if self.nav_curiosity_countdown > 0:
            self.nav_curiosity_countdown -= 1
            if self.nav_curiosity_countdown <= 0:
                return True
        return False

    def get_nav_targets_status(self):
        if not self.taught_nav_loaded:
            return {'loaded': False, 'total': 0, 'visited': 0, 'remaining': 0}
        total = sum(len(t) for t in self.taught_nav_targets.values())
        visited = len(self.nav_visited_targets)
        return {'loaded': True, 'total': total, 'visited': visited, 'remaining': total - visited}


# ============================================================================
# CELL 3 PART 5: Stagnation, Blend Triggers, Mode Swap, Repetition/Pattern
# ============================================================================
# NO CHANGES — reorganized from original Cell 3 Part 3
# ============================================================================

    # =========================================================================
    # BLEND TIER DETECTION
    # =========================================================================
    
    def get_blend_tier(self):
        """
        Determine blend tier based on current stagnation metrics.
        Returns 0 (no blend), 1 (light), 2 (medium), or 3 (hard).
        Higher tier = more taught model influence.
        """
        # Tier 3 (hard): extreme stagnation
        t3 = self.BLEND_TIER_TRIGGERS[3]
        if (self.detected_pattern and self.pattern_repeat_count >= t3['pattern_repeats']):
            return 3
        if self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * t3['state_stagnation_mult']:
            return 3
        
        # Tier 2 (medium): significant stagnation
        t2 = self.BLEND_TIER_TRIGGERS[2]
        if (self.detected_pattern and self.pattern_repeat_count >= t2['pattern_repeats']):
            return 2
        if self.get_position_stagnation() >= t2['pos_stagnation']:
            return 2
        if self.consecutive_action_count >= t2['consecutive']:
            return 2
        
        # Tier 1 (light): early stagnation
        t1 = self.BLEND_TIER_TRIGGERS[1]
        if (self.detected_pattern and self.pattern_repeat_count >= t1['pattern_repeats']):
            return 1
        if self.get_position_stagnation() >= t1['pos_stagnation']:
            return 1
        if self.consecutive_action_count >= t1['consecutive']:
            return 1
        
        return 0
    
    def try_blend_if_needed(self):
        """
        Check if blend should trigger. Called from action selection.
        Returns True if a blend was performed.
        """
        if not self.taught_reference['loaded']:
            return False
        
        tier = self.get_blend_tier()
        
        if tier == 0:
            return False
        
        # Only blend if tier escalated or cooldown passed
        if tier <= self.blend_tier and (self.timestep - self.last_blend_timestep) < self.BLEND_COOLDOWN:
            return False
        
        self.blend_from_taught(tier)
        return True

    # =========================================================================
    # MODE SWAP & STAGNATION  
    # =========================================================================
    
    def get_context_state_hash(self, context_state):
        return (round(context_state[0], 2), round(context_state[1], 2), int(context_state[2]),
                int(context_state[3]), round(context_state[4], 2), int(context_state[5]))

    def check_state_stagnation(self, context_state):
        current_hash = self.get_context_state_hash(context_state)
        if current_hash == self.last_context_state_hash:
            self.state_stagnation_count += 1
            if self.state_stagnation_count == 1 and self.last_action:
                self.stagnation_initiator_action = self.last_action
        else:
            self.state_stagnation_count = 0
            self.stagnation_initiator_action = None
        self.last_context_state_hash = current_hash
        return self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD

    def check_position_stagnation(self):
        return self.get_position_stagnation()

    def should_force_random(self):
        """
        Returns True if the agent is badly stuck and needs forced randomization.
        Also triggers blend from taught model before randomizing.
        """
        force = False
        
        if self.get_position_stagnation() >= 8:
            force = True
        if self.consecutive_action_count >= 15:
            force = True
        if self.detected_pattern and self.pattern_repeat_count >= 4:
            force = True
        if self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * 2:
            force = True
        
        if force:
            # Attempt blend before randomizing — blend fixes priorities,
            # random breaks the immediate loop
            self.try_blend_if_needed()
        
        return force

    def get_forced_random_action_name(self):
        """Pick a random action that ISN'T the currently repeated one or in the pattern."""
        candidates = ["UP", "DOWN", "LEFT", "RIGHT", "A", "B"]
        
        if self.current_repeated_action and self.current_repeated_action in candidates:
            candidates.remove(self.current_repeated_action)
        
        if self.detected_pattern:
            for a in self.detected_pattern:
                if a in candidates:
                    candidates.remove(a)
        
        if not candidates:
            candidates = ["UP", "DOWN", "LEFT", "RIGHT"]
            if self.current_repeated_action in candidates:
                candidates.remove(self.current_repeated_action)
        
        if not candidates:
            candidates = ["UP", "DOWN", "LEFT", "RIGHT"]
        
        return random.choice(candidates)

    def check_direction_change_progress(self, context_state):
        current_dir = int(context_state[5])
        if self.last_direction_for_progress is None:
            self.last_direction_for_progress = current_dir
            return False
        changed = current_dir != self.last_direction_for_progress
        self.last_direction_for_progress = current_dir
        return changed

    def apply_stagnation_initiator_penalty(self):
        if self.stagnation_initiator_action is None:
            return
        for a in self.actions():
            if a.action == self.stagnation_initiator_action:
                old_util = a.utility
                floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                a.utility = max(floor, a.utility * 0.5)
                print(f"  📍 STAGNATION PENALTY: {self.stagnation_initiator_action} {old_util:.3f} → {a.utility:.3f}")
                break
        self.stagnation_initiator_action = None

    def check_productive_change(self, context_state):
        current_map = int(context_state[2])
        current_battle = context_state[3] > 0.5
        current_pos = (context_state[0], context_state[1])
        productive, reason = False, ""
        
        if self.last_map_id is not None and current_map != self.last_map_id:
            productive, reason = True, "map change"
        if self.last_battle_state is not None and current_battle != self.last_battle_state:
            productive, reason = True, "battle change"
        if self.position_at_mode_swap is not None:
            dist = np.sqrt((current_pos[0] - self.position_at_mode_swap[0])**2 + 
                          (current_pos[1] - self.position_at_mode_swap[1])**2)
            if dist > 0.03:
                productive, reason = True, f"moved {dist*255:.1f} tiles"
        
        if self.direction_change_counts_as_progress and self.check_direction_change_progress(context_state):
            self.state_stagnation_count = max(0, self.state_stagnation_count - 5)
        
        self.last_map_id = current_map
        self.last_battle_state = current_battle
        return productive, reason

    def on_productive_change(self, reason):
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.swap_chain_count = 0
        self.state_stagnation_count = 0
        self.stagnation_initiator_action = None
        self.unproductive_swap_count = 0
        
        # Reset blend tier on productive progress
        if self.blend_tier > 0:
            print(f"  ✅ Blend tier reset: {self.blend_tier} → 0 ({reason})")
            self.blend_tier = 0
        
        # Reset known area counter — productive change means fresh territory
        self.known_area_counter = 0

    def on_mode_swap(self, from_mode, to_mode):
        self.swap_chain_count += 1
        self.frames_in_current_mode = 0
        self.unproductive_swap_count += 1
        if self.unproductive_swap_count >= self.UNPRODUCTIVE_SWAP_THRESHOLD:
            self._reset_highest_to_third(to_mode)
            self.unproductive_swap_count = 0
        if to_mode == "interact":
            self.interact_to_move_threshold = min(self.MAX_THRESHOLD, self.interact_to_move_threshold + self.THRESHOLD_INCREMENT)
        else:
            self.move_to_interact_threshold = min(self.MAX_THRESHOLD, self.move_to_interact_threshold + self.THRESHOLD_INCREMENT)

    def _reset_highest_to_third(self, mode):
        if mode in ["battle", "both"]:
            return
        group = "move" if mode == "move" else "interact"
        group_actions = [a for a in self.actions() if a.group == group]
        if len(group_actions) < 3:
            return
        sorted_actions = sorted(group_actions, key=lambda a: a.utility, reverse=True)
        floor = self.INTERACT_UTILITY_FLOOR if group == "interact" else self.MOVE_UTILITY_FLOOR
        sorted_actions[0].utility = max(sorted_actions[2].utility * 0.9, floor)

    def should_use_both_mode(self):
        return (self.state_stagnation_count > self.BOTH_MODE_STAGNATION_THRESHOLD or 
                self.unproductive_swap_count > self.BOTH_MODE_SWAP_THRESHOLD)

    def determine_control_mode(self, context_state, raw_position=None):
        if context_state[3] > 0.5:
            return "battle"
        
        self.frames_in_current_mode += 1
        position_stagnation = self.get_position_stagnation()
        
        productive, reason = self.check_productive_change(context_state)
        if productive:
            self.on_productive_change(reason)
        
        if self.should_use_both_mode():
            return "both"
        
        if self.check_state_stagnation(context_state):
            self.apply_stagnation_initiator_penalty()
            new_mode = "interact" if self.control_mode == "move" else "move"
            self.control_mode = new_mode
            self.position_at_mode_swap = (context_state[0], context_state[1])
            self.on_mode_swap(self.control_mode, new_mode)
            self.state_stagnation_count = 0
            return self.control_mode
        
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_map = int(context_state[2])
        
        tile_needs_probing = self.should_interact_at_tile(raw_x, raw_y, current_map)
        untried_directions = self.get_untried_directions(raw_x, raw_y, current_map)
        
        if tile_needs_probing and untried_directions and self.control_mode == "move" and self.frames_in_current_mode >= 3:
            self.control_mode = "interact"
            self.position_at_mode_swap = (context_state[0], context_state[1])
            self.frames_in_current_mode = 0
            return self.control_mode
        
        if self.control_mode == "move" and position_stagnation >= self.move_to_interact_threshold:
            self.control_mode = "interact"
            self.position_at_mode_swap = (context_state[0], context_state[1])
            self.on_mode_swap("move", "interact")
        elif self.control_mode == "interact":
            if (not tile_needs_probing or not untried_directions) and self.frames_in_current_mode >= 5:
                self.control_mode = "move"
                self.position_at_mode_swap = (context_state[0], context_state[1])
                self.frames_in_current_mode = 0
            elif self.frames_in_current_mode >= self.interact_to_move_threshold:
                self.control_mode = "move"
                self.position_at_mode_swap = (context_state[0], context_state[1])
                self.on_mode_swap("interact", "move")
        
        return self.control_mode

    # =========================================================================
    # EXPLORATION TRACKING
    # =========================================================================
    
    def update_exploration_tracking(self, context_state, prev_context_state, raw_position=None, prev_raw_position=None):
        current_map = int(context_state[2])
        raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
        raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
        current_pos = (raw_x, raw_y)
        
        if self.current_map_id is not None and current_map != self.current_map_id:
            prev_map = self.current_map_id
            if prev_context_state is not None and prev_raw_position is not None:
                self.record_transition(prev_raw_position, prev_map, current_map,
                    int(prev_context_state[5]), 'interact' if self.last_action == 'A' else 'walk')
            if prev_raw_position is not None:
                entry_dir = int(context_state[5]) if prev_context_state is not None else 0
                self.create_transition_ban(current_map, current_pos, (entry_dir + 2) % 4)
            self.on_map_change(current_map)
        
        self.current_map_id = current_map
        self.record_visited_tile(raw_x, raw_y, current_map)
        self.accumulate_temp_debt(current_map)
        self.update_transition_ban(current_map, current_pos)
        self.check_ban_lift_conditions(current_map)
        
        if prev_context_state is not None and prev_raw_position is not None:
            self.detect_obstruction(prev_context_state, context_state, raw_position, prev_raw_position)
        
        self.check_interaction_verification(context_state, prev_context_state)
        self.last_direction = int(context_state[5])
        
        if self.timestep % 300 == 0:
            self.decay_all_debts()

    def on_map_change(self, new_map):
        self.save_exploration_memory()
        self.control_mode = "move"
        self.frames_in_current_mode = 0
        
        # Abort navigation on map change
        if self.nav_active:
            self.abort_navigation("map changed")
        self.known_area_counter = 0
        self.nav_struck_targets.clear()
        
        memory = self.get_current_map_memory(new_map)
        tile_interactions = memory.get('tile_interactions', {})
        print(f"  🗺️ MAP CHANGE → {new_map}: {len(memory['visited_tiles'])} visited, {len(memory['obstructions'])} obs")
        print(f"     Tiles probed: {len(tile_interactions)}, exhausted: {sum(1 for t in tile_interactions.values() if t.get('exhausted', False))}")

    # =========================================================================
    # REPETITION & PATTERN HANDLING
    # =========================================================================
    
    def track_consecutive_action(self, action_name):
        if action_name == self.current_repeated_action:
            self.consecutive_action_count += 1
        else:
            self.current_repeated_action = action_name
            self.consecutive_action_count = 1

    def get_learning_multiplier(self, action_name):
        if action_name != self.current_repeated_action or self.consecutive_action_count < self.LEARNING_SLOWDOWN_START:
            return 1.0
        progress = min(1.0, (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) / 
                       (self.LEARNING_SLOWDOWN_MAX - self.LEARNING_SLOWDOWN_START))
        return max(0.05, 1.0 - 0.95 * progress)

    def get_nth_highest_utility(self, group, n=3):
        utilities = sorted([a.utility for a in self.actions() if a.group == group], reverse=True)
        if len(utilities) < n:
            return self.INTERACT_UTILITY_FLOOR if group == "interact" else self.MOVE_UTILITY_FLOOR
        return utilities[n-1]

    def detect_pattern(self):
        if len(self.action_history) < 6:
            return None, 0
        recent = list(self.action_history)[-self.PATTERN_CHECK_WINDOW:]
        for pattern_len in range(1, self.PATTERN_MAX_LENGTH + 1):
            if len(recent) < pattern_len * self.PATTERN_MIN_REPEATS:
                continue
            candidate = tuple(recent[-pattern_len:])
            repeat_count, idx = 0, len(recent) - pattern_len
            while idx >= 0 and tuple(recent[idx:idx + pattern_len]) == candidate:
                repeat_count += 1
                idx -= pattern_len
            if repeat_count >= self.PATTERN_MIN_REPEATS:
                return candidate, repeat_count
        return None, 0

    def apply_pattern_penalty(self):
        pattern, repeat_count = self.detect_pattern()
        if pattern is None:
            self.detected_pattern, self.pattern_repeat_count = None, 0
            return
        self.detected_pattern, self.pattern_repeat_count = pattern, repeat_count
        
        penalty_factor = max(0.3, 1.0 - repeat_count * 0.15)
        
        for action_name in set(pattern):
            for a in self.actions():
                if a.action == action_name:
                    floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                    a.utility = max(floor, a.utility * penalty_factor)
                    break

    def apply_repetition_penalty(self):
        if self.current_repeated_action is None:
            return
        for a in self.actions():
            if a.action == self.current_repeated_action:
                floor = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                if self.consecutive_action_count >= self.HARD_RESET_THRESHOLD:
                    a.utility = floor
                    self.consecutive_action_count = 0
                    print(f"  🔨 HARD RESET: {a.action} → {floor:.3f}")
                elif self.consecutive_action_count >= self.PENALTY_THRESHOLD:
                    a.utility = max(a.utility * 0.5, floor)
                break


# ============================================================================
# CELL 3 PART 6: Entity Spawning/Clustering, Learning, Save/Load
# ============================================================================
# CHANGES from v1:
# 1. save_model_checkpoint() now includes roster + move_knowledge + enemy_move_knowledge
# 2. load_taught_model() now restores roster + move_knowledge from checkpoint
# 3. NEW: save calls brain.save_roster() and brain.save_move_knowledge() at intervals
# 4. Periodic save in learn() also saves roster/move_knowledge if dirty
# 5. All other methods (entity spawning, clustering, learning, exploration) UNCHANGED
# ============================================================================

    # =========================================================================
    # ENTITY SPAWNING & CLUSTERING
    # =========================================================================

    def spawn_entity_from_novelty(self, learning_state, context_state, raw_position=None):
        """
        Spawn a new entity perceptron from the current novel state.
        The entity's weights are initialized from the state that surprised the AI,
        so it learns to detect similar situations in the future.
        No duplicate checking — clustering handles redundancy later.
        """
        entity = Perceptron("entity", entity_type=f"spawned_{self.entity_spawn_count}")
        entity.ensure_weights(len(learning_state))

        state_norm = np.linalg.norm(learning_state)
        if state_norm > 0:
            entity.weights = (learning_state / state_norm) * 0.1
        else:
            entity.weights = np.random.randn(len(learning_state)) * 0.001

        entity.utility = 1.0
        self.add(entity)
        self.entity_spawn_count += 1

        self.check_entity_capacity()

    def check_entity_capacity(self):
        """
        If entity count exceeds capacity, run clustering.
        If clustering didn't reduce count enough, expand capacity by 50%.
        """
        n_entities = len(self.entities())

        if n_entities < self.entity_capacity:
            return

        before_count = n_entities
        self.cluster_entities()
        after_count = len(self.entities())

        if after_count >= before_count * 0.9:
            old_cap = self.entity_capacity
            self.entity_capacity = int(self.entity_capacity * self.ENTITY_CAPACITY_GROWTH)
            print(f"  🧩 Entity capacity expanded: {old_cap} → {self.entity_capacity} "
                  f"(clustering only reduced {before_count} → {after_count})")

    def cluster_entities(self):
        """
        Cluster similar entity perceptrons using cosine similarity on activation patterns.
        Merge similar entities by averaging their weights.
        """
        entities = self.entities()

        innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"}
        spawned = [e for e in entities if e.entity_type not in innate_types]
        innate = [e for e in entities if e.entity_type in innate_types]

        if len(spawned) < 2:
            return

        clusterable = []
        too_young = []

        for e in spawned:
            if len(e.cluster_activations) >= self.ENTITY_MIN_ACTIVATIONS:
                clusterable.append(e)
            else:
                too_young.append(e)

        if len(clusterable) < 2:
            return

        max_len = max(len(e.cluster_activations) for e in clusterable)
        activation_vecs = []
        for e in clusterable:
            vec = list(e.cluster_activations)
            while len(vec) < max_len:
                vec.append(0.0)
            activation_vecs.append(np.array(vec))

        merged_indices = set()
        merge_groups = []

        for i in range(len(clusterable)):
            if i in merged_indices:
                continue
            group = [i]
            vec_i = activation_vecs[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10:
                continue

            for j in range(i + 1, len(clusterable)):
                if j in merged_indices:
                    continue
                vec_j = activation_vecs[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10:
                    continue

                cosine_sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)

                if cosine_sim >= self.ENTITY_CLUSTER_SIMILARITY:
                    group.append(j)
                    merged_indices.add(j)

            if len(group) > 1:
                merged_indices.add(i)
                merge_groups.append(group)

        if not merge_groups:
            return

        new_entities = []
        merged_set = set()

        for group in merge_groups:
            group_entities = [clusterable[idx] for idx in group]

            min_dim = min(len(e.weights) for e in group_entities if e.weights is not None)
            if min_dim == 0:
                continue

            avg_weights = np.zeros(min_dim)
            for e in group_entities:
                avg_weights += e.weights[:min_dim]
            avg_weights /= len(group_entities)

            merged = Perceptron("entity", entity_type=f"merged_{self.entity_merge_count}")
            merged.weights = avg_weights
            merged.utility = max(e.utility for e in group_entities)
            merged.familiarity = np.mean([e.familiarity for e in group_entities])
            merged.learning_rate = np.mean([e.learning_rate for e in group_entities])

            new_entities.append(merged)
            self.entity_merge_count += 1

            for idx in group:
                merged_set.add(id(clusterable[idx]))

        kept_spawned = [e for e in clusterable if id(e) not in merged_set]

        self.perceptrons = (
            [p for p in self.perceptrons if p.kind == "action"] +
            innate +
            kept_spawned +
            too_young +
            new_entities
        )
        self._cache_valid = False

        total_merged = sum(len(g) for g in merge_groups)
        print(f"  🧩 CLUSTERED: {total_merged} entities → {len(new_entities)} merged "
              f"| Total entities now: {len(self.entities())}")

    # =========================================================================
    # ENTITY & LEARNING
    # =========================================================================

    def spawn_innate_entities(self, learning_state):
        if self.innate_entities_spawned:
            return
        for etype, indices in [("sense_menu", [5, 6]), ("sense_battle", [3, 4]),
                                ("sense_movement", [0, 1]), ("sense_map_transition", [2])]:
            entity = Perceptron("entity", entity_type=etype)
            entity.ensure_weights(len(learning_state))
            entity.weights = np.zeros(len(learning_state))
            for idx in indices:
                entity.weights[idx] = 0.5 if len(indices) > 1 else 1.0
            self.add(entity)
        self.innate_entities_spawned = True

    def enforce_utility_floors(self):
        for a in self.actions():
            floor = self.MOVE_UTILITY_FLOOR if a.group == "move" else self.INTERACT_UTILITY_FLOOR
            a.utility = max(a.utility, floor)

    def get_spawn_threshold_adaptive(self, error_type='combined', percentile=50):
        history = {'numeric': self.numeric_error_history, 'visual': self.visual_error_history}.get(error_type, self.error_history)
        return max(0.001, np.percentile(history, percentile)) if len(history) >= 100 else 0.0005

    def stagnation_level(self, window=10):
        if len(self.prev_learning_states) < window:
            return 0.0
        recent = list(self.prev_learning_states)[-window:]
        diffs = []
        for i in range(1, len(recent)):
            a, b = recent[i], recent[i-1]
            min_dim = min(len(a), len(b))
            diffs.append(np.linalg.norm(a[:min_dim] - b[:min_dim]))
        return 1.0 - np.tanh(np.mean(diffs) * 2.0)

    def predict_future_error(self, state, action, context_state, raw_position=None):
        entity_novelty = np.mean([e.predict(state) * e.utility for e in self.entities()]) if self.entities() else 0.5
        combined = entity_novelty * 0.7 + action.utility * 0.3

        current_map = int(context_state[2])
        loc = self.get_location_key(*(raw_position if raw_position else (context_state[0]*255, context_state[1]*255)), current_map)

        map_debt = min(self.map_novelty_debt.get(current_map, 0.0), self.MAX_MAP_DEBT)
        loc_debt = min(self.location_novelty.get(loc, 0.0), self.MAX_LOCATION_DEBT)
        total_debt = map_debt + self.get_temp_debt(current_map) + loc_debt * 0.5
        combined *= 1.0 / (1.0 + total_debt * 5.0)

        if action.action == self.current_repeated_action and self.consecutive_action_count > self.LEARNING_SLOWDOWN_START:
            combined *= 1.0 / (1.0 + (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) * 0.15)
        if self.detected_pattern and action.action in self.detected_pattern:
            combined *= 1.0 / (1.0 + self.pattern_repeat_count * 0.2)

        return combined + np.random.randn() * 0.05

    def compute_multi_modal_error(self, state, next_state):
        diffs = [abs(next_state[i] - state[i]) for i in range(min(8, len(state), len(next_state)))]
        weights = [0.5, 0.5, 10.0, 5.0, 3.0, 2.0, 1.5, 0.3]
        weighted = sum(d * w for d, w in zip(diffs, weights)) + np.linalg.norm(next_state[8:] - state[8:]) * 2.0
        numeric = sum(diffs)
        visual = np.linalg.norm(next_state[8:] - state[8:])
        return weighted, numeric, visual

    def learn(self, learning_state, next_learning_state, context_state, next_context_state, dead=False,
              raw_position=None, next_raw_position=None):
        if learning_state.shape != next_learning_state.shape:
            max_dim = max(len(learning_state), len(next_learning_state))
            learning_state = np.pad(learning_state, (0, max(0, max_dim - len(learning_state))))
            next_learning_state = np.pad(next_learning_state, (0, max(0, max_dim - len(next_learning_state))))

        if not self.innate_entities_spawned:
            self.spawn_innate_entities(learning_state)

        prev_context = self.prev_context_states[-1] if self.prev_context_states else None
        prev_raw = getattr(self, '_last_raw_position', None)
        self.update_exploration_tracking(context_state, prev_context, raw_position, prev_raw)
        self._last_raw_position = raw_position

        weighted_error, numeric_error, visual_error = self.compute_multi_modal_error(learning_state, next_learning_state)
        self.error_history.append(weighted_error)
        self.numeric_error_history.append(numeric_error)
        self.visual_error_history.append(visual_error)

        # === ENTITY SPAWNING: spawn freely when novelty exceeds threshold ===
        spawn_threshold = self.get_spawn_threshold_adaptive('combined', percentile=75)
        if weighted_error > spawn_threshold and len(self.error_history) >= 100:
            menu_active = context_state[4] > 0.5
            battle_active = context_state[3] > 0.5
            if not menu_active and not battle_active:
                self.spawn_entity_from_novelty(learning_state, context_state, raw_position)

        current_map = int(context_state[2])
        loc = self.get_location_key(*(raw_position if raw_position else (context_state[0]*255, context_state[1]*255)), current_map)

        self.visited_maps[current_map] = self.visited_maps.get(current_map, 0) + 1
        self.location_memory[loc] = self.location_memory.get(loc, 0) + 1

        if self.visited_maps[current_map] > 10:
            self.map_novelty_debt[current_map] = min(self.MAX_MAP_DEBT,
                self.map_novelty_debt.get(current_map, 0.0) + 0.05 * (self.visited_maps[current_map] - 10))
        if self.location_memory[loc] > 15:
            self.location_novelty[loc] = min(self.MAX_LOCATION_DEBT,
                self.location_novelty.get(loc, 0.0) + 0.1 * (self.location_memory[loc] - 15))

        if self.visited_maps[current_map] > 30:
            weighted_error *= 0.5
        if self.location_memory[loc] > 25:
            weighted_error *= 0.7

        stagnation = self.stagnation_level()
        learning_mult = self.get_learning_multiplier(self.last_action) if self.last_action else 1.0
        if self.detected_pattern and self.last_action in self.detected_pattern:
            learning_mult *= 0.5

        for p in self.perceptrons:
            mult = learning_mult if (p.kind == "action" and p.action == self.last_action) else 1.0
            if p.kind == "action" and self.detected_pattern and p.action in self.detected_pattern:
                mult *= 0.5
            p.update(learning_state, weighted_error * mult, stagnation=stagnation)

        for a in self.actions():
            if a.action in ['Start', 'Select'] and a.weights is not None:
                a.weights *= 0.999

        self.apply_repetition_penalty()
        self.apply_pattern_penalty()
        self.enforce_utility_floors()

        # Movement boost
        if prev_context is not None and np.linalg.norm(context_state[:2] - prev_context[:2]) > 0.001:
            if self.last_action and self.consecutive_action_count < self.PENALTY_THRESHOLD:
                menu_active = context_state[4] > 0.5
                battle_active = context_state[3] > 0.5
                if not menu_active and not battle_active:
                    if not self.nav_active:
                        for a in self.actions():
                            if a.action == self.last_action:
                                boost = 1.15 if raw_position and self.is_near_map_edge(*raw_position) else 1.08
                                a.utility = min(a.utility * boost, 2.0)
                                break
                    else:
                        for a in self.actions():
                            if a.action == self.last_action:
                                boost = 1.0 + (0.08 * self.NAV_LEARNING_DAMPENING)
                                a.utility = min(a.utility * boost, 2.0)
                                break

        # === PERIODIC SAVES ===
        if self.timestep % self.SAVE_INTERVAL == 0:
            self.save_exploration_memory()
            # Save roster and move knowledge if dirty
            if self.roster_dirty:
                self.save_roster()
            if self.move_knowledge_dirty:
                self.save_move_knowledge()

        self.action_history.append(self.last_action)

    def log_state(self, learning_state, context_state):
        self.prev_learning_states.append(learning_state)
        self.prev_context_states.append(context_state)

    def update_position(self, x, y):
        self.last_positions.append((int(x), int(y)))

    def get_tile_interaction_stats(self, map_id):
        memory = self.get_current_map_memory(map_id)
        tile_interactions = memory.get('tile_interactions', {})
        return {
            'probed': len(tile_interactions),
            'exhausted': sum(1 for t in tile_interactions.values() if t.get('exhausted', False)),
            'with_success': sum(1 for t in tile_interactions.values() if any(t.get('direction_successes', {}).get(d, 0) > 0 for d in range(4)))
        }

    def load_taught_model(self, filepath):
        try:
            with open(filepath, 'r') as f:
                model = json.load(f)

            if "perceptrons" not in model:
                print(f"  ⚠️ Model file empty or invalid, starting fresh")
                return 0

            for saved_action in model["perceptrons"]["actions"]:
                for a in self.actions():
                    if a.action == saved_action["action"]:
                        a.utility = saved_action["utility"]
                        a.learning_rate = saved_action.get("learning_rate", 0.01)
                        a.familiarity = saved_action.get("familiarity", 0.0)
                        if saved_action.get("weights_nonzero"):
                            dim = saved_action.get("weights_shape", 1376)
                            a.weights = np.zeros(dim)
                            for idx, val in saved_action["weights_nonzero"]:
                                if idx < dim:
                                    a.weights[idx] = val
                        break
                    if a.action in ['Start', 'Select'] and a.weights is not None:
                        a.weights = np.zeros(len(a.weights))
                        a.utility = 0.05

            for saved_entity in model["perceptrons"].get("entities", []):
                for e in self.entities():
                    if e.entity_type == saved_entity["entity_type"]:
                        e.utility = saved_entity.get("utility", 1.0)
                        e.familiarity = saved_entity.get("familiarity", 0.0)
                        if saved_entity.get("weights_nonzero"):
                            dim = saved_entity.get("weights_shape", 1376)
                            e.weights = np.zeros(dim)
                            for idx, val in saved_entity["weights_nonzero"]:
                                if idx < dim:
                                    e.weights[idx] = val
                        break

            if "debt_tracking" in model:
                debt = model["debt_tracking"]
                self.map_novelty_debt = {int(k): v for k, v in debt.get("map_novelty_debt", {}).items()}
                self.visited_maps = {int(k): v for k, v in debt.get("visited_maps", {}).items()}
                for k, v in debt.get("location_novelty", {}).items():
                    self.location_novelty[eval(k)] = v

            # === RESTORE BATTLE KNOWLEDGE FROM CHECKPOINT ===
            if "roster" in model:
                self.roster = {}
                for k, v in model["roster"].items():
                    self.roster[int(k)] = v
                    if isinstance(v.get('fingerprint'), list):
                        v['fingerprint'] = tuple(v['fingerprint'])
                print(f"  📋 Roster restored from checkpoint: {len(self.roster)} slots")

            if "move_knowledge" in model:
                mk_data = model["move_knowledge"]
                self.move_knowledge = {}
                for mk, mv in mk_data.get('player_moves', {}).items():
                    mid = int(mk)
                    vs = {}
                    for sk, sv in mv.get('vs_species', {}).items():
                        vs[int(sk)] = sv
                    mv['vs_species'] = vs
                    self.move_knowledge[mid] = mv

                self.enemy_move_knowledge = {}
                for ek, ev in mk_data.get('enemy_moves', {}).items():
                    self.enemy_move_knowledge[int(ek)] = ev

                total_moves = len(self.move_knowledge)
                total_uses = sum(m.get('total_uses', 0) for m in self.move_knowledge.values())
                print(f"  📖 Move knowledge restored: {total_moves} moves, {total_uses} uses")

            if "battle_tracking" in model:
                bt = model["battle_tracking"]
                self.battle_low_hp_exits = bt.get("battle_low_hp_exits", 0)

            loaded_timestep = model.get("timestep", 0)
            self.timestep = loaded_timestep
            return loaded_timestep

        except Exception as e:
            print(f"  ⚠️ Error loading model: {e}, starting fresh")
            return 0

    def merge_taught_exploration(self, taught_filepath):
        if not Path(taught_filepath).exists():
            print(f"  No taught exploration memory found at {taught_filepath}")
            return

        with open(taught_filepath, 'r') as f:
            taught_data = json.load(f)

        transitions_added = 0
        interactables_added = 0

        for map_key, taught_map in taught_data.items():
            map_id = int(map_key.replace('map_', ''))
            ai_map = self.get_current_map_memory(map_id)

            for t_trans in taught_map.get('transitions', []):
                t_pos = tuple(t_trans['position'])
                t_dir = t_trans['direction']
                exists = any(
                    tuple(existing['position']) == t_pos and existing['direction'] == t_dir
                    for existing in ai_map['transitions']
                )
                if not exists:
                    ai_map['transitions'].append(t_trans)
                    transitions_added += 1

            for t_inter in taught_map.get('interactable_objects', []):
                if t_inter not in ai_map['interactable_objects']:
                    ai_map['interactable_objects'].append(t_inter)
                    interactables_added += 1

        print(f"  Merged: {transitions_added} transitions, {interactables_added} interactables")

    def save_model_checkpoint(self, filepath):
        model = {
            "timestep": self.timestep,
            "perceptrons": {"actions": [], "entities": []},
            "debt_tracking": {
                "map_novelty_debt": {str(k): v for k, v in self.map_novelty_debt.items()},
                "location_novelty": {str(k): v for k, v in self.location_novelty.items()},
                "visited_maps": {str(k): v for k, v in self.visited_maps.items()}
            },
            "control_mode": self.control_mode,
            "markov_stats": {
                "markov_action_count": self.markov_action_count,
                "curiosity_action_count": self.curiosity_action_count
            },
            "blend_stats": {
                "blend_count": self.blend_count,
                "last_blend_tier": self.blend_tier
            },
            "battle_stats": {
                "battle_action_count": self.battle_action_count,
                "battle_markov_action_count": self.battle_markov_action_count,
                "current_battle_id": self.current_battle_id
            },
            # === NEW: Roster embedded in checkpoint ===
            "roster": {},
            # === NEW: Move knowledge embedded in checkpoint ===
            "move_knowledge": {
                "player_moves": {},
                "enemy_moves": {}
            },
            # === NEW: Battle tracking persistence ===
            "battle_tracking": {
                "battle_low_hp_exits": self.battle_low_hp_exits
            }
        }

        # Serialize roster (convert tuples to lists for JSON)
        for k, v in self.roster.items():
            entry = v.copy()
            if isinstance(entry.get('fingerprint'), tuple):
                entry['fingerprint'] = list(entry['fingerprint'])
            model["roster"][str(k)] = entry

        # Serialize move knowledge (convert int keys to strings)
        for mk, mv in self.move_knowledge.items():
            mv_copy = mv.copy()
            vs = mv_copy.get('vs_species', {})
            mv_copy['vs_species'] = {str(sk): sv for sk, sv in vs.items()}
            model["move_knowledge"]["player_moves"][str(mk)] = mv_copy

        for ek, ev in self.enemy_move_knowledge.items():
            model["move_knowledge"]["enemy_moves"][str(ek)] = ev

        # Perceptrons
        for a in self.actions():
            action_data = {
                "action": a.action,
                "group": a.group,
                "utility": float(a.utility),
                "weights_shape": len(a.weights) if a.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(a.weights) if abs(v) > 1e-10] if a.weights is not None else [],
                "learning_rate": float(a.learning_rate),
                "familiarity": float(a.familiarity)
            }
            model["perceptrons"]["actions"].append(action_data)

        for e in self.entities():
            entity_data = {
                "entity_type": e.entity_type,
                "utility": float(e.utility),
                "weights_shape": len(e.weights) if e.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(e.weights) if abs(v) > 1e-10] if e.weights is not None else [],
                "familiarity": float(e.familiarity)
            }
            model["perceptrons"]["entities"].append(entity_data)

        with open(filepath, 'w') as f:
            json.dump(model, f, indent=2)

        # Also save standalone files (for external tools / teaching code)
        if self.roster_dirty:
            self.save_roster()
        if self.move_knowledge_dirty:
            self.save_move_knowledge()

In [ ]:
# ============================================================================
# CELL 4: Action Selection - Party Menu + Bag + Battle + Overworld
# ============================================================================
# CHANGES from v2:
# 1. NEW: bag_action() — bag thread handler with Markov + fallback B exit
# 2. NEW: _record_bag_action() — helper to record bag actions consistently
# 3. UPDATED: anticipatory_action() priority order:
#    0. Party menu thread (highest)
#    1. Bag thread (NEW — between party menu and battle)
#    2. Battle thread
#    3. Overworld (unchanged)
# 4. bag_action() tracks item use for observation pipeline:
#    detects when submenu cursor is on "Use" and A is pressed,
#    triggers start_item_observation() with current item at cursor
# 5. All other code UNCHANGED
# ============================================================================

import random

GBA_ACTIONS = ["Up", "Down", "Left", "Right", "A", "B", "Start", "Select"]
ACTION_DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
DIRECTION_TO_ACTION = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
ACTION_TO_DIRECTION = {"DOWN": 0, "UP": 1, "LEFT": 2, "RIGHT": 3}

# Battle menu layout
# Layout:  FIGHT(0)   BAG(1)
#          POKEMON(2)  RUN(3)
BATTLE_CURSOR_FIGHT = 0
BATTLE_CURSOR_BAG = 1
BATTLE_CURSOR_POKEMON = 2
BATTLE_CURSOR_RUN = 3


def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])


def _get_action_perceptron(actions_list, action_name):
    """Helper to find action perceptron by name."""
    for a in actions_list:
        if a.action == action_name:
            return a
    return None


def _record_battle_action(brain, action_perceptron):
    """Helper to record a battle action consistently."""
    brain.record_action_execution(action_perceptron.action)
    brain.track_consecutive_action(action_perceptron.action)
    brain.battle_action_history.append(action_perceptron.action)
    brain.set_party_menu_last_action(action_perceptron.action)
    return action_perceptron


def _record_party_menu_action(brain, action_perceptron):
    """Helper to record a party menu action consistently."""
    brain.record_action_execution(action_perceptron.action)
    brain.track_consecutive_action(action_perceptron.action)
    brain.party_menu_action_count += 1
    brain.set_party_menu_last_action(action_perceptron.action)
    if brain.party_menu_context in ("battle_forced", "battle_voluntary"):
        brain.battle_action_history.append(action_perceptron.action)
        brain.battle_action_count += 1
    return action_perceptron


def _record_bag_action(brain, action_perceptron):
    """Helper to record a bag thread action consistently."""
    brain.record_action_execution(action_perceptron.action)
    brain.track_consecutive_action(action_perceptron.action)
    brain.bag_thread_action_count += 1
    brain.bag_thread_total_actions += 1
    brain.bag_action_history.append(action_perceptron.action)
    brain.set_bag_thread_last_action(action_perceptron.action)
    # Also track in battle history if in battle bag
    if brain.bag_thread_context == "battle":
        brain.battle_action_history.append(action_perceptron.action)
        brain.battle_action_count += 1
    return action_perceptron


# ============================================================================
# 2x2 GRID NAVIGATION HELPERS
# ============================================================================

def _navigate_2x2(current, target):
    """
    Navigate a 2x2 grid from current position to target position.
    Grid layout:  0  1
                  2  3
    Returns: direction string or "A" if already there.
    """
    if current == target:
        return "A"

    cur_col, cur_row = current % 2, current // 2
    tgt_col, tgt_row = target % 2, target // 2

    if cur_row < tgt_row:
        return "DOWN"
    elif cur_row > tgt_row:
        return "UP"
    elif cur_col < tgt_col:
        return "RIGHT"
    elif cur_col > tgt_col:
        return "LEFT"

    return "A"


def _navigate_party_cursor(current_slot, target_slot):
    """
    Navigate party menu cursor (vertical list 0-5, 6=cancel).
    Returns: direction string or "A" if already there.
    """
    if current_slot == target_slot:
        return "A"

    if current_slot < target_slot:
        return "DOWN"
    else:
        return "UP"


# ============================================================================
# PARTY MENU ACTION — Unified handler (battle + overworld)
# ============================================================================

def party_menu_action(brain, context_state):
    """
    Party menu handler. Called when brain.is_party_menu_active() is True.
    Takes control ABOVE all other threads.
    """
    actions_list = brain.actions()
    context = brain.party_menu_context
    bd = brain.battle_data
    pc = bd.get('party_cursor', -1)

    # === FORCED SWITCH: fainted mon, must pick replacement ===
    if context == "battle_forced":
        target = brain.party_menu_target_slot

        if target < 0:
            target = brain.get_best_switch_slot()
            brain.party_menu_target_slot = target
            brain.forced_switch_target_slot = target
            if target >= 0:
                print(f"  🔄 FORCED SWITCH: targeting slot {target}")
            else:
                print(f"  🔄 FORCED SWITCH: no living party members! pressing B")
                a = _get_action_perceptron(actions_list, "B")
                if a:
                    return _record_party_menu_action(brain, a)

        if target >= 0 and 0 <= pc <= 5:
            nav = _navigate_party_cursor(pc, target)

            if nav == "A":
                print(f"  🔄 FORCED SWITCH: confirming slot {target}")
                brain.forced_switch_pending = False
                brain.forced_switch_target_slot = -1
                a = _get_action_perceptron(actions_list, "A")
                if a:
                    return _record_party_menu_action(brain, a)
            else:
                a = _get_action_perceptron(actions_list, nav)
                if a:
                    return _record_party_menu_action(brain, a)

        a = _get_action_perceptron(actions_list, "A")
        if a:
            return _record_party_menu_action(brain, a)

    # === VOLUNTARY BATTLE PARTY MENU: press B to exit ===
    if context == "battle_voluntary":
        a = _get_action_perceptron(actions_list, "B")
        if a:
            return _record_party_menu_action(brain, a)

    # === OVERWORLD PARTY MENU: press B to exit ===
    if context == "overworld":
        a = _get_action_perceptron(actions_list, "B")
        if a:
            return _record_party_menu_action(brain, a)

    # === FALLBACK ===
    a = _get_action_perceptron(actions_list, "B")
    if a:
        return _record_party_menu_action(brain, a)

    return actions_list[0]


# ============================================================================
# BAG ACTION — Markov imitation + item observation + fallback B exit
# ============================================================================

def bag_action(brain, context_state):
    """
    Bag thread handler. Called when brain.is_bag_thread_active() is True.
    Takes control ABOVE battle and overworld, BELOW party menu.

    HIERARCHY:
    1. MARKOV IMITATION: match against taught bag demonstrations
       using bag-specific state (pocket, cursor, item ID, mc, party HP).
       If a good match is found, replay the human's action.
    2. AUTONOMOUS DECISIONS (future): use item knowledge to decide
       what to do based on party state and learned item effects.
       Currently stubbed — will be implemented when item knowledge
       has enough data.
    3. FALLBACK: press B to exit the bag safely.
       This is the intentional exit, replacing the old blind B-spam
       from the menu trap system.

    ITEM OBSERVATION:
    When the bag Markov (or future autonomous system) causes an item
    to be used, we detect it by watching:
    - submenu cursor (mc) == 0 ("Use" option) AND action == "A"
    This triggers start_item_observation() which snapshots party state
    and watches for HP/status changes on subsequent frames.
    """
    actions_list = brain.actions()
    bgd = brain.bag_data
    md = brain.menu_data

    # =================================================================
    # === DETECT ITEM USE (for observation pipeline) ===
    # =================================================================
    # Check if the PREVIOUS action caused an item use.
    # Item use happens when: mc == 0 (Use selected) AND we pressed A.
    # We check prev state because the effect happens AFTER the press.
    prev_mc = brain.prev_menu_data.get('mc', -1)
    prev_action = brain.bag_thread_last_action

    if prev_action == "A" and prev_mc == 0:
        # We just pressed A on "Use" — an item is being used
        # Figure out which item it was from the previous bag state
        prev_items = brain.prev_bag_data.get('items', [])
        prev_cursor = brain.prev_bag_data.get('cursor', -1)

        item_id = -1
        if 0 <= prev_cursor < len(prev_items):
            item_id = prev_items[prev_cursor].get('id', -1)

        if item_id > 0 and brain.pending_item_observation is None:
            # Determine target slot from party cursor if available
            target_slot = md.get('pc', -1)
            brain.start_item_observation(item_id, target_slot=target_slot)

    # =================================================================
    # === 1. BAG MARKOV IMITATION ===
    # =================================================================
    matched, action_name, score = brain.get_bag_markov_action()

    if matched and action_name:
        # Filter dangerous actions in bag context
        if action_name in ["START", "SELECT"]:
            action_name = "B"  # Safe fallback — exit bag

        brain.bag_thread_markov_actions += 1
        brain.last_bag_markov_action = action_name

        a = _get_action_perceptron(actions_list, action_name)
        if a:
            return _record_bag_action(brain, a)

    # =================================================================
    # === 2. AUTONOMOUS DECISIONS (future — stubbed) ===
    # =================================================================
    # When item knowledge has enough data, this section will:
    # - Check party HP ratios → find healing items → navigate to them → use
    # - Check party status → find status cure items → use
    # - In battle: check enemy HP → find Pokeballs → use
    #
    # For now, fall through to B exit.
    #
    # Placeholder check: if we have known healing items and party needs healing,
    # log it (but don't act yet — need navigation logic first)
    if brain.get_lowest_hp_ratio() < 0.5:
        healing = brain.get_healing_items()
        if healing:
            # Future: navigate to the healing item and use it
            # For now just note it in the action history for debugging
            pass

    # =================================================================
    # === 3. FALLBACK: press B to exit bag ===
    # =================================================================
    # This is an intentional, clean exit — NOT the blind B-spam from
    # menu trap. The bag thread knows it's in the bag and is choosing
    # to leave because it has no Markov match and no autonomous plan.
    a = _get_action_perceptron(actions_list, "B")
    if a:
        return _record_bag_action(brain, a)

    return actions_list[0]


# ============================================================================
# BATTLE ACTION — Smart Move Selection + Running (no party menu handling)
# ============================================================================

def battle_action(brain, context_state, palette_state=None):
    """
    Battle-only action selection. Called when in_battle > 0.5 AND
    party menu is NOT active AND bag thread is NOT active.

    COMPLETELY ISOLATED from overworld, party menu, and bag.
    """
    actions_list = brain.actions()

    brain.battle_frame_count += 1
    brain.battle_action_count += 1

    bd = brain.battle_data
    has_data = brain.has_battle_data()

    # =================================================================
    # === CURSOR-AWARE BATTLE LOGIC ===
    # =================================================================
    if has_data:
        menu_state = brain.infer_battle_menu_state()
        brain.battle_menu_state = menu_state

        bc = bd['battle_cursor']
        mc = bd['move_cursor']

        want_run = brain.should_run()

        # --- MAIN MENU: navigate to target (FIGHT or RUN) ---
        if menu_state == "main_menu" and 0 <= bc <= 3:
            brain.battle_cursor_action_count += 1

            if want_run:
                target_cursor = BATTLE_CURSOR_RUN
            else:
                target_cursor = BATTLE_CURSOR_FIGHT

            nav = _navigate_2x2(bc, target_cursor)
            a = _get_action_perceptron(actions_list, nav)
            if a:
                return _record_battle_action(brain, a)

        # --- MOVE SELECT: pick best move, navigate to slot ---
        if menu_state == "move_select" and 0 <= mc <= 3:
            brain.battle_cursor_action_count += 1

            target_slot = 0

            enemy_species = bd.get('enemy_species', -1)
            if enemy_species > 0:
                ranked = brain.get_best_move_for_enemy(enemy_species)
                if ranked:
                    target_slot = ranked[0][1]
                else:
                    available = brain.get_moves_with_pp()
                    if available:
                        target_slot = available[0][0]
            else:
                available = brain.get_moves_with_pp()
                if available:
                    target_slot = available[0][0]

            nav = _navigate_2x2(mc, target_slot)

            if nav == "A":
                move_keys = ['move0', 'move1', 'move2', 'move3']
                brain.last_move_slot = target_slot
                brain.last_move_used = bd.get(move_keys[target_slot], -1)

            a = _get_action_perceptron(actions_list, nav)
            if a:
                return _record_battle_action(brain, a)

    # =================================================================
    # === MARKOV FALLBACK ===
    # =================================================================
    matched, action_name, score = brain.get_battle_markov_action(
        context_state, palette_state
    )

    if matched and action_name:
        if action_name in ["START", "SELECT"]:
            action_name = "A"

        brain.battle_markov_action_count += 1
        brain.last_battle_markov_action = action_name

        a = _get_action_perceptron(actions_list, action_name)
        if a:
            return _record_battle_action(brain, a)

    # =================================================================
    # === A-BUTTON FALLBACK ===
    # =================================================================
    a = _get_action_perceptron(actions_list, "A")
    if a:
        return _record_battle_action(brain, a)

    return actions_list[0]


# ============================================================================
# CURIOSITY OVERRIDE CHECK (overworld only — unchanged)
# ============================================================================

def _check_curiosity_override(brain, learning_state, context_state, raw_position, map_density):
    """
    Check if curiosity detects something novel enough to override navigation.
    """
    raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
    raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
    current_map = int(context_state[2])

    memory = brain.get_current_map_memory(current_map)

    if (raw_x, raw_y) not in memory['visited_tiles']:
        return True

    if brain.should_interact_at_tile(raw_x, raw_y, current_map):
        untried = brain.get_untried_directions(raw_x, raw_y, current_map)
        if untried:
            return True

    if context_state[3] <= 0.5 and context_state[4] <= 0.5 and brain.entities():
        entity_signal = np.mean([abs(e.predict(learning_state)) * e.utility for e in brain.entities()])
        density = map_density or {'tier': 'medium'}
        novelty_threshold = {
            'sparse': 0.15, 'thin': 0.25, 'medium': 0.35, 'dense': 0.45
        }.get(density['tier'], 0.35)

        if entity_signal > novelty_threshold:
            return True

    return False


# ============================================================================
# MAIN ACTION SELECTION — Party Menu > Bag > Battle > Overworld
# ============================================================================

def anticipatory_action(brain, learning_state, context_state,
                       exploration_weight=1.3, min_interact_prob=0.15,
                       raw_position=None,
                       forced_explore_prob=0.18,
                       override_threshold=1.5,
                       taught_frames=None,
                       map_density=None,
                       palette_state=None):
    """
    ACTION SELECTION HIERARCHY:
    0. PARTY MENU THREAD (if party_menu_active — forced switch, voluntary, overworld)
    1. BAG THREAD (if bag_thread_active — Markov imitation, item observation, B exit)
    2. BATTLE THREAD (if in_battle — smart moves + running + Markov + A)
    3. Forced random + blend (when badly stuck)
    4. Navigation mode (single-map A* or cross-map chain)
    5. Markov imitation (familiar taught situation)
    6. Curiosity exploration (default)
    """
    actions_list = brain.actions()
    if not actions_list:
        return Perceptron("action", action="UP", group="move")

    # =================================================================
    # === 0. PARTY MENU INTERCEPT — above ALL other logic ===
    # =================================================================
    if brain.is_party_menu_active():
        return party_menu_action(brain, context_state)

    # =================================================================
    # === 1. BAG THREAD INTERCEPT — above battle and overworld ===
    # =================================================================
    if brain.is_bag_thread_active():
        return bag_action(brain, context_state)

    # =================================================================
    # === 2. BATTLE INTERCEPT — before overworld logic ===
    # =================================================================
    in_battle = context_state[3] > 0.5

    if in_battle:
        return battle_action(brain, context_state, palette_state)

    # =================================================================
    # === OVERWORLD LOGIC BELOW — only runs when NOT in battle ===
    # === and NOT in party menu and NOT in bag thread ===
    # === (COMPLETELY UNCHANGED from previous version) ===
    # =================================================================

    # === ADAPT THRESHOLDS TO DATA DENSITY ===
    density = map_density or {'taught_frames': 0, 'tier': 'sparse', 'coverage': 0.0, 'visited': 0}
    tier = density['tier']

    markov_threshold = {
        'sparse': 0.72, 'thin': 0.65, 'medium': 0.58, 'dense': 0.50
    }.get(tier, MARKOV_FAMILIARITY_THRESHOLD)

    adapted_explore_prob = {
        'sparse': 0.30, 'thin': 0.24, 'medium': 0.18, 'dense': 0.12
    }.get(tier, forced_explore_prob)

    adapted_exploration_weight = {
        'sparse': 1.8, 'thin': 1.5, 'medium': 1.3, 'dense': 1.1
    }.get(tier, exploration_weight)

    transition_weight_mult = {
        'sparse': 0.3, 'thin': 0.6, 'medium': 1.0, 'dense': 1.4
    }.get(tier, 1.0)

    raw_x = raw_position[0] if raw_position else int(context_state[0] * 255)
    raw_y = raw_position[1] if raw_position else int(context_state[1] * 255)
    current_map = int(context_state[2])
    current_dir = int(context_state[5])
    current_pos = (raw_x, raw_y)

    # === 3. FORCED RANDOMIZATION + BLEND ===
    brain.check_state_stagnation(context_state)

    if brain.should_force_random():
        if brain.is_nav_active():
            brain.abort_navigation("forced random triggered")

        forced_name = brain.get_forced_random_action_name()
        for a in actions_list:
            if a.action == forced_name:
                brain.curiosity_action_count += 1
                brain.record_action_execution(a.action)
                brain.track_consecutive_action(a.action)
                print(f"  🎲 FORCED RANDOM: {forced_name} (pos_stag={brain.get_position_stagnation()}, "
                      f"repeat={brain.consecutive_action_count}, pattern={brain.pattern_repeat_count})")
                return a

    # === 4. NAVIGATION MODE ===

    if context_state[3] <= 0.5 and context_state[4] <= 0.5:
        brain.update_known_area_counter(raw_x, raw_y, current_map)

    if not brain.is_nav_active() and brain.should_start_navigation():
        if context_state[3] <= 0.5 and context_state[4] <= 0.5:
            started = brain.start_navigation(current_pos, current_map)
            if started:
                brain.known_area_counter = 0

    if brain.is_nav_active():
        if brain.is_nav_paused():
            brain.update_nav_state(current_pos, current_map)
        else:
            if _check_curiosity_override(brain, learning_state, context_state, raw_position, map_density):
                if brain.nav_map_chain:
                    pass
                else:
                    brain.abort_navigation("curiosity override")
                    print(f"  🧭→🔍 NAV interrupted: curiosity detected novelty at ({raw_x}, {raw_y})")
            else:
                nav_continue = brain.update_nav_state(current_pos, current_map)

                if nav_continue:
                    nav_action_name = brain.get_nav_action(current_pos)

                    if nav_action_name:
                        for a in actions_list:
                            if a.action == nav_action_name:
                                brain.curiosity_action_count += 1
                                brain.record_action_execution(a.action)
                                brain.track_consecutive_action(a.action)
                                return a
                    else:
                        if not brain.nav_paused:
                            brain.abort_navigation("path invalid")

    if brain.is_in_nav_curiosity_window():
        window_expired = brain.tick_nav_curiosity_window()

        if window_expired:
            memory = brain.get_current_map_memory(current_map)
            tile_needs_probing = brain.should_interact_at_tile(raw_x, raw_y, current_map)
            found_novelty = tile_needs_probing or (current_pos not in memory['visited_tiles'])

            brain.complete_nav_target(found_novelty)

    # === 5. MARKOV CHECK ===
    use_markov = False
    markov_action = None
    markov_confidence = 0.0

    if brain.markov_enabled and taught_frames:
        score, action, idx = brain.compute_markov_similarity(
            context_state, raw_position, taught_frames=taught_frames
        )
        brain.last_markov_score = score

        if score >= markov_threshold and action:
            use_markov = True
            markov_action = action
            markov_confidence = score
            brain.last_markov_action = action

    if use_markov and markov_action:
        for a in actions_list:
            if a.action == markov_action:
                brain.markov_action_count += 1
                brain.record_action_execution(a.action)
                brain.track_consecutive_action(a.action)

                if a.action == 'A':
                    if brain.should_interact_at_tile(raw_x, raw_y, current_map):
                        brain.start_interaction_verification(
                            raw_x, raw_y, current_map, int(context_state[5])
                        )

                return a

    # === 6. CURIOSITY-DRIVEN SELECTION ===
    brain.curiosity_action_count += 1

    mode = brain.determine_control_mode(context_state, raw_position=raw_position)

    memory = brain.get_current_map_memory(current_map)
    visited_tiles = memory['visited_tiles']
    obstructions = memory['obstructions']

    tile_needs_probing = brain.should_interact_at_tile(raw_x, raw_y, current_map)
    probe_action, probe_dir = brain.get_best_probe_action(raw_x, raw_y, current_map, current_dir)

    transition_attraction, best_transition = brain.get_transition_attraction(current_map)
    coverage = brain.get_exploration_coverage(current_map)

    if random.random() < adapted_explore_prob:
        valid = [a for a in actions_list if a.action not in ['Start', 'Select']]
        chosen = random.choice(valid)
        brain.record_action_execution(chosen.action)
        brain.track_consecutive_action(chosen.action)
        if chosen.action == 'A' and tile_needs_probing:
            brain.start_interaction_verification(raw_x, raw_y, current_map, current_dir)
        return chosen

    action_scores = {}

    for a in actions_list:
        if a.action in ['Start', 'Select']:
            action_scores[a.action] = (a, 0.0)
            continue

        predicted = brain.predict_future_error(learning_state, a, context_state, raw_position=raw_position)

        if a.group == "move":
            predicted *= adapted_exploration_weight

            dx, dy = ACTION_DELTAS.get(a.action, (0, 0))
            target_tile = (raw_x + dx, raw_y + dy)
            action_direction = ACTION_TO_DIRECTION.get(a.action, -1)

            if target_tile not in visited_tiles:
                predicted *= brain.UNVISITED_TILE_BONUS

            if target_tile in obstructions:
                predicted *= brain.OBSTRUCTION_PENALTY

            if brain.is_position_banned(current_map, raw_x, raw_y, action_direction):
                predicted *= 0.05

            if transition_attraction > 0.3 and best_transition and coverage > 0.5:
                trans_pos = tuple(best_transition['position']) if isinstance(best_transition['position'], list) else best_transition['position']
                if manhattan_distance(target_tile, trans_pos) < manhattan_distance(current_pos, trans_pos):
                    predicted *= (1.0 + transition_attraction * transition_weight_mult)

            if probe_action == a.action and probe_dir is not None:
                predicted *= 2.0

            predicted *= (0.9 + random.random() * 0.2)

        elif a.group == "interact":
            predicted = max(predicted, min_interact_prob)

            if a.action == 'B':
                predicted *= brain.menu_trap_b_boost

            if a.action == 'A':
                if tile_needs_probing and probe_action == 'A':
                    predicted *= 3.0
                elif tile_needs_probing:
                    predicted *= 0.5
                else:
                    predicted *= 0.3

        action_scores[a.action] = (a, predicted)

    if mode == "battle":
        preferred_group = "interact"
    elif mode == "interact":
        preferred_group = "interact"
    else:
        preferred_group = "move"

    in_mode = [(a, s) for name, (a, s) in action_scores.items() if a.group == preferred_group and s > 0]
    out_mode = [(a, s) for name, (a, s) in action_scores.items() if a.group != preferred_group and s > 0 and a.action not in ['Start', 'Select']]

    best_in_mode = max(in_mode, key=lambda x: x[1]) if in_mode else None
    best_out_mode = max(out_mode, key=lambda x: x[1]) if out_mode else None

    chosen = None

    if best_in_mode and best_out_mode:
        if best_out_mode[1] > best_in_mode[1] * override_threshold:
            chosen = best_out_mode[0]
        else:
            chosen = best_in_mode[0]
    elif best_in_mode:
        chosen = best_in_mode[0]
    elif best_out_mode:
        chosen = best_out_mode[0]
    else:
        chosen = max(actions_list, key=lambda a: a.utility)

    brain.record_action_execution(chosen.action)
    brain.track_consecutive_action(chosen.action)

    if chosen.action == 'A' and tile_needs_probing:
        brain.start_interaction_verification(raw_x, raw_y, current_map, current_dir)

    return chosen

In [ ]:
# ============================================================================
# CELL 5: Cache System - MapCache, CacheManager, IOThread
# ============================================================================
# CHANGES from v2:
# 1. MapCache: added game_state_raw, menu_data, bag_data fields
# 2. MapCache.get_state() returns 10 values
# 3. MapCache.update_state() accepts game_state_raw, menu_data, bag_data
# 4. IOThread: read_game_state() returns 10 values, all passed through
# 5. CacheManager.detect_and_set_initial_map() handles 10 values
# ============================================================================

import threading
import gc

class MapCache:
    """Thread-safe container for one map's data."""

    def __init__(self, map_id):
        self.map_id = map_id
        self.lock = threading.Lock()

        # From exploration_memory[map_id] - synced to/from Brain
        self.exploration_data = None

        # From taught_transitions filtered by map_id
        self.taught_frames = []

        # Live state (IOThread writes, Brain reads)
        self.current_state = np.zeros(EXPECTED_STATE_DIM)
        self.palette = np.zeros(PALETTE_DIM)
        self.tiles = np.zeros(TILE_DIM)
        self.raw_position = (0, 0)
        self.dead = False
        self.battle_data = DEFAULT_BATTLE_DATA.copy()
        self.party_data = DEFAULT_PARTY_DATA.copy()
        self.game_state_raw = 0
        self.menu_data = DEFAULT_MENU_DATA.copy()
        self.bag_data = DEFAULT_BAG_DATA.copy()
        self.state_fresh = False
        self.state_version = 0

        # Pending action (Brain writes, IOThread reads)
        self.pending_action_out = None

    def get_state(self):
        with self.lock:
            return (
                self.current_state.copy(),
                self.palette.copy(),
                self.tiles.copy(),
                self.dead,
                self.raw_position,
                self.battle_data.copy(),
                self.party_data.copy(),
                self.game_state_raw,
                self.menu_data.copy(),
                self.bag_data.copy(),
            )

    def update_state(self, context_state, palette, tiles, dead, raw_position,
                     battle_data=None, party_data=None,
                     game_state_raw=None, menu_data=None, bag_data=None):
        with self.lock:
            self.current_state = context_state
            self.palette = palette
            self.tiles = tiles
            self.dead = dead
            self.raw_position = raw_position
            if battle_data is not None:
                self.battle_data = battle_data
            if party_data is not None:
                self.party_data = party_data
            if game_state_raw is not None:
                self.game_state_raw = game_state_raw
            if menu_data is not None:
                self.menu_data = menu_data
            if bag_data is not None:
                self.bag_data = bag_data
            self.state_fresh = True
            self.state_version += 1

    def is_fresh(self):
        with self.lock:
            return self.state_fresh

    def mark_consumed(self):
        with self.lock:
            self.state_fresh = False

    def get_version(self):
        with self.lock:
            return self.state_version

    def set_pending_action(self, action_name):
        with self.lock:
            self.pending_action_out = action_name

    def get_pending_action(self):
        with self.lock:
            a = self.pending_action_out
            self.pending_action_out = None
            return a

    def get_taught_frames(self):
        """Return taught transitions for this map (no lock needed, read-only after init)."""
        return self.taught_frames


class CacheManager:
    """Manages all MapCaches. Pre-indexes at startup, handles map switching."""

    def __init__(self, brain):
        self.brain = brain
        self.caches = {}
        self.active_cache = None
        self.active_map_id = None
        self.lock = threading.Lock()

    def load_all(self, exploration_path=None, taught_path=None):
        exploration_path = exploration_path or EXPLORATION_MEMORY_FILE
        taught_path = taught_path or TAUGHT_TRANSITIONS_FILE

        for map_id, mem_data in self.brain.exploration_memory.items():
            cache = self._get_or_create(map_id)
            cache.exploration_data = mem_data

        taught_by_map = {}
        for t in self.brain.taught_transitions:
            t_map = t.get('state', {}).get('map_id')
            if t_map is not None:
                taught_by_map.setdefault(t_map, []).append(t)

        for map_id, frames in taught_by_map.items():
            cache = self._get_or_create(map_id)
            cache.taught_frames = frames

        total_maps = len(self.caches)
        total_taught = sum(len(c.taught_frames) for c in self.caches.values())
        print(f"  📦 CacheManager: {total_maps} maps cached, {total_taught} taught frames indexed")

    def _get_or_create(self, map_id):
        if map_id not in self.caches:
            self.caches[map_id] = MapCache(map_id)
        return self.caches[map_id]

    def get_active(self):
        return self.active_cache

    def detect_and_set_initial_map(self):
        (ctx, pal, til, dead, raw_pos, battle_data, party_data,
         game_state_raw, menu_data, bag_data) = read_game_state()
        map_id = int(ctx[2])
        self._switch_to(map_id)
        self.active_cache.update_state(ctx, pal, til, dead, raw_pos,
                                       battle_data, party_data,
                                       game_state_raw, menu_data, bag_data)
        print(f"  📦 Initial map: {map_id}")
        return map_id

    def switch_map(self, new_map_id):
        if new_map_id == self.active_map_id:
            return
        self._sync_from_brain()
        self._switch_to(new_map_id)
        self._sync_to_brain()

    def _switch_to(self, map_id):
        with self.lock:
            cache = self._get_or_create(map_id)
            self.active_cache = cache
            self.active_map_id = map_id

    def _sync_to_brain(self):
        cache = self.active_cache
        if cache and cache.exploration_data is not None:
            self.brain.exploration_memory[cache.map_id] = cache.exploration_data

    def _sync_from_brain(self):
        cache = self.active_cache
        if cache and cache.map_id in self.brain.exploration_memory:
            cache.exploration_data = self.brain.exploration_memory[cache.map_id]

    def sync_all_from_brain(self):
        for map_id, mem_data in self.brain.exploration_memory.items():
            cache = self._get_or_create(map_id)
            cache.exploration_data = mem_data

    def save_exploration_memory(self):
        self._sync_from_brain()
        self.brain.save_exploration_memory()

    def get_active_taught_frames(self):
        if self.active_cache:
            return self.active_cache.get_taught_frames()
        return []

    def get_map_density(self):
        if not self.active_cache:
            return {'taught_frames': 0, 'tier': 'sparse', 'coverage': 0.0, 'visited': 0}

        n_frames = len(self.active_cache.get_taught_frames())
        map_id = self.active_map_id

        coverage = self.brain.get_exploration_coverage(map_id) if map_id is not None else 0.0
        memory = self.brain.get_current_map_memory(map_id) if map_id is not None else {}
        visited = len(memory.get('visited_tiles', set()))

        if n_frames < 50:
            tier = 'sparse'
        elif n_frames < 200:
            tier = 'thin'
        elif n_frames < 1000:
            tier = 'medium'
        else:
            tier = 'dense'

        return {
            'taught_frames': n_frames,
            'tier': tier,
            'coverage': coverage,
            'visited': visited
        }


class IOThread(threading.Thread):
    """Background thread: reads game_state.json, writes action.json."""

    def __init__(self, cache_manager, interval=0.02, gc_interval=300):
        super().__init__(daemon=True)
        self.cm = cache_manager
        self.interval = interval
        self.gc_interval = gc_interval
        self.running = False
        self._iteration = 0

    def run(self):
        self.running = True
        print(f"  🔄 IOThread started (interval={self.interval*1000:.0f}ms)")

        while self.running:
            try:
                cache = self.cm.get_active()
                if cache is None:
                    time.sleep(self.interval)
                    continue

                # --- READ game_state.json (10 values) ---
                (ctx, pal, til, dead, raw_pos, battle_data, party_data,
                 game_state_raw, menu_data, bag_data) = read_game_state()
                cache.update_state(ctx, pal, til, dead, raw_pos,
                                   battle_data, party_data,
                                   game_state_raw, menu_data, bag_data)

                # --- WRITE action.json ---
                action = cache.get_pending_action()
                if action is not None:
                    write_action(action)

                # --- PERIODIC GC ---
                self._iteration += 1
                if self._iteration % self.gc_interval == 0:
                    gc.collect()

            except Exception as e:
                print(f"  [IOThread ERROR] {e}")

            time.sleep(self.interval)

    def stop(self):
        self.running = False
        print("  🔄 IOThread stopped")

In [ ]:
# ============================================================================
# CELL 6: Main Loop - Full Battle + Bag Thread + Item Knowledge + Timeline
# ============================================================================
# CHANGES from v16:
# 1. Load event_timeline.json at startup
# 2. Timeline stats in startup banner
# 3. Timeline status in periodic logging (nearest nav order, upcoming events,
#    HP cost estimate, preparation point detection)
# 4. Timeline stats in milestones
# 5. All other code unchanged from v16
# ============================================================================

brain = Brain()

for b in ["UP", "DOWN", "LEFT", "RIGHT"]:
    brain.add(Perceptron("action", action=b, group="move"))
for b in ["A", "B", "Start", "Select"]:
    brain.add(Perceptron("action", action=b, group="interact"))

# === FILE PATHS ===
TAUGHT_MODEL_PATH = BASE_PATH / "taught_model_checkpoint.json"
TAUGHT_EXPLORATION_PATH = BASE_PATH / "taught_exploration_memory.json"

# === LOAD AI'S OWN MODEL (primary) ===
if MODEL_CHECKPOINT_FILE.exists():
    loaded_ts = brain.load_taught_model(MODEL_CHECKPOINT_FILE)
    print(f"🤖 AI MODEL: Loaded from timestep {loaded_ts}")
    print(f"   Utilities: {[f'{a.action}:{a.utility:.3f}' for a in brain.actions()]}")
else:
    print("🤖 AI MODEL: No existing model — starting fresh")

# === LOAD TAUGHT REFERENCE (read-only, for stagnation blending) ===
brain.load_taught_reference(TAUGHT_MODEL_PATH)

# === MERGE TAUGHT EXPLORATION (additive) ===
brain.merge_taught_exploration(TAUGHT_EXPLORATION_PATH)

# === LOAD TAUGHT TRANSITIONS FOR OVERWORLD MARKOV ===
brain.load_taught_transitions(TAUGHT_TRANSITIONS_FILE)

# === LOAD TAUGHT BATTLE TRANSITIONS FOR BATTLE MARKOV ===
brain.load_taught_battle_transitions(TAUGHT_BATTLE_TRANSITIONS_FILE)

# === LOAD TAUGHT BAG TRANSITIONS FOR BAG MARKOV ===
brain.load_taught_bag_transitions(TAUGHT_BAG_TRANSITIONS_FILE)

# === LOAD TAUGHT NAVIGATION TARGETS ===
brain.load_taught_nav_targets(TAUGHT_NAV_TARGETS_FILE)

# === LOAD ROSTER + MOVE KNOWLEDGE (persistent battle data) ===
brain.load_roster(ROSTER_FILE)
brain.load_move_knowledge(MOVE_KNOWLEDGE_FILE)

# === LOAD ITEM KNOWLEDGE (persistent bag data) ===
brain.load_item_knowledge(ITEM_KNOWLEDGE_FILE)

# === LOAD EVENT TIMELINE (phase 2 — structured playthrough history) ===
brain.load_event_timeline(EVENT_TIMELINE_FILE)

# === BUILD INITIAL MAP GRAPH ===
map_graph = brain.build_map_graph()
graph_edges = sum(len(v) for v in map_graph.values())
graph_maps = list(map_graph.keys())

# === INITIALIZE CACHE SYSTEM ===
cache_manager = CacheManager(brain)
cache_manager.load_all()
cache_manager.detect_and_set_initial_map()

# === START I/O THREAD ===
io_thread = IOThread(cache_manager, interval=0.02, gc_interval=300)
io_thread.start()

exploration_weight = 1.3
forced_explore_prob = 0.18
prev_context_state = None
prev_raw_position = None
last_processed_version = -1

# === BATTLE OUTCOME TRACKING ===
battle_outcomes = {'win': 0, 'loss': 0, 'run': 0, 'catch': 0, 'unknown': 0}

print("="*70)
print("AI CONTROL - v16.1 (Phase 2: Event Timeline)")
print("="*70)
print("MODELS:")
print(f"  - AI model: {MODEL_CHECKPOINT_FILE}")
print(f"  - Taught reference: {TAUGHT_MODEL_PATH} ({'loaded' if brain.taught_reference['loaded'] else 'NOT FOUND'})")
if brain.taught_reference['loaded']:
    taught_utils = ', '.join(f"{k}:{v:.3f}" for k, v in brain.taught_reference['utilities'].items())
    print(f"  - Taught utilities: {taught_utils}")
print("="*70)
print("LUA v3 FIELDS:")
print(f"  - gs (game_state_raw): 0=OW, 1=menu/party, 14=bag")
print(f"  - mu (menu_data): mc=menu cursor, mm=max, pc=party cursor, sc=swap")
print(f"  - bg (bag_data): pk=pocket, bc=cursor, a=active, it=items[]")
print("="*70)
print("THREAD PRIORITY:")
print(f"  0. Party Menu  (forced switch, voluntary, overworld)")
print(f"  1. Bag Thread   (Markov imitation → autonomous → B exit)")
print(f"  2. Battle       (cursor-aware moves, running, Markov, A)")
print(f"  3. Overworld    (nav, Markov, curiosity exploration)")
print("="*70)
print("BAG THREAD:")
print(f"  - Bag transitions: {'LOADED' if brain.bag_loaded else 'NOT FOUND (B-exit fallback)'}")
if brain.bag_loaded:
    print(f"  - Bag frames: {len(brain.bag_transitions)}")
    print(f"  - Sessions recorded: {brain.bag_metadata.get('bag_sessions_recorded', 0)}")
    items_used = brain.bag_metadata.get('items_used', [])
    if items_used:
        print(f"  - Items in demos: {items_used}")
print(f"  - Bag Markov threshold: {BAG_MARKOV_THRESHOLD:.2f}")
print(f"  - Bag timeout: {brain.BAG_THREAD_TIMEOUT} frames")
print(f"  ITEM KNOWLEDGE:")
print(f"  - File: {ITEM_KNOWLEDGE_FILE}")
print(f"  - Items tracked: {len(brain.item_knowledge)}")
known_cats = {}
for iid, ik in brain.item_knowledge.items():
    cat = ik.get('category', 'unknown')
    known_cats[cat] = known_cats.get(cat, 0) + 1
if known_cats:
    cats_str = ', '.join(f"{k}:{v}" for k, v in known_cats.items())
    print(f"  - Categories: {cats_str}")
print(f"  MENU TRAP EXCEPTIONS:")
print(f"  - Bag (gs==14): EXEMPT | Start menu (gs==1): EXEMPT")
print(f"  - Party menu: EXEMPT | PC/dialogue/other: B-boosted")
print("="*70)
print("EVENT TIMELINE:")
tl_status = brain.get_timeline_status()
if tl_status['loaded']:
    print(f"  - Events: {tl_status['events']} | Segments: {tl_status['segments']} | Prep points: {tl_status['prep_points']}")
    print(f"  - Nav targets covered: {tl_status['nav_covered']}")
    # Show preparation points
    for pp in brain.event_preparation_points[:5]:
        reason = pp.get('reason', '?')
        before = pp.get('before_nav_order', '?')
        threshold = pp.get('party_hp_threshold', '?')
        action = pp.get('action_taken', '?')
        print(f"  - Prep #{before}: {reason} (HP<{threshold}) → {action}")
else:
    print(f"  - NOT LOADED (teaching code needs to generate event_timeline.json)")
    print(f"  - Phase 3 preparation triggers will be inactive until loaded")
print("="*70)
print("BATTLE SYSTEM:")
print(f"  - Battle transitions: {'LOADED' if brain.battle_loaded else 'NOT FOUND (A-button fallback)'}")
if brain.battle_loaded:
    print(f"  - Battle frames: {len(brain.battle_transitions)}")
    print(f"  - Battles recorded: {brain.battle_metadata.get('battles_recorded', 0)}")
    outcomes = brain.battle_metadata.get('outcomes', {})
    if outcomes:
        print(f"  - Taught outcomes: win={outcomes.get('win',0)} run={outcomes.get('run',0)} loss={outcomes.get('loss',0)}")
    print(f"  - Markov threshold: {BATTLE_MARKOV_THRESHOLD_LOW:.2f}-{BATTLE_MARKOV_THRESHOLD_HIGH:.2f}")
print(f"  - Smart move selection | PP awareness | Forced switch | Running | Turn tracking")
print("="*70)
print("ROSTER & MOVE KNOWLEDGE:")
print(f"  - Roster entries: {len(brain.roster)}")
for slot, entry in brain.roster.items():
    moves_str = ','.join(str(m) for m in entry.get('moves', []))
    print(f"    Slot {slot}: species {entry.get('species')} Lv{entry.get('level')} moves=[{moves_str}]")
print(f"  - Moves tracked: {len(brain.move_knowledge)}")
total_uses = sum(m.get('total_uses', 0) for m in brain.move_knowledge.values())
print(f"  - Total move uses: {total_uses}")
print(f"  - Enemy moves observed: {len(brain.enemy_move_knowledge)}")
print("="*70)
print("CACHE SYSTEM:")
print(f"  - Maps cached: {len(cache_manager.caches)}")
print(f"  - Active map: {cache_manager.active_map_id}")
active_taught = len(cache_manager.get_active_taught_frames())
print(f"  - Active map taught frames: {active_taught}")
print(f"  - IOThread interval: {io_thread.interval*1000:.0f}ms")
print("="*70)
print("BLEND | NAV | MARKOV:")
print(f"  - Blend tiers: 1(80/20) 2(60/40) 3(40/60) cooldown={brain.BLEND_COOLDOWN}")
nav_status = brain.get_nav_targets_status()
if nav_status['loaded']:
    print(f"  - Nav targets: {nav_status['total']} across {list(brain.taught_nav_targets.keys())}")
else:
    print(f"  - Nav targets: NOT LOADED (frontier fallback)")
print(f"  - Map graph: {len(graph_maps)} maps, {graph_edges} edges")
print(f"  - OW Markov: {len(brain.taught_transitions)} frames, density-adaptive thresholds")
print("="*70)

try:
    while True:
        # === WAIT FOR NEW STATE FROM IOTHREAD ===
        active_cache = cache_manager.get_active()
        current_version = active_cache.get_version()

        if current_version == last_processed_version:
            time.sleep(0.005)
            continue

        # === READ STATE (10 values) ===
        (context_state, palette_state, tile_state, dead, raw_position,
         battle_data, party_data, game_state_raw, menu_data, bag_data) = active_cache.get_state()
        last_processed_version = current_version

        if np.sum(np.abs(context_state)) < 0.001:
            time.sleep(0.01)
            continue

        raw_x, raw_y = raw_position
        in_battle = context_state[3]
        current_map = int(context_state[2])
        current_dir = int(context_state[5])

        # === UPDATE ALL DATA EVERY FRAME ===
        brain.update_battle_data(battle_data)
        brain.update_party_data(party_data)
        brain.update_menu_data(menu_data)
        brain.update_bag_data(bag_data)
        brain.game_state_raw = game_state_raw

        # === THREAD STATE DETECTION — EVERY FRAME ===
        brain.update_party_menu_state(context_state)
        brain.update_bag_thread_state(context_state)

        # === ITEM OBSERVATION — EVERY FRAME ===
        brain.check_item_observation()

        # === BATTLE START/END DETECTION ===
        currently_in_battle = in_battle > 0.5

        if currently_in_battle and not brain.in_battle_last_frame:
            brain.current_battle_id += 1
            brain.battle_frame_count = 0
            brain.battle_action_history.clear()

            brain.on_battle_start_with_data()

            bd = brain.battle_data
            species_info = ""
            if bd['player_species'] > 0 and bd['enemy_species'] > 0:
                species_info = f" | Species: {bd['player_species']} vs {bd['enemy_species']}"
            hp_info = ""
            if bd['player_hp'] > 0:
                hp_info = f" | HP: {bd['player_hp']}/{bd['player_max_hp']}"
            bt = bd.get('battle_type', 0)
            bt_str = "TRAINER" if (bt & 8) != 0 else "WILD"
            moves_info = ""
            if bd['move0'] > 0:
                m_ids = [bd['move0'], bd['move1'], bd['move2'], bd['move3']]
                pp_vals = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
                moves_parts = []
                for i in range(4):
                    if m_ids[i] > 0:
                        moves_parts.append(f"m{m_ids[i]}({pp_vals[i]}pp)")
                moves_info = f" | Moves: {' '.join(moves_parts)}"

            print(f"\n  ⚔️ BATTLE START (#{brain.current_battle_id}) {bt_str} at Map {current_map} ({raw_x}, {raw_y}){species_info}{hp_info}{moves_info}")

            if brain.has_battle_data():
                print(f"     🎮 Battle awareness: ACTIVE")
                if bd['enemy_species'] > 0:
                    ranked = brain.get_best_move_for_enemy(bd['enemy_species'])
                    if ranked:
                        ranked_str = ', '.join(f"m{mid}(slot{sl} score={sc:.1f})" for mid, sl, sc in ranked[:4])
                        print(f"     📖 Move ranking vs species {bd['enemy_species']}: {ranked_str}")
                    else:
                        print(f"     📖 No move knowledge vs species {bd['enemy_species']} — will learn")
            else:
                print(f"     🎮 Battle awareness: INACTIVE (old Lua)")

            if brain.is_nav_active():
                brain.abort_navigation("battle started")

        elif currently_in_battle:
            if brain.has_battle_data() and brain.detect_turn_resolved():
                brain.on_battle_turn_end()

                bd = brain.battle_data
                turn_info = f"Turn {brain.turn_count}"
                hp_info = f"HP: {bd['player_hp']}/{bd['player_max_hp']}"
                ehp_info = f"Enemy: {bd['enemy_hp']}/{bd.get('enemy_max_hp', '?')}"

                move_info = ""
                if brain.last_move_used > 0:
                    move_info = f" | Used move {brain.last_move_used} (slot {brain.last_move_slot})"

                enemy_move = brain.detect_enemy_move()
                enemy_info = ""
                if enemy_move > 0:
                    enemy_info = f" | Enemy used move {enemy_move}"

                run_info = ""
                if brain.should_run():
                    run_info = " | 🏃 WANT TO RUN"
                elif brain.battle_no_damage_turns > 0:
                    run_info = f" | no_dmg_turns={brain.battle_no_damage_turns}"

                print(f"  ⚔️ {turn_info} | {hp_info} | {ehp_info}{move_info}{enemy_info}{run_info}")

        elif not currently_in_battle and brain.in_battle_last_frame:
            outcome = brain.on_battle_end_with_data()
            battle_outcomes[outcome] = battle_outcomes.get(outcome, 0) + 1

            battle_markov_rate = brain.battle_markov_action_count / max(1, brain.battle_action_count)
            cursor_rate = brain.battle_cursor_action_count / max(1, brain.battle_action_count)

            outcome_emoji = {'win': '🏆', 'loss': '💀', 'run': '🏃', 'catch': '🎊', 'unknown': '❓'}.get(outcome, '❓')

            print(f"\n  ⚔️ BATTLE END (#{brain.current_battle_id}) — {brain.battle_frame_count} frames, "
                  f"{brain.turn_count} turns — {outcome_emoji} {outcome.upper()}")
            print(f"     Markov: {brain.battle_markov_action_count}/{brain.battle_action_count} ({battle_markov_rate:.1%})"
                  f" | Cursor-aware: {brain.battle_cursor_action_count} ({cursor_rate:.1%})")

            if brain.move_knowledge:
                mk_summary = []
                for mid, mk in brain.move_knowledge.items():
                    if mk['total_uses'] > 0:
                        mk_summary.append(f"m{mid}:{mk['total_uses']}uses avg{mk['avg_damage']:.0f}dmg "
                                           f"{mk['misses']}miss")
                if mk_summary:
                    print(f"     📖 Move knowledge: {' | '.join(mk_summary[:4])}")

            print(f"     Session outcomes: W:{battle_outcomes.get('win',0)} L:{battle_outcomes.get('loss',0)}"
                  f" R:{battle_outcomes.get('run',0)} C:{battle_outcomes.get('catch',0)}"
                  f" ?:{battle_outcomes.get('unknown',0)}")

            brain.battle_frame_count = 0

            if brain.roster_dirty:
                brain.save_roster()
            if brain.move_knowledge_dirty:
                brain.save_move_knowledge()

        brain.in_battle_last_frame = currently_in_battle

        # === MAP CHANGE DETECTION ===
        if not currently_in_battle and current_map != cache_manager.active_map_id:
            old_map = cache_manager.active_map_id
            cache_manager.switch_map(current_map)
            active_cache = cache_manager.get_active()
            print(f"  📦 Cache switched to map {current_map} "
                  f"(taught frames: {len(active_cache.get_taught_frames())})")

            if brain.nav_map_chain and brain.is_nav_active() and not brain.is_nav_paused():
                current_pos = (raw_x, raw_y)
                chain_continues = brain.advance_map_chain(current_map, current_pos)
                if not chain_continues:
                    brain.abort_navigation("cross-map chain broken")

        brain.update_position(raw_x, raw_y)

        derived = compute_derived_features(context_state, prev_context_state)
        learning_state = build_learning_state(derived, palette_state, tile_state, in_battle)

        brain.log_state(learning_state, context_state)

        brain.confirm_action_executed(context_state, prev_context_state)

        if brain.should_send_new_action():
            taught_frames = cache_manager.get_active_taught_frames()
            map_density = cache_manager.get_map_density()

            action = anticipatory_action(
                brain, learning_state, context_state,
                exploration_weight=exploration_weight,
                raw_position=raw_position,
                forced_explore_prob=forced_explore_prob,
                taught_frames=taught_frames,
                map_density=map_density,
                palette_state=palette_state
            )

            if action is not None:
                active_cache.set_pending_action(action.action)
                brain.last_action = action.action
                brain.set_pending_action(action.action)
                if not currently_in_battle:
                    brain.update_menu_trap_tracking(context_state, action.action, raw_position=raw_position)
            else:
                active_cache.set_pending_action("NONE")
        else:
            if brain.pending_action:
                active_cache.set_pending_action(brain.pending_action)

        # === LOGGING ===
        if brain.timestep % 100 == 0:
            memory = brain.get_current_map_memory(current_map)
            visited_count = len(memory['visited_tiles'])
            obs_count = len(memory['obstructions'])
            interactables = len(memory['interactable_objects'])
            coverage = brain.get_exploration_coverage(current_map)
            transitions = memory.get('transitions', [])
            tile_stats = brain.get_tile_interaction_stats(current_map)

            tile_needs_probing = brain.should_interact_at_tile(raw_x, raw_y, current_map)
            probe_action, probe_dir = brain.get_best_probe_action(raw_x, raw_y, current_map, current_dir)

            dir_name = brain.DIRECTION_NAMES.get(current_dir, '?')
            mode = brain.control_mode
            is_both_mode = brain.should_use_both_mode()
            mode_display = "BOTH ⚡" if is_both_mode else mode

            total_actions = brain.markov_action_count + brain.curiosity_action_count
            markov_ratio = brain.markov_action_count / max(1, total_actions)

            density = cache_manager.get_map_density()

            gs_names = {0: "OW", 1: "MENU", 14: "BAG"}
            gs_str = gs_names.get(game_state_raw, f"GS{game_state_raw}")

            print(f"\n{'='*70}")
            print(f"Step {brain.timestep} | Map {current_map} | Pos ({raw_x}, {raw_y}) facing {dir_name} | gs={gs_str}")
            print(f"  Mode: {mode_display} | Battle: {int(in_battle)} | Stagnation: {brain.state_stagnation_count}")

            # === ACTIVE THREAD STATUS ===
            md = brain.menu_data
            bgd = brain.bag_data

            if brain.is_party_menu_active():
                pm_frames = brain.timestep - brain.party_menu_entered_at
                print(f"  📋 PARTY MENU ACTIVE: {brain.party_menu_context} "
                      f"({pm_frames} frames, {brain.party_menu_action_count} actions)")
                if brain.party_menu_target_slot >= 0:
                    print(f"     Target slot: {brain.party_menu_target_slot}")

            if brain.is_bag_thread_active():
                bag_frames = brain.timestep - brain.bag_thread_entered_at
                bag_markov_rate = brain.bag_thread_markov_actions / max(1, brain.bag_thread_total_actions)
                pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                pk_str = pocket_names.get(bgd['pocket'], f"?{bgd['pocket']}")
                n_items = len(bgd['items'])
                current_item = brain.get_item_at_cursor()
                item_str = f" item_at_cursor={current_item}" if current_item > 0 else ""

                print(f"  🎒 BAG THREAD ACTIVE: {brain.bag_thread_context} "
                      f"({bag_frames} frames, {brain.bag_thread_action_count} actions)")
                print(f"     Pocket: {pk_str} | Cursor: {bgd['cursor']} | Items: {n_items}{item_str}")
                print(f"     mc={md['mc']} mm={md['mm']}")
                print(f"     Bag Markov: {brain.bag_thread_markov_actions}/{brain.bag_thread_total_actions} ({bag_markov_rate:.1%})")
                if brain.pending_item_observation:
                    obs = brain.pending_item_observation
                    print(f"     📝 OBSERVING item {obs['item_id']} (waiting {obs['frames_waiting']} frames)")

            elif game_state_raw == 14 and not brain.is_bag_thread_active():
                pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                pk_str = pocket_names.get(bgd['pocket'], f"?{bgd['pocket']}")
                n_items = len(bgd['items'])
                print(f"  🎒 BAG OPEN (thread inactive): pocket={pk_str} cursor={bgd['cursor']} items={n_items}")

            if game_state_raw == 1 and not brain.is_party_menu_active() and not brain.is_bag_thread_active():
                print(f"  📋 MENU ACTIVE: mc={md['mc']}/{md['mm']} pc={md['pc']} sc={md['sc']}")

            # === BATTLE STATUS ===
            if currently_in_battle:
                battle_total = brain.battle_action_count
                battle_markov = brain.battle_markov_action_count
                battle_markov_rate = battle_markov / max(1, battle_total)
                cursor_rate = brain.battle_cursor_action_count / max(1, battle_total)

                bd = brain.battle_data

                print(f"\n  ⚔️ IN BATTLE (#{brain.current_battle_id}):")
                print(f"     Frame: {brain.battle_frame_count} | Turn: {brain.turn_count} | Actions: {battle_total}")

                if brain.has_battle_data():
                    menu_state = brain.battle_menu_state
                    cursor_names = {0: 'FIGHT', 1: 'BAG', 2: 'PKMN', 3: 'RUN'}
                    bc_str = cursor_names.get(bd['battle_cursor'], f"?({bd['battle_cursor']})")
                    mc_str = f"Move {bd['move_cursor']}" if bd['move_cursor'] >= 0 else "N/A"
                    bt = bd.get('battle_type', 0)
                    bt_str = "TRAINER" if (bt & 8) != 0 else "WILD"

                    print(f"     🎮 Menu: {menu_state} | Cursor: {bc_str} | Move: {mc_str} | Type: {bt_str}")
                    print(f"     📋 Menu addr: mc={md['mc']} mm={md['mm']} pc={md['pc']} sc={md['sc']}")

                    if bd['battle_cursor'] == 1:
                        if bgd['active']:
                            n_items = len(bgd['items'])
                            pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                            pk_str = pocket_names.get(bgd['pocket'], f"?{bgd['pocket']}")
                            print(f"     🎒 BAG selected: pocket={pk_str} cursor={bgd['cursor']} items={n_items}")

                    if bd['player_species'] > 0:
                        hp_pct = f"{bd['player_hp']}/{bd['player_max_hp']}" if bd['player_max_hp'] > 0 else f"{bd['player_hp']}"
                        lv = bd.get('player_level', '?')
                        pst = bd.get('player_status', 0)
                        status_str = f" status:{pst}" if pst != 0 else ""
                        print(f"     👤 Player: species {bd['player_species']} Lv{lv} HP {hp_pct}{status_str}")

                        m_ids = [bd['move0'], bd['move1'], bd['move2'], bd['move3']]
                        pp_vals = [bd['pp0'], bd['pp1'], bd['pp2'], bd['pp3']]
                        moves_parts = []
                        for i in range(4):
                            if m_ids[i] > 0:
                                moves_parts.append(f"m{m_ids[i]}({pp_vals[i]}pp)")
                        if moves_parts:
                            print(f"     🗡️ Moves: {' '.join(moves_parts)}")

                    if bd['enemy_species'] > 0:
                        ehp = f"{bd['enemy_hp']}/{bd.get('enemy_max_hp', '?')}"
                        elv = bd.get('enemy_level', '?')
                        est = bd.get('enemy_status', 0)
                        estatus_str = f" status:{est}" if est != 0 else ""
                        print(f"     👾 Enemy: species {bd['enemy_species']} Lv{elv} HP {ehp}{estatus_str}")

                    if brain.should_run():
                        print(f"     🏃 RUNNING: no_dmg={brain.battle_no_damage_turns}, low_exits={brain.battle_low_hp_exits}")

                    if brain.forced_switch_pending:
                        print(f"     🔄 FORCED SWITCH pending → slot {brain.forced_switch_target_slot}")

                    print(f"     Cursor-aware: {brain.battle_cursor_action_count} ({cursor_rate:.1%})")
                else:
                    print(f"     🎮 Battle awareness: INACTIVE (old Lua)")

                print(f"     Markov: {battle_markov}/{battle_total} ({battle_markov_rate:.1%})")

            else:
                # === NAVIGATION STATUS (overworld only) ===
                cross_status = brain.get_cross_map_status()

                if brain.is_nav_active():
                    if cross_status['active']:
                        chain_str = ' → '.join(str(m) for m in cross_status['chain'])
                        current_step = cross_status['chain_index']
                        total_steps = cross_status['chain_length']

                        if brain.is_nav_paused():
                            print(f"\n  🧭🌍 CROSS-MAP NAV PAUSED:")
                            print(f"     Chain: {chain_str} (step {current_step + 1}/{total_steps})")
                            print(f"     Reason: {cross_status['paused_reason']}")
                        else:
                            nav_progress = f"{brain.nav_path_index}/{len(brain.nav_path)}"
                            print(f"\n  🧭🌍 CROSS-MAP NAV ACTIVE:")
                            print(f"     Chain: {chain_str} (step {current_step + 1}/{total_steps})")
                            print(f"     Path: {nav_progress} | Steps: {brain.nav_steps_taken}")
                    else:
                        nav_target = brain.nav_target
                        nav_progress = f"{brain.nav_path_index}/{len(brain.nav_path)}"
                        print(f"\n  🧭 NAV: → ({nav_target[0]}, {nav_target[1]}) | {nav_progress} | Steps: {brain.nav_steps_taken}")

                elif brain.is_in_nav_curiosity_window():
                    print(f"\n  🧭 NAV CURIOSITY: {brain.nav_curiosity_countdown} steps remaining")
                else:
                    nav_status = brain.get_nav_targets_status()
                    print(f"\n  🧭 Nav: inactive (known area: {brain.known_area_counter}/{brain.KNOWN_AREA_TRIGGER})")
                    if nav_status['loaded']:
                        print(f"     Targets: {nav_status['remaining']}/{nav_status['total']} remaining")

            # === DECISIONS ===
            print(f"\n  🧠 DECISIONS:")
            print(f"     OW Markov: {brain.markov_action_count} ({markov_ratio:.1%}) | Curiosity: {brain.curiosity_action_count}")
            print(f"     Battle: {brain.battle_action_count} actions, {brain.battle_markov_action_count} Markov")
            print(f"     Bag: {brain.bag_thread_total_actions} actions, {brain.bag_thread_markov_actions} Markov")
            print(f"     Density: {density['tier']} ({density['taught_frames']} frames, {density['coverage']:.0%} cov)")

            if any(v > 0 for v in battle_outcomes.values()):
                print(f"\n  🏆 OUTCOMES: W:{battle_outcomes.get('win',0)} L:{battle_outcomes.get('loss',0)}"
                      f" R:{battle_outcomes.get('run',0)} C:{battle_outcomes.get('catch',0)}"
                      f" ?:{battle_outcomes.get('unknown',0)}")

            # === EVENT TIMELINE STATUS ===
            if brain.event_timeline_loaded:
                current_nav = brain.get_nearest_nav_order(raw_x, raw_y, current_map)
                if current_nav >= 0:
                    upcoming = brain.get_upcoming_events(current_nav, lookahead=5)
                    upcoming_battles = [e for e in upcoming if e.get('type') == 'battle']
                    upcoming_trainers = [e for e in upcoming
                                         if e.get('type') == 'battle' and
                                         e.get('battle_info', {}).get('battle_type') == 'trainer']
                    hp_cost = brain.get_estimated_hp_cost_ahead(current_nav, lookahead=5)
                    prep = brain.get_preparation_point(current_nav)
                    lowest_hp = brain.get_lowest_hp_ratio()

                    print(f"\n  📅 TIMELINE: near nav #{current_nav}")
                    print(f"     Upcoming: {len(upcoming)} events ({len(upcoming_battles)} battles, {len(upcoming_trainers)} trainer)")
                    print(f"     Est HP cost ahead: {hp_cost:.0%} | Lowest party HP: {lowest_hp:.0%}")
                    if prep:
                        threshold = prep.get('party_hp_threshold', 1.0)
                        reason = prep.get('reason', '?')
                        action = prep.get('action_taken', '?')
                        needs_prep = lowest_hp < threshold
                        status = "⚠️ SHOULD PREPARE" if needs_prep else "✅ OK"
                        print(f"     Prep point: {reason} (HP<{threshold}) → {action} | {status}")

            # === ITEM KNOWLEDGE ===
            if brain.item_knowledge:
                known = [(iid, ik) for iid, ik in brain.item_knowledge.items() if ik.get('category', 'unknown') != 'unknown']
                if known:
                    print(f"\n  🎒📖 ITEMS ({len(brain.item_knowledge)} tracked, {len(known)} categorized):")
                    for iid, ik in sorted(known, key=lambda x: x[1].get('uses', 0), reverse=True)[:6]:
                        cat = ik['category']
                        conf = ik['confidence']
                        uses = ik['uses']
                        extra = ""
                        if cat in ('heal_hp', 'heal_both'):
                            extra = f" avg {ik.get('avg_hp_restored', 0):.0f}HP"
                        if cat == 'catch':
                            extra = f" {ik.get('catch_successes', 0)}/{ik.get('catch_attempts', 0)} catches"
                        if ik.get('status_cured'):
                            extra += f" cures:{ik['status_cured']}"
                        print(f"     Item {iid}: {cat} ({conf:.0%}) {uses} uses{extra}")

            # === ROSTER + PARTY ===
            if brain.roster:
                print(f"\n  📋 ROSTER ({len(brain.roster)}):")
                for slot, entry in brain.roster.items():
                    moves_str = ','.join(str(m) for m in entry.get('moves', []))
                    print(f"     [{slot}] sp{entry.get('species')} Lv{entry.get('level')} [{moves_str}]")

            if brain.party_data.get('count', 0) > 0:
                print(f"\n  👥 PARTY ({brain.party_data['count']}):")
                for i, slot in enumerate(brain.party_data.get('slots', [])):
                    hp = slot.get('hp', 0)
                    mhp = slot.get('max_hp', 0)
                    lv = slot.get('level', 0)
                    st = slot.get('status', 0)
                    ratio = f"{hp/mhp:.0%}" if mhp > 0 else "?"
                    status_str = f" ⚠️st:{st}" if st != 0 else ""
                    print(f"     [{i}] Lv{lv} HP:{hp}/{mhp} ({ratio}){status_str}")

            if brain.taught_reference['loaded']:
                blend_status = f"Tier {brain.blend_tier}" if brain.blend_tier > 0 else "Inactive"
                print(f"\n  🔀 BLEND: {blend_status} | Blends: {brain.blend_count}")

            print(f"\n  📊 EXPLORATION:")
            print(f"     Visited: {visited_count} | Obs: {obs_count} | Coverage: {coverage:.0%} | Interactables: {interactables}")

            print(f"\n  🎯 PROBING:")
            print(f"     Probed: {tile_stats['probed']} | Exhausted: {tile_stats['exhausted']} | Success: {tile_stats['with_success']}")

            if transitions:
                print(f"\n  🚪 TRANSITIONS: {len(transitions)}")
                for t in transitions[:3]:
                    pos = tuple(t['position']) if isinstance(t['position'], list) else t['position']
                    banned = "🚫" if brain.is_transition_banned(current_map, pos, t['direction']) else ""
                    print(f"     ({pos[0]},{pos[1]}) → Map {t['destination_map']} (x{t['use_count']}) {banned}")

            if brain.menu_trap_b_boost > 1.0:
                print(f"\n  🔒 MENU TRAP: B boost {brain.menu_trap_b_boost:.2f}x ({brain.menu_trap_frames} frames)")

            action_utils = sorted([(a.action, a.utility) for a in brain.actions()], key=lambda x: x[1], reverse=True)
            print(f"\n  ⚡ Utils: {' '.join([f'{k}:{v:.2f}' for k,v in action_utils])}")
            print(f"  🧩 Perceptrons: {len(brain.perceptrons)} ({len(brain.actions())} act, {len(brain.entities())} ent)")

            if brain.state_stagnation_count > 10:
                print(f"  ⚠️ STAGNATION: {brain.state_stagnation_count}/{brain.STATE_STAGNATION_THRESHOLD}")
            if brain.detected_pattern:
                pattern_str = '-'.join(str(a) for a in brain.detected_pattern)
                print(f"  🔄 PATTERN: {pattern_str} x{brain.pattern_repeat_count}")

        # === MILESTONES ===
        if brain.timestep % 500 == 0 and brain.timestep > 0:
            total_visited = sum(len(m['visited_tiles']) for m in brain.exploration_memory.values())
            total_obs = sum(len(m['obstructions']) for m in brain.exploration_memory.values())
            total_interactables = sum(len(m['interactable_objects']) for m in brain.exploration_memory.values())
            total_transitions = sum(len(m.get('transitions', [])) for m in brain.exploration_memory.values())
            total_probed = sum(len(m.get('tile_interactions', {})) for m in brain.exploration_memory.values())
            total_exhausted = sum(
                sum(1 for t in m.get('tile_interactions', {}).values() if t.get('exhausted', False))
                for m in brain.exploration_memory.values()
            )

            total_actions = brain.markov_action_count + brain.curiosity_action_count
            markov_ratio = brain.markov_action_count / max(1, total_actions)

            battle_total = brain.battle_action_count
            battle_markov = brain.battle_markov_action_count
            battle_markov_rate = battle_markov / max(1, battle_total)

            bag_total = brain.bag_thread_total_actions
            bag_markov = brain.bag_thread_markov_actions
            bag_markov_rate = bag_markov / max(1, bag_total)

            mg = brain.build_map_graph()
            mg_edges = sum(len(v) for v in mg.values())

            total_move_uses = sum(m.get('total_uses', 0) for m in brain.move_knowledge.values())
            total_item_uses = sum(ik.get('uses', 0) for ik in brain.item_knowledge.values())
            items_categorized = sum(1 for ik in brain.item_knowledge.values() if ik.get('category', 'unknown') != 'unknown')

            print(f"\n{'#'*70}")
            print(f"# MILESTONE {brain.timestep}")
            print(f"# Maps: {len(brain.exploration_memory)} | Tiles: {total_visited} | Obs: {total_obs}")
            print(f"# Interactables: {total_interactables} | Transitions: {total_transitions}")
            print(f"# Probed: {total_probed} | Exhausted: {total_exhausted}")
            print(f"#")
            print(f"# THREADS:")
            gs_names_m = {0: "OW", 1: "MENU", 14: "BAG"}
            gs_str_m = gs_names_m.get(brain.game_state_raw, f"GS{brain.game_state_raw}")
            print(f"#   gs={gs_str_m} | party_menu={'ACTIVE('+brain.party_menu_context+')' if brain.party_menu_active else 'off'}"
                  f" | bag={'ACTIVE('+brain.bag_thread_context+')' if brain.bag_thread_active else 'off'}")
            print(f"#")
            print(f"# DECISIONS:")
            print(f"#   OW Markov: {brain.markov_action_count} ({markov_ratio:.1%}) | Curiosity: {brain.curiosity_action_count}")
            print(f"#   Battle: {battle_total} actions, {battle_markov} Markov ({battle_markov_rate:.1%})")
            print(f"#   Bag: {bag_total} actions, {bag_markov} Markov ({bag_markov_rate:.1%})")
            print(f"#   Cursor-aware: {brain.battle_cursor_action_count}")
            print(f"#")
            print(f"# BATTLES:")
            print(f"#   Fought: {brain.current_battle_id}")
            print(f"#   Outcomes: W:{battle_outcomes.get('win',0)} L:{battle_outcomes.get('loss',0)}"
                  f" R:{battle_outcomes.get('run',0)} C:{battle_outcomes.get('catch',0)}"
                  f" ?:{battle_outcomes.get('unknown',0)}")
            print(f"#   Low HP exits: {brain.battle_low_hp_exits}")
            print(f"#")
            print(f"# KNOWLEDGE:")
            print(f"#   Roster: {len(brain.roster)} slots")
            print(f"#   Moves: {len(brain.move_knowledge)} tracked, {total_move_uses} uses")
            print(f"#   Enemy moves: {len(brain.enemy_move_knowledge)}")
            print(f"#   Items: {len(brain.item_knowledge)} tracked, {total_item_uses} uses, {items_categorized} categorized")
            print(f"#")
            print(f"# TIMELINE:")
            if brain.event_timeline_loaded:
                tl = brain.get_timeline_status()
                print(f"#   Events: {tl['events']} | Segments: {tl['segments']} | Prep: {tl['prep_points']}")
                current_nav = brain.get_nearest_nav_order(raw_x, raw_y, current_map)
                if current_nav >= 0:
                    hp_cost = brain.get_estimated_hp_cost_ahead(current_nav)
                    prep = brain.get_preparation_point(current_nav)
                    print(f"#   Current nav: #{current_nav} | HP cost ahead: {hp_cost:.0%}")
                    if prep:
                        print(f"#   Prep point: {prep.get('reason', '?')} (HP<{prep.get('party_hp_threshold', '?')})")
            else:
                print(f"#   Not loaded (phase 3 inactive)")
            print(f"#")
            print(f"# MAP GRAPH: {len(mg)} maps, {mg_edges} edges")
            print(f"# NAV: {'active' if brain.is_nav_active() else 'inactive'}")
            cross_status = brain.get_cross_map_status()
            if cross_status['active']:
                chain_str = ' → '.join(str(m) for m in cross_status['chain'])
                print(f"#   Cross-map: {chain_str}")
            nav_status = brain.get_nav_targets_status()
            if nav_status['loaded']:
                print(f"#   Targets: {nav_status['remaining']}/{nav_status['total']} remaining")
            print(f"#")
            print(f"# BLEND: tier={brain.blend_tier} count={brain.blend_count}")
            print(f"{'#'*70}")

            brain.save_model_checkpoint(BASE_PATH / "model_checkpoint.json")
            cache_manager.save_exploration_memory()
            if brain.item_knowledge_dirty:
                brain.save_item_knowledge()
            print(f"# Saved: model + exploration + roster + moves + items")

        # === WAIT FOR NEXT STATE ===
        for _ in range(10):
            time.sleep(0.005)
            if active_cache.get_version() > last_processed_version:
                break

        # === LEARN ===
        (next_ctx, next_pal, next_til, dead,
         next_raw_pos, next_battle_data, next_party_data,
         next_gs_raw, next_menu_data, next_bag_data) = active_cache.get_state()
        last_processed_version = active_cache.get_version()

        next_in_battle = next_ctx[3]
        next_derived = compute_derived_features(next_ctx, context_state)
        next_learning_state = build_learning_state(next_derived, next_pal, next_til, next_in_battle)

        brain.learn(learning_state, next_learning_state, context_state, next_ctx, dead=dead,
                    raw_position=raw_position, next_raw_position=next_raw_pos)

        prev_context_state = context_state.copy()
        prev_raw_position = raw_position
        brain.timestep += 1

except KeyboardInterrupt:
    print("\n\n🛑 Stopping AI...")
    io_thread.stop()
    io_thread.join(timeout=2)
    cache_manager.save_exploration_memory()
    brain.save_model_checkpoint(BASE_PATH / "model_checkpoint.json")
    brain.save_roster()
    brain.save_move_knowledge()
    brain.save_item_knowledge()
    print(f"  Roster: {len(brain.roster)} entries")
    print(f"  Move knowledge: {len(brain.move_knowledge)} moves")
    print(f"  Item knowledge: {len(brain.item_knowledge)} items")
    print(f"  Bag thread: {brain.bag_thread_total_actions} total, {brain.bag_thread_markov_actions} Markov")
    print(f"  Timeline: {'loaded' if brain.event_timeline_loaded else 'not loaded'}")
    print(f"  Battle outcomes: W:{battle_outcomes.get('win',0)} L:{battle_outcomes.get('loss',0)}"
          f" R:{battle_outcomes.get('run',0)} C:{battle_outcomes.get('catch',0)}"
          f" ?:{battle_outcomes.get('unknown',0)}")
    print("✅ Saved and stopped.")